# Stage 1 - Gardiner Code -> Phonetic Fragments

Builds a high-confidence Gardiner-code -> phonetic stream for downstream assembly.
The notebook combines direct lexicon lookup, exact multi-code overrides, beam decoding,
and ranked inference only for truly unknown codes.

**Output**: `stage1_output.csv` (consumed by the Stage 2 notebook)

| Column | Description |
|--------|-------------|
| `spaced_phonetics` | final post-processed phonetic output |
| `codes` | JSON list of normalized Gardiner codes used for decoding |
| `input_type` | `gardiner` or `phonetic` |
| `decoder_mode` | `beam_decoder`, `exact_sequence_override`, or `phonetic_passthrough` |
| `stage1_confidence` | confidence score in the range `[0, 1]` |
| `unknown_codes` | JSON list of unresolved codes |
| `inferred_codes` | JSON list of codes handled by the unknown-code inferrer |
| `inference_details` | JSON provenance for inferred codes |


In [1]:
!pip install torch==2.7.1+cu118 torchvision==0.22.1+cu118 --extra-index-url https://download.pytorch.org/whl/cu118 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 12.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 3.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 43.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 42.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 9.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 23.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 12.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━

## 1. Install Dependencies

In [2]:
import importlib, subprocess, sys


def has_module(module_name, min_version=None):
    try:
        module = importlib.import_module(module_name)
    except ImportError:
        return False

    if min_version is None:
        return True

    try:
        from packaging.version import Version
        return Version(getattr(module, "__version__", "0")) >= Version(min_version)
    except Exception:
        return True


def ensure_package(module_name, package_name=None, *, required=True, min_version=None):
    package_name = package_name or module_name
    if has_module(module_name, min_version=min_version):
        return True

    print(f"Installing {'required' if required else 'optional'} dependency: {package_name}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    except Exception as exc:
        if required:
            raise RuntimeError(f"Failed to install required dependency '{package_name}': {exc}") from exc
        optional_failures.append(f"{package_name} ({exc})")
        return False

    return has_module(module_name, min_version=min_version)


optional_failures = []

ensure_package("packaging", "packaging", required=True)
ensure_package("sklearn", "scikit-learn", required=True)
ensure_package("torch", "torch", required=True)
ensure_package("ipywidgets", "ipywidgets", required=False)
ensure_package("Levenshtein", "python-Levenshtein", required=False)
ensure_package("langdetect", "langdetect", required=False)

import torch

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch available : {torch.__version__} | device={device_name}")

if optional_failures:
    print("Optional install warnings:", optional_failures)
else:
    print("Optional dependencies are ready.")

print("Dependency check complete.")


Installing optional dependency: python-Levenshtein
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.3 MB/s eta 0:00:00
Installing optional dependency: langdetect
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.3 MB/s eta 0:00:00
Torch available : 2.7.1+cu118 | device=cpu
Optional dependencies are ready.
Dependency check complete.


## 2. Imports

In [3]:
import json, os, platform, random, re, sys, time, unicodedata, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import Levenshtein
except ImportError:
    Levenshtein = None

warnings.filterwarnings("ignore")
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print("Imports ready.")


Python   : 3.12.13
Platform : Linux-6.6.122+-x86_64-with-glibc2.35
Imports ready.


## 3. Configuration

In [4]:
def candidate_search_roots() -> List[Path]:
    roots = []

    env_root = os.environ.get("HIERO_DATA_DIR")
    if env_root:
        roots.append(Path(env_root).expanduser())

    user_profile = os.environ.get("USERPROFILE") or os.environ.get("HOME")
    if user_profile:
        user_root = Path(user_profile).expanduser()
        roots.extend([
            user_root / "Downloads",
            user_root / "Documents",
            user_root / "Desktop",
        ])

    roots.extend([
        Path("/kaggle/input/datasets/judyzox/hiero-dataset"),
        Path("/kaggle/input"),
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Downloads",
        Path.home() / "Documents",
        Path.home() / "Desktop",
        Path("D:/Downloads"),
        Path("D:/Documents"),
        Path("D:/Desktop"),
    ])

    unique_roots = []
    seen = set()
    for root in roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root
        key = str(resolved)
        if key in seen or not root.exists():
            continue
        seen.add(key)
        unique_roots.append(root)
    return unique_roots


SEARCH_ROOTS = candidate_search_roots()


def find_input_file(*filenames):
    for root in SEARCH_ROOTS:
        for filename in filenames:
            direct = root / filename
            if direct.exists():
                return direct

        for filename in filenames:
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0]
    return None


@dataclass
class Config:
    main_corpus_path:      Optional[Path] = find_input_file("gardiner_updated.csv", "dataset_cleaned.csv")
    gardiner_json_path:    Optional[Path] = find_input_file("gardiner_lexicon.json")
    gardiner_updated_path: Optional[Path] = find_input_file("gardiner_updated.csv")
    gardiner_table_path:   Optional[Path] = find_input_file("Gardiner_Sign_List.csv")
    character_level_path:  Optional[Path] = find_input_file("character_level.csv")

    work_dir: Path = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    seed: int = 42

    col_raw_translit:   str = "raw_transliteration"
    col_clean_translit: str = "clean_transliteration"
    col_raw_german:     str = "raw_german"
    col_clean_german:   str = "clean_german"

    # Stage 1
    stage1_unknown_strategy: str = "code_lower"   # code_lower | placeholder | epsilon
    stage1_anchor_known_codes_when_unknown_present: bool = True
    stage1_strict_known_code_lookup: bool = True

    # Transformer inferrer
    transformer_enabled: bool = True
    transformer_force_cpu: bool = False
    transformer_train_epochs: int = 80
    transformer_batch_size: int = 64
    transformer_learning_rate: float = 3e-4
    transformer_validation_split: float = 0.12
    transformer_patience: int = 14
    transformer_beam_size: int = 5
    transformer_similarity_top_k: int = 5
    transformer_min_confidence: float = 0.42
    transformer_max_src_len: int = 20
    transformer_max_tgt_len: int = 24
    transformer_classifier_enabled: bool = True
    ngram_classifier_enabled: bool = True
    hybrid_candidate_top_k: int = 6
    hybrid_neighbor_top_k: int = 8
    hybrid_ngram_weight: float = 0.42
    hybrid_transformer_weight: float = 0.28
    hybrid_similarity_weight: float = 0.30
    hybrid_blank_threshold: float = 0.62
    hybrid_force_blank_threshold: float = 0.82

    # Stage 2 (used in Stage 2 notebook)
    assembler_short_input_threshold:    int  = 8
    assembler_deduplicate_repetitions: bool  = True


CFG = Config()
CFG.work_dir.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print("search_roots          :")
for root in SEARCH_ROOTS:
    print(f"  - {root}")
print(f"work_dir              : {CFG.work_dir}")
print(f"main_corpus_path      : {CFG.main_corpus_path}")
print(f"gardiner_json_path    : {CFG.gardiner_json_path}")
print(f"gardiner_updated_path : {CFG.gardiner_updated_path}")
print(f"gardiner_table_path   : {CFG.gardiner_table_path}")
print(f"character_level_path  : {CFG.character_level_path}")
print(f"transformer_enabled   : {CFG.transformer_enabled}")
print(f"anchor_known_on_unk   : {CFG.stage1_anchor_known_codes_when_unknown_present}")
print(f"strict_known_lookup   : {CFG.stage1_strict_known_code_lookup}")
print(f"ngram_classifier      : {CFG.ngram_classifier_enabled}")
print(f"transformer_classifier: {CFG.transformer_classifier_enabled}")


search_roots          :
  - /kaggle/input/datasets/judyzox/hiero-dataset
  - /kaggle/input
  - /kaggle/working
  - /kaggle
work_dir              : /kaggle/working
main_corpus_path      : /kaggle/input/datasets/judyzox/hiero-dataset/gardiner_updated.csv
gardiner_json_path    : /kaggle/input/datasets/judyzox/hiero-dataset/gardiner_lexicon.json
gardiner_updated_path : /kaggle/input/datasets/judyzox/hiero-dataset/gardiner_updated.csv
gardiner_table_path   : /kaggle/input/datasets/judyzox/hiero-dataset/Gardiner_Sign_List.csv
character_level_path  : /kaggle/input/datasets/judyzox/hiero-dataset/character_level.csv
transformer_enabled   : True
anchor_known_on_unk   : True
strict_known_lookup   : True
ngram_classifier      : True
transformer_classifier: True


## 4. Utilities

In [5]:
def tstamp():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(message, tag="INFO"):
    print(f"[{tstamp()}] [{tag}] {message}")

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2, default=str)

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

MDC_TO_EGYPTO = {
    "A": "\ua723", "i": "\ua7bd", "j": "\ua7bd", "y": "y", "a": "\ua725",
    "H": "\u1e25", "x": "\u1e2b", "X": "\u1e96", "S": "\u0161",
    "T": "\u1e6f", "D": "\u1e0f",
}
APOSTROPHES = {
    "\u02be": "'",
    "\u2018": "'",
    "\u2019": "'",
    "\u201b": "'",
    "\u2032": "'",
    "\u0060": "'",
    "\u00b4": "'",
}
EGYPTO_SIGS = set("\ua723\ua7bd\ua725\u1e25\u1e2b\u1e96\u0161\u1e6f\u1e0f")
TRANSLIT_CONTROL_RE = re.compile(r"[\u200b-\u200d\u2060\ufeff]")
TRANSLIT_SEPARATOR_RE = re.compile(r"[.()\[\]<>={}:;,_/\-+*|\\\u2010\u2011\u2012\u2013\u2014\u2015\u2212]+")

def looks_like_mdc(text: str) -> bool:
    if any(ch in EGYPTO_SIGS for ch in text):
        return False
    return bool(re.search(r"[AHSTDXxj]", text))

def mdc_to_egypto(text: str) -> str:
    return "".join(MDC_TO_EGYPTO.get(ch, ch) for ch in text)

def normalize_translit(text: str) -> str:
    text = clean_text(text)
    text = TRANSLIT_CONTROL_RE.sub("", text)
    for src, tgt in APOSTROPHES.items():
        text = text.replace(src, tgt)
    if looks_like_mdc(text):
        text = mdc_to_egypto(text)
    text = TRANSLIT_SEPARATOR_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def char_edit_distance(left: str, right: str) -> int:
    if Levenshtein is not None:
        return Levenshtein.distance(left, right)
    if len(left) < len(right):
        left, right = right, left
    if not right:
        return len(left)
    prev = list(range(len(right) + 1))
    for i, cl in enumerate(left, 1):
        cur = [i]
        for j, cr in enumerate(right, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(cl!=cr)))
        prev = cur
    return prev[-1]

def levenshtein_ratio(left: str, right: str) -> float:
    if Levenshtein is not None:
        return float(Levenshtein.ratio(left, right))
    if not left and not right:
        return 1.0
    return 1.0 - char_edit_distance(left, right) / max(len(left), len(right), 1)

print("Utilities ready.")


Utilities ready.


## 4.5 Comprehensive Cleaning Rules (from `Clean_dataset_comprehensive`)

Canonical transliteration / German cleaners (PDF Tables 1, 2, 4) plus the per-row processing pipeline (drop-empty → dedup → language filter → confidence). These are applied to the **parallel German corpus** before any Stage 1 decoding runs. Stage 1 decoding logic itself is unchanged; the Gardiner lexicon path (`gardiner_updated.csv`) keeps its precomputed phonetic readings untouched.

In [6]:
# =============================================================================
#  COMPREHENSIVE CLEANING RULES  (ported from Clean_dataset_comprehensive)
#  These define the canonical cleaners + per-row processing. Stage 1 decoding
#  logic is NOT touched — only the corpus that feeds it is cleaned here.
# =============================================================================
import re
import unicodedata

import pandas as pd
import numpy as np

# ---- langdetect (optional; safe fallback if missing) -----------------------
try:
    from langdetect import detect_langs, DetectorFactory, LangDetectException
    _LANGDETECT_OK = True
except Exception:
    _LANGDETECT_OK = False
    class LangDetectException(Exception):
        pass

# ---- cleaning configuration (same knobs as the comprehensive notebook) -----
REMOVE_GERMAN_PUNCTUATION = False     # keep . ? ! (better for translation)

MIN_TRANSLIT_TOKENS = 1
MAX_TRANSLIT_TOKENS = 150
MIN_GERMAN_TOKENS   = 1
MAX_GERMAN_TOKENS   = 150
MAX_TOKEN_RATIO     = 5.0

MIN_CONF_THRESHOLD  = 0.5             # rows below this are flagged low_conf (kept, not dropped)

DROP_OTHER_LANGUAGES = False          # False = keep latin/french (may be valid)
DROP_LOW_CONFIDENCE  = False          # Stage 1 keeps every cleaned row by default
LANGDETECT_MIN_LEN   = 5
LANGDETECT_MIN_PROB  = 0.85

try:
    _CLEAN_SEED = int(CFG.seed)       # reuse the notebook seed if available
except Exception:
    _CLEAN_SEED = 42

if _LANGDETECT_OK:
    DetectorFactory.seed = _CLEAN_SEED


# =============================================================================
#  TRANSLITERATION CLEANER (PDF TABLE 2)
# =============================================================================
RE_GRAM_TAG_PAREN     = re.compile(r'\(\s*\.[A-Z]{2,}[A-Za-z0-9]*\s*\)')
RE_DOT_UPPERCASE_TAG  = re.compile(r'\.[A-Z]{2,}[A-Za-z0-9]*')
RE_PL_MARKER          = re.compile(r'\.pl\b|\.Pl\b|,pl\b|\bpl\b|\{\.?pl\}|\.\{pl\}')
RE_DU_MARKER          = re.compile(r'\.du\b|,du\b')
RE_SIMPLE_NOISE       = re.compile(r'\b(ON|GN|Pr[äa]p\.?|oder\s+ḫr\s*=\s*s|L[üu]ckeL[üu]ckeGap)\b')
RE_TRANS_NOISE_CHARS  = re.compile(r'[!ø⁝:+~]')
RE_SQUARE_DESTRUCTION = re.compile(r'\.\.\s*\d+\s*Q\s*\.\.')
RE_TRANS_BRACKETS     = re.compile(r'[(){}\[\]⸢⸣⟨⟩〈〉𓍹𓍺𓊆𓊇𓉘𓊂]')
RE_TRANS_PUNCT        = re.compile(r'[?⸮.,]')
RE_TRANS_HYPHEN       = re.compile(r'[-‑‐‒–—]')
RE_TRANS_UNDERSCORE   = re.compile(r'[_⸗]')

TRANS_KILL_PATTERNS = [
    re.compile(r'^\s*[-\.…_?⸮]+\s*$'),
    re.compile(r'^\s*-\?\?-\s*$'),
    re.compile(r'^\s*\.\s*\.\s*\.\s*$'),
    re.compile(r'^\s*/+\s*$'),
]


def _split_equals(text: str) -> str:
    out = []
    for tok in text.split():
        parts = [p for p in tok.split('=') if p]
        out.extend(parts)
    return ' '.join(out)


def clean_transliteration(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''

    t = unicodedata.normalize('NFC', text)

    for pat in TRANS_KILL_PATTERNS:
        if pat.match(t):
            return ''

    t = t.replace('Ꜥ', 'ꜥ')
    t = t.replace('ʿ', 'ꜥ')
    t = t.replace('≡', '=')
    t = RE_SQUARE_DESTRUCTION.sub(' / ', t)
    t = RE_GRAM_TAG_PAREN.sub('', t)
    t = RE_TRANS_BRACKETS.sub('', t)
    t = RE_DOT_UPPERCASE_TAG.sub('', t)
    t = RE_PL_MARKER.sub('', t)
    t = RE_DU_MARKER.sub('', t)
    t = RE_SIMPLE_NOISE.sub('', t)
    t = RE_TRANS_PUNCT.sub('', t)
    t = RE_TRANS_HYPHEN.sub(' ', t)
    t = _split_equals(t)
    t = RE_TRANS_UNDERSCORE.sub(' ', t)
    t = RE_TRANS_NOISE_CHARS.sub('', t)
    return ' '.join(t.split())


# =============================================================================
#  GERMAN CLEANER (PDF TABLE 1)
# =============================================================================
DE_FULL_KILL_IF_ALONE = [
    r'^\s*\?+\s*$',
    r'^\s*[-\.…_⸮\?\s]+$',
    r'^\s*\.\s*\.\s*\.\s*$',
    r'^\s*⸮\s*_\s*\?\s*$',
    r'^\s*-+\s*\?+\s*-+\s*$',
    r'^\s*\[?---\]?\s*$',
    r'^\s*〈\s*〉\s*-+\s*$',
]
RE_DE_FULL_KILL_IF_ALONE = [re.compile(p) for p in DE_FULL_KILL_IF_ALONE]

DE_DESTRUCTION_MARKERS = [
    r'--\s*zerst[öo]rt\s*--',
    r'--\s*Zerst[öo]rung\s*--',
    r'--\s*Zeichenreste\s*--',
    r'--\s*Beischrift\s+zerst[öo]rt\s*--',
    r'--\s*unklar\s*--',
    r'keine\s+Übersetzung\s+vorhanden',
    r'Keine\s+Übersetzung\s+möglich',
    r'---\s*LEER\s+GEFUNDEN\s*---',
    r'No\s+translation\s+available',
    r'No\s+translation\s+possible',
    r'---\s*FOUND\s+EMPTY\s*---',
    r'-\s*Variante[^-]*-',
]
RE_DE_DESTRUCTION = [re.compile(p, re.IGNORECASE) for p in DE_DESTRUCTION_MARKERS]

DE_DELETE_WITH_CONTENT = [
    r'\[§[^\]]*\]',
    r'§\s*[\d]+[a-zA-Z\-]*[\s.,:]?',
    r'\$\[[^\]]*\]\$',
    r'\(\s*wört[^)]*\)',
    r'\(\s*wört[^)]*$',
    r'\[\s*ältere\s+Fassung[^\]]*\]',
    r'\(\s*älterer\s+Text[^)]*\)',
    r'\(\s*oder[^)]*\)',
    r'\[[^\]]*Beischrift[^\]]*\]\s*:?',
    r'\(\s*d\.\s*h\.?[^)]*\)',
    r'\(\s*i\.\s*e\.?[^)]*\)',
    r'\(\s*\?\s*\)',
    r'\(\s*=\s[^)]*\)',
    r'\(\s*Glosse[^)]*\)',
]
RE_DE_DELETE_WITH_CONTENT = [re.compile(p, re.IGNORECASE) for p in DE_DELETE_WITH_CONTENT]

DE_LHG_PATTERNS = [
    (re.compile(r'-\s*\{?\s*LHG\s*\}?\s*LHG\s*-'), 'Leben, Heil, Gesundheit'),
    (re.compile(r'-\s*LHG\s*-'),                    'Leben, Heil, Gesundheit'),
    (re.compile(r'-\s*LHG\b'),                       'Leben, Heil, Gesundheit'),
    (re.compile(r'\bLHG\b'),                         'Leben, Heil, Gesundheit'),
    (re.compile(r'\bLPH\b'),                         'Leben, Heil, Gesundheit'),
    (re.compile(r'\bl\.h[\.,\-]?g[\.\-\s]*'),        'Leben, Heil, Gesundheit'),
    (re.compile(r'\bl\.p\.h\.?'),                    'Leben, Heil, Gesundheit'),
]

DE_EGYPT_PATTERNS = [
    (re.compile(r'\bO\.?\s*Äg\.?'),     'Oberägypten'),
    (re.compile(r'\bO\.?\s*Ä\.?(?!g)'), 'Oberägypten'),
    (re.compile(r'\bU\.?\s*Äg\.?'),     'Unterägypten'),
    (re.compile(r'\bU\.?\s*Ä\.?(?!g)'), 'Unterägypten'),
]

DE_FRENCH_PATTERNS = [
    (re.compile(r'"arbustes\s+à\s+épines"'), 'dornige Sträucher'),
    (re.compile(r'\brôdeurs\b'),              'Plünderer'),
]

RE_DE_NN_PATTERNS = [re.compile(r'--NN--'), re.compile(r'\|NN\|'), re.compile(r'\bNN\b')]
RE_DE_STRAY_PIPE  = re.compile(r'\|')
RE_DE_DOUBLE_TRANS_CURLY_FIRST  = re.compile(r'\{([^{}]*)\}\s*〈[^〉]*〉')
RE_DE_DOUBLE_TRANS_CURLY_SECOND = re.compile(r'〈[^〉]*〉\s*\{([^{}]*)\}')
RE_DE_SLASH_PAIR        = re.compile(r'([^\s/(){}\[\].,;:!?]+)\s*/\s*([^\s/(){}\[\].,;:!?]+)')
RE_DE_VARIANTE_PREFIX   = re.compile(r'^\s*\(?\s*Variante\s*[:\)]?\s*', re.IGNORECASE)
RE_DE_PAREN_INNER       = re.compile(r'\([^()]*\)')
RE_DE_REMAINING_BRACKETS = re.compile(r'[{}\[\]⟨⟩〈〉⸢⸣<>𓉘𓊂𓍹𓍺"„"‚‘$#]')
RE_DE_EDITORIAL_MARKS    = re.compile(r'[❡¶†‡»«›‹❬❭❮❯]')
RE_DE_HYPHEN_LINEBREAK   = re.compile(r'-\s*[\r\n]+\s*')
RE_DE_SPACE_BEFORE_PUNCT = re.compile(r'\s+([.,;:!?])')
RE_DE_TRAILING_ELLIPSIS  = re.compile(r'\s*\.{3,}\s*$')
RE_DE_LEADING_JUNK       = re.compile(r'^[\s\.\-–—\'"‚‘„"]+')
RE_DE_TRAILING_JUNK      = re.compile(r'[\s\-–—\'"‚‘„"]+$')
RE_DE_REPEATED_PUNCT     = re.compile(r'([.!?,;:])\1{1,}')


def clean_german(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text.strip()
    if t in ('', 'None', 'nan', 'NaN', 'none'):
        return ''

    t = unicodedata.normalize('NFC', t)

    t = RE_DE_HYPHEN_LINEBREAK.sub('-', t)
    t = t.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
    t = ' '.join(t.split())

    t = t.replace('Ꜥ', 'ꜥ')
    t = t.replace('`', "'")
    t = t.replace('≡', '=')
    t = t.replace('&', ' und ')
    t = RE_DE_EDITORIAL_MARKS.sub('', t)

    for rx in RE_DE_FULL_KILL_IF_ALONE:
        if rx.match(t):
            return ''

    for rx in RE_DE_DESTRUCTION:
        if rx.search(t):
            return ''

    for rx, repl in DE_LHG_PATTERNS:
        t = rx.sub(repl, t)
    for rx, repl in DE_EGYPT_PATTERNS:
        t = rx.sub(repl, t)
    for rx, repl in DE_FRENCH_PATTERNS:
        t = rx.sub(repl, t)

    for rx in RE_DE_DELETE_WITH_CONTENT:
        t = rx.sub('', t)

    for rx in RE_DE_NN_PATTERNS:
        t = rx.sub('', t)
    t = RE_DE_STRAY_PIPE.sub('', t)

    t = RE_DE_VARIANTE_PREFIX.sub('', t)

    t = RE_DE_DOUBLE_TRANS_CURLY_FIRST.sub(lambda m: m.group(1), t)
    t = RE_DE_DOUBLE_TRANS_CURLY_SECOND.sub(lambda m: m.group(1), t)

    prev = None
    while prev != t:
        prev = t
        t = RE_DE_SLASH_PAIR.sub(r'\1', t)

    prev = None
    while prev != t:
        prev = t
        t = RE_DE_PAREN_INNER.sub('', t)

    t = RE_DE_REMAINING_BRACKETS.sub('', t)
    t = RE_DE_TRAILING_ELLIPSIS.sub('', t)
    t = RE_DE_LEADING_JUNK.sub('', t)
    t = RE_DE_TRAILING_JUNK.sub('', t)
    t = RE_DE_SPACE_BEFORE_PUNCT.sub(r'\1', t)
    t = RE_DE_REPEATED_PUNCT.sub(r'\1', t)
    t = ' '.join(t.split())

    if REMOVE_GERMAN_PUNCTUATION:
        t = re.sub(r'[.?!,;:]', '', t)
        t = ' '.join(t.split())

    if not t or re.fullmatch(r'[\s\.,;:\-!?]+', t):
        return ''

    return t


# =============================================================================
#  LANGUAGE DETECTION  (robust, deterministic; safe if langdetect missing)
# =============================================================================
def detect_language(text: str) -> str:
    if not _LANGDETECT_OK:
        return 'german'      # without langdetect we keep every row (no drops)
    if not isinstance(text, str):
        return 'unknown'
    s = text.strip()
    if len(s) < LANGDETECT_MIN_LEN:
        return 'unknown'
    try:
        results = detect_langs(s)
        if not results:
            return 'unknown'
        top = results[0]
        if top.prob < LANGDETECT_MIN_PROB:
            return 'uncertain'
        if top.lang == 'de':
            return 'german'
        if top.lang == 'en':
            return 'english'
        return 'other'
    except LangDetectException:
        return 'unknown'


# =============================================================================
#  PER-ROW CONFIDENCE SCORE (data quality 0..1)
# =============================================================================
ARTIFACT_CHARS = set('()[]{}〈〉⸢⸣⟨⟩<>𓉘𓊂𓍹𓍺')


def compute_confidence(row) -> float:
    raw_t   = row.get('raw_transliteration', '') or ''
    clean_t = row.get('clean_transliteration', '') or ''
    raw_g   = row.get('raw_german', '') or ''
    clean_g = row.get('clean_german', '') or ''

    if not clean_t.strip() or not clean_g.strip():
        return 0.0

    score = 1.0

    if len(raw_g) > 0:
        keep = len(clean_g) / len(raw_g)
        if   keep < 0.30: score -= 0.20
        elif keep < 0.50: score -= 0.10

    t_tok = clean_t.split()
    g_tok = clean_g.split()
    if len(t_tok) < MIN_TRANSLIT_TOKENS or len(g_tok) < MIN_GERMAN_TOKENS:
        score -= 0.30
    if len(t_tok) > MAX_TRANSLIT_TOKENS or len(g_tok) > MAX_GERMAN_TOKENS:
        score -= 0.20

    if t_tok and g_tok:
        ratio = max(len(t_tok), len(g_tok)) / min(len(t_tok), len(g_tok))
        if ratio > MAX_TOKEN_RATIO:
            score -= 0.20

    if any(c in clean_g for c in ARTIFACT_CHARS):
        score -= 0.30

    g_letters = sum(1 for c in clean_g if c.isalpha())
    if len(clean_g) > 0:
        density = g_letters / len(clean_g)
        if density < 0.40:
            score -= 0.20

    if 4 <= len(g_tok) <= 30:
        score += 0.05

    return float(max(0.0, min(1.0, score)))


# =============================================================================
#  CORPUS-LEVEL PROCESSING  (Table 4 -> clean -> drop-empty -> dedup ->
#  language filter -> confidence). Applied only to the parallel German corpus.
# =============================================================================
def apply_comprehensive_cleaning(df: "pd.DataFrame", cfg) -> "pd.DataFrame":
    df = df.copy()
    rt, ct = cfg.col_raw_translit, cfg.col_clean_translit
    rg, cg = cfg.col_raw_german,   cfg.col_clean_german

    for col in (rt, ct, rg, cg):
        if col not in df.columns:
            df[col] = ''
    df = df.fillna('')

    # ---- 1. PDF TABLE 4 FILTER (POS='/' + lKey missing -> drop). No-op if
    #         the source CSV has no POS column (local Stage-1 corpora).
    if '_pos' in df.columns:
        if '_lkey' in df.columns:
            mask = (df['_pos'].astype(str).str.strip() == '/') & (
                df['_lkey'].isna()
                | (df['_lkey'].astype(str).str.strip() == '')
                | (df['_lkey'].astype(str).str.lower() == 'nan')
            )
        else:
            mask = (df['_pos'].astype(str).str.strip() == '/')
        df = df[~mask].reset_index(drop=True)
    for c in ('_pos', '_lkey'):
        if c in df.columns:
            df = df.drop(columns=[c])

    # ---- 2. APPLY CLEANERS (regenerate clean cols from raw; fall back to the
    #         existing clean value when raw is empty so we never mass-drop).
    src_t = df[rt].astype(str)
    src_t = src_t.where(src_t.str.strip() != '', df[ct].astype(str))
    df[ct] = src_t.apply(clean_transliteration)

    src_g = df[rg].astype(str)
    src_g = src_g.where(src_g.str.strip() != '', df[cg].astype(str))
    df[cg] = src_g.apply(clean_german)

    # ---- 3. DROP rows whose cleaned columns are empty
    df = df[(df[ct].str.len() > 0) & (df[cg].str.len() > 0)].reset_index(drop=True)

    # ---- 4. DEDUPLICATE on the cleaned pair
    df = df.drop_duplicates(subset=[ct, cg]).reset_index(drop=True)

    # ---- 5. LANGUAGE DETECTION on clean_german
    df['language'] = df[cg].apply(detect_language)

    # ---- 6. DROP english + unknown (+ other/uncertain if configured)
    drop_set = {'english', 'unknown'}
    if DROP_OTHER_LANGUAGES:
        drop_set |= {'other', 'uncertain'}
    df = df[~df['language'].isin(drop_set)].reset_index(drop=True)

    # ---- 7. CONFIDENCE score (kept as a column; rows are not dropped unless
    #         DROP_LOW_CONFIDENCE is explicitly enabled)
    df['confidence'] = df.apply(compute_confidence, axis=1)
    if DROP_LOW_CONFIDENCE:
        df = df[df['confidence'] >= MIN_CONF_THRESHOLD].reset_index(drop=True)

    return df

print("Comprehensive cleaning rules ready.")


Comprehensive cleaning rules ready.


## 5. Load Corpus

In [7]:
def load_main_corpus(cfg: Config) -> pd.DataFrame:
    path = cfg.main_corpus_path
    if path is None or not Path(path).exists():
        log("No main corpus found - using seed corpus.", "WARN")
        return pd.DataFrame([
            {"raw_transliteration": "ḥtp-di-nsw",  "clean_transliteration": "ḥtp di nsw",
             "raw_german": "ein Opfer das der König gibt", "clean_german": "ein Opfer das der König gibt"},
            {"raw_transliteration": "nsw-bꞽtꞽ",    "clean_transliteration": "nsw bꞽtꞽ",
             "raw_german": "König von Ober und Unterägypten", "clean_german": "König von Ober und Unterägypten"},
            {"raw_transliteration": "sꜣ-rꜥ",        "clean_transliteration": "sꜣ rꜥ",
             "raw_german": "Sohn des Re",              "clean_german": "Sohn des Re"},
        ])

    path = Path(path)
    df = pd.read_csv(path).fillna("").copy()
    lowered = {str(column).strip().lower(): column for column in df.columns}

    if (
        cfg.col_raw_translit not in df.columns
        and "code" in lowered
        and any(name in lowered for name in ("phonetic_tla", "phonetic", "reading", "transliteration"))
    ):
        code_col = lowered["code"]
        phonetic_col = next(
            lowered[name]
            for name in ("phonetic_tla", "phonetic", "reading", "transliteration")
            if name in lowered
        )
        meaning_col = next(
            (lowered[name] for name in ("meaning", "translation", "gloss", "english") if name in lowered),
            None,
        )
        unicode_col = lowered.get("unicode")

        mapped = pd.DataFrame({
            cfg.col_raw_translit: df[code_col].astype(str),
            cfg.col_clean_translit: df[phonetic_col].astype(str),
            cfg.col_raw_german: df[meaning_col].astype(str) if meaning_col else "",
            cfg.col_clean_german: df[meaning_col].astype(str) if meaning_col else "",
            "gardiner_code": df[code_col].astype(str),
            "gardiner_phonetic_source": df[phonetic_col].astype(str),
        })
        if meaning_col:
            mapped["gardiner_meaning"] = df[meaning_col].astype(str)
        if unicode_col:
            mapped["gardiner_unicode"] = df[unicode_col].astype(str)
        df = mapped
        log(
            f"Main corpus remapped from {path.name} using code='{code_col}' and phonetic='{phonetic_col}'."
        )

    legacy = {
        "translit_raw": cfg.col_raw_translit, "raw_text": cfg.col_raw_translit,
        "translit_clean": cfg.col_clean_translit, "clean_text": cfg.col_clean_translit,
        "german": cfg.col_clean_german, "german_translation": cfg.col_clean_german,
        "raw_german_text": cfg.col_raw_german,
    }
    for src, tgt in legacy.items():
        if src in df.columns and tgt not in df.columns:
            df = df.rename(columns={src: tgt})
    for col in [cfg.col_raw_translit, cfg.col_clean_translit, cfg.col_raw_german, cfg.col_clean_german]:
        if col not in df.columns:
            df[col] = ""

    df = df.fillna("").copy()
    path_name = path.name.lower()

    if path_name == "gardiner_updated.csv":
        df = df[df[cfg.col_raw_translit].astype(str).str.strip() != ""].reset_index(drop=True)
    else:
        df = df[
            (df[cfg.col_clean_translit].astype(str).str.strip() != "") &
            (df[cfg.col_clean_german].astype(str).str.strip() != "")
        ].reset_index(drop=True)

    log(f"Loaded main corpus: {len(df)} rows from {path.name}")
    return df

DF = load_main_corpus(CFG)

# ── Apply comprehensive cleaning rules + processing (from Clean_dataset_comprehensive) ──
# Only the parallel German corpus is run through the comprehensive pipeline
# (regenerate clean cols from raw → drop-empty → dedup → language filter → confidence).
# The Gardiner lexicon path (gardiner_updated.csv) keeps its precomputed phonetic
# readings, so Stage 1 lookup behaviour is preserved exactly.
_main_corpus_name = Path(CFG.main_corpus_path).name.lower() if CFG.main_corpus_path else ""
if _main_corpus_name != "gardiner_updated.csv":
    _rows_before_clean = len(DF)
    DF = apply_comprehensive_cleaning(DF, CFG)
    log(f"Comprehensive cleaning applied: {_rows_before_clean} → {len(DF)} rows "
        f"(language/confidence columns added).")

for col in [CFG.col_raw_translit, CFG.col_clean_translit, CFG.col_raw_german, CFG.col_clean_german]:
    DF[col] = DF[col].astype(str).apply(clean_text)

DF["translit_norm"] = DF[CFG.col_clean_translit].apply(normalize_translit)
mask = DF[CFG.col_raw_german].astype(str).str.strip() == ""
DF.loc[mask, CFG.col_raw_german] = DF.loc[mask, CFG.col_clean_german]

main_corpus_name = Path(CFG.main_corpus_path).name.lower() if CFG.main_corpus_path else ""
if main_corpus_name == "gardiner_updated.csv":
    DF = DF[
        (DF[CFG.col_raw_translit].str.len() > 0) &
        (DF[CFG.col_clean_german].str.len() > 0)
    ].reset_index(drop=True)
else:
    DF = DF[
        (DF["translit_norm"].str.len() > 0) &
        (DF[CFG.col_clean_german].str.len() > 0)
    ].reset_index(drop=True)

print(f"Corpus rows : {len(DF)}")
print(DF[[CFG.col_raw_translit, "translit_norm", CFG.col_clean_german]].head(5).to_string(index=False))


[2026-05-29 14:49:59] [INFO] Main corpus remapped from gardiner_updated.csv using code='code' and phonetic='phonetic_tla'.
[2026-05-29 14:49:59] [INFO] Loaded main corpus: 867 rows from gardiner_updated.csv
Corpus rows : 863
raw_transliteration translit_norm               clean_german
                 A1             s                 Seated man
                A10                       Man holding an oar
                A11               Man with scepter and crook
                A12                  Man with bow and quiver
                A13                      Man with arms bound


## 6. Stage 1 — Gardiner Decoder

Builds a Gardiner-code → phonetic-fragment lexicon from the first available source:
`gardiner_lexicon.json` → `gardiner_updated.csv` → `Gardiner_Sign_List.csv`

Unknown codes fall back to `stage1_unknown_strategy` (default: lowercased code string).

In [8]:
# ── Stage 1: Probabilistic Beam Decoder ──────────────────────────────────────

import csv
import math
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple


# ── Config ───────────────────────────────────────────────────────────────────

BEAM_SIZE = 6
N_BEST = 3
ALPHA = 1.0
BETA = 1.35
GAMMA = 0.55
EPSILON_PENALTY = -1.5
MAX_CONFIDENCE = 0.995
SINGLE_PATH_CONFIDENCE = 0.95
ALIGNMENT_LM_WEIGHT = 0.75
ALIGNMENT_MATCH_WEIGHT = 1.0
ALIGNMENT_OVERSHOOT_WEIGHT = 1.5
ALIGNMENT_REMAINDER_WEIGHT = 1.0
LM_BOOTSTRAP_CONFIDENCE = 0.55
ROLE_TRAIN_CONFIDENCE = 0.60
MIN_WEIGHT_TUNING_ROWS = 24
_STAGE1_EPS = 1e-12
START_TOKEN = "<S>"
END_TOKEN = "</S>"

GARDINER_RE = re.compile(r"^[A-Z]+[0-9]+[A-Z]*$")
EXACT_OVERRIDE_TOKEN_RE = re.compile(r"^(?:[A-Z]+[0-9]+[A-Z]*|[0-9]+)$")


# ── Data ─────────────────────────────────────────────────────────────────────

@dataclass(frozen=True)
class GardinerCandidate:
    fragment: str
    weight: float


@dataclass(frozen=True)
class DecoderWeights:
    alpha: float = ALPHA
    beta: float = BETA
    gamma: float = GAMMA


@dataclass
class BeamHypothesis:
    score: float
    path_by_code: List[str]
    previous_fragment: Optional[str]
    epsilon_count: int = 0


@dataclass
class AlignmentHypothesis:
    score: float
    path_by_code: List[str]
    previous_fragment: Optional[str]
    target_pos: int
    epsilon_count: int = 0


@dataclass
class AlignmentResult:
    codes: List[str]
    target: str
    path_by_code: List[str]
    aligned_fragments: List[str]
    confidence: float
    edit_distance: int
    path_score: float
    unknown_codes: List[str]


@dataclass
class Stage1DecodeResult:
    codes: List[str]
    fragments: List[str]
    spaced_phonetics: str
    unknown_codes: List[str]
    confidence: float = 1.0
    path_score: float = 0.0
    alternatives: List[Dict[str, Any]] = field(default_factory=list)
    path_by_code: List[str] = field(default_factory=list)
    decoder_metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class BigramLanguageModel:
    bigram_counts: Dict[str, Counter]
    unigram_counts: Counter
    vocab_size: int
    source: str
    sequence_count: int

    def log_prob(self, prev_fragment: Optional[str], current_fragment: str) -> float:
        current_fragment = normalize_fragment(current_fragment)
        if not current_fragment:
            return 0.0

        prev_fragment = START_TOKEN if not prev_fragment else normalize_fragment(prev_fragment)
        prev_total = sum(self.bigram_counts.get(prev_fragment, Counter()).values())
        numerator = self.bigram_counts.get(prev_fragment, Counter()).get(current_fragment, 0) + 1.0
        denominator = prev_total + self.vocab_size
        return safe_log(numerator / max(denominator, 1.0))


# ── Normalization helpers ────────────────────────────────────────────────────

def normalize_gardiner_code(code: Any) -> str:
    return str(code).strip().upper()


def normalize_fragment(fragment: Any) -> str:
    if fragment is None:
        return ""

    raw_text = str(fragment).strip()
    if not raw_text:
        return ""

    if GARDINER_RE.fullmatch(raw_text.upper()):
        return raw_text.lower()

    text = normalize_translit(raw_text)
    text = re.sub(r"\s+", "", text).strip()
    return text


def safe_log(value: float) -> float:
    return math.log(max(float(value), _STAGE1_EPS))


def sigmoid(value: float) -> float:
    clipped = max(-60.0, min(60.0, value))
    return 1.0 / (1.0 + math.exp(-clipped))


def confidence_from_margin(best_score: float, second_score: Optional[float]) -> float:
    if second_score is None:
        return SINGLE_PATH_CONFIDENCE
    return min(MAX_CONFIDENCE, sigmoid(best_score - second_score))


def strip_empty_fragments(path_by_code: Sequence[str]) -> List[str]:
    return [normalize_fragment(fragment) for fragment in path_by_code if normalize_fragment(fragment)]


# ── Phonology post-processing ────────────────────────────────────────────────

def collapse_repeated_phonemes(tokens: Sequence[str]) -> List[str]:
    """
    Remove immediately adjacent IDENTICAL fragments only.
    Complement removal is handled downstream by remove_phonetic_complements().

    FIX (Bug 3): removed the boundary-merging branch (previous[-1] == token[0])
    that was incorrectly fusing e.g. mn+nw -> mnw and pr+rẼ -> prẼ.
    """
    cleaned: List[str] = []

    for token in tokens:
        token = normalize_fragment(token)
        if not token:
            continue

        if cleaned and token == cleaned[-1]:
            continue   # skip exact duplicate

        cleaned.append(token)

    return cleaned


# ── Lexicon loading + merging ────────────────────────────────────────────────

def _parse_weighted_candidate(raw_item: Any) -> Tuple[str, float]:
    if isinstance(raw_item, dict):
        fragment = normalize_fragment(raw_item.get("fragment", raw_item.get("phonetic", "")))
        weight = raw_item.get("weight", 1.0)
        return fragment, max(float(weight or 0.0), 0.0)

    if isinstance(raw_item, str):
        text = raw_item.strip()
        if ":" in text:
            fragment_text, weight_text = text.rsplit(":", 1)
            try:
                return normalize_fragment(fragment_text), max(float(weight_text), 0.0)
            except ValueError:
                pass
        return normalize_fragment(text), 1.0

    return normalize_fragment(raw_item), 1.0


def _merge_candidate(raw_map: Dict[str, List[Tuple[str, float]]], code: Any, fragment: Any, weight: float = 1.0) -> None:
    code_norm = normalize_gardiner_code(code)
    if not code_norm:
        return
    raw_map[code_norm].append((normalize_fragment(fragment), max(float(weight), 0.0)))


def _load_json_lexicon(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    merged: Dict[str, List[Tuple[str, float]]] = defaultdict(list)

    for code, raw_value in payload.items():
        allow_epsilon = False
        epsilon_weight = 0.1

        if isinstance(raw_value, dict):
            allow_epsilon = bool(raw_value.get("allow_epsilon", False))
            epsilon_weight = float(raw_value.get("epsilon_weight", 0.1) or 0.1)
            candidate_values = raw_value.get("candidates", raw_value.get("fragments", []))
        elif isinstance(raw_value, list):
            candidate_values = raw_value
        else:
            candidate_values = [raw_value]

        seen_any = False
        for item in candidate_values:
            fragment, weight = _parse_weighted_candidate(item)
            _merge_candidate(merged, code, fragment, weight or 1.0)
            seen_any = True

        code_norm = normalize_gardiner_code(code)
        if allow_epsilon and (not seen_any or all(fragment != "" for fragment, _ in merged[code_norm])):
            _merge_candidate(merged, code_norm, "", epsilon_weight)

    return merged


def _source_allows_empty_rows(path: Path) -> bool:
    name = path.name.lower()
    return name in {"gardiner_updated.csv", "gardiner_lexicon.csv"}


def _row_looks_like_header(row: Sequence[str]) -> bool:
    normalized = [clean_text(cell).lower() for cell in row]
    if not normalized:
        return False
    return any(
        token in {"code", "gardiner", "sign", "sign_code", "sign code", "phonetic", "reading", "phonetic_tla", "type", "class", "category"}
        for token in normalized
    )


def _load_csv_lexicon(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    merged: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    allow_empty_rows = _source_allows_empty_rows(path)

    with path.open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.reader(handle))

    if not rows:
        return merged

    if _row_looks_like_header(rows[0]):
        header = [clean_text(cell).lower() for cell in rows[0]]
        field_to_idx = {field: idx for idx, field in enumerate(header)}
        code_idx = next(
            (field_to_idx[name] for name in ("code", "gardiner", "sign", "sign_code", "sign code") if name in field_to_idx),
            None,
        )

        # Prefer the TLA-normalized phonetic column when available so known-code lookup
        # stays deterministic and does not merge multiple parallel transliteration columns.
        if "phonetic_tla" in field_to_idx:
            fragment_idxs = [field_to_idx["phonetic_tla"]]
        elif "phonetic" in field_to_idx:
            fragment_idxs = [field_to_idx["phonetic"]]
        else:
            fragment_idxs = [
                field_to_idx[name]
                for name in ("fragment", "reading", "transliteration", "phoneme", "value")
                if name in field_to_idx
            ]

        role_idx = next(
            (field_to_idx[name] for name in ("role", "type", "class", "category") if name in field_to_idx),
            None,
        )
        data_rows = rows[1:]
    else:
        code_idx = 0 if len(rows[0]) >= 1 else None
        fragment_idxs = [1] if len(rows[0]) >= 2 else []
        role_idx = 2 if len(rows[0]) >= 3 else None
        data_rows = rows

    if code_idx is None:
        return merged

    for row in data_rows:
        if not row or code_idx >= len(row):
            continue

        code = row[code_idx]
        if not str(code).strip():
            continue

        row_fragments = set()

        for idx in fragment_idxs:
            if idx >= len(row):
                continue
            value = str(row[idx] or "").strip()
            if not value:
                continue

            # FIX (Bug 9): added \s to split pattern so space-separated values
            # like "mn m n" are handled correctly.
            for raw_fragment in re.split(r"[|,;/\s]+", value):
                fragment = normalize_fragment(raw_fragment)
                if not fragment or fragment in row_fragments:
                    continue
                row_fragments.add(fragment)
                _merge_candidate(merged, code, fragment, 1.0)

        role_value = row[role_idx] if role_idx is not None and role_idx < len(row) else ""
        role_label = _canonical_role_label(role_value)

        if not row_fragments and (allow_empty_rows or role_label == "DET"):
            _merge_candidate(merged, code, "", 1.0)

    return merged

def _finalize_lexicon(raw_map: Dict[str, List[Tuple[str, float]]]) -> Dict[str, List[GardinerCandidate]]:
    lexicon: Dict[str, List[GardinerCandidate]] = {}

    for code, items in raw_map.items():
        grouped: Dict[str, float] = defaultdict(float)
        order: List[str] = []

        for fragment, weight in items:
            if fragment not in grouped:
                order.append(fragment)
            grouped[fragment] += max(float(weight), 0.0)

        total_weight = sum(grouped.values())
        if total_weight <= 0.0:
            total_weight = float(len(order) or 1)
            for fragment in order:
                grouped[fragment] = 1.0

        lexicon[code] = [
            GardinerCandidate(fragment=fragment, weight=grouped[fragment] / total_weight)
            for fragment in order
        ]

    return lexicon


def build_gardiner_candidates(cfg: Config) -> Dict[str, List[GardinerCandidate]]:
    primary_raw: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    secondary_raw: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    fallback_raw: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    loaded_sources: List[str] = []
    source_specs = [
        ("primary", cfg.gardiner_updated_path),
        ("secondary", cfg.gardiner_json_path),
        ("fallback", cfg.gardiner_table_path),
    ]

    for tier, path in source_specs:
        if path is None:
            continue

        path = Path(path)
        if not path.exists():
            continue

        if path.suffix.lower() == ".json":
            partial = _load_json_lexicon(path)
        else:
            partial = _load_csv_lexicon(path)

        if tier == "primary":
            target_map = primary_raw
        elif tier == "secondary":
            target_map = secondary_raw
        else:
            target_map = fallback_raw

        for code, candidates in partial.items():
            target_map[code].extend(candidates)

        loaded_sources.append(f"{path.name} ({tier})")

    if not primary_raw and not secondary_raw and not fallback_raw:
        raise FileNotFoundError("No Gardiner lexicon source could be loaded.")

    merged_candidates: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    primary_codes = set(primary_raw)
    secondary_added = 0
    fallback_added = 0

    for code, candidates in primary_raw.items():
        merged_candidates[code].extend(candidates)

    for code, candidates in secondary_raw.items():
        if code in primary_codes:
            continue
        merged_candidates[code].extend(candidates)
        secondary_added += 1

    covered_codes = set(merged_candidates)
    for code, candidates in fallback_raw.items():
        if code in covered_codes:
            continue
        merged_candidates[code].extend(candidates)
        fallback_added += 1

    lexicon = _finalize_lexicon(merged_candidates)
    log(f"Loaded Stage 1 lexicon from: {', '.join(loaded_sources)}")
    log(
        "Lexicon merge policy: gardiner_updated.csv wins per code; "
        "gardiner_lexicon.json fills gaps; Gardiner_Sign_List.csv fills remaining missing codes."
    )
    log(
        f"Lexicon coverage: {len(primary_codes)} primary codes "
        f"| {secondary_added} json-gapfill codes | {fallback_added} table-gapfill codes | {len(lexicon)} total"
    )
    return lexicon

def _canonical_role_label(raw_role: Any, reading_hint: Any = "") -> Optional[str]:
    role_text = clean_text(str(raw_role or "")).lower()
    reading = normalize_fragment(reading_hint)

    if "det" in role_text:
        return "DET"

    if any(marker in role_text for marker in ("phon", "ideo", "logo", "syllab", "alphabet")):
        return "PHON"

    if not role_text and reading:
        return "PHON"

    return None


def load_gardiner_role_supervision(path: Optional[Path]) -> Tuple[Dict[str, str], Dict[str, Any]]:
    metadata: Dict[str, Any] = {
        "source": None,
        "status": "missing",
        "explicit_role_rows": 0,
        "resolved_codes": 0,
        "det_count": 0,
        "phon_count": 0,
        "conflict_count": 0,
    }

    if path is None:
        return {}, metadata

    path = Path(path)
    if not path.exists():
        metadata["status"] = "not_found"
        return {}, metadata

    with path.open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.reader(handle))

    if not rows:
        metadata["source"] = path.name
        metadata["status"] = "empty"
        return {}, metadata

    first_row = [clean_text(cell).lower() for cell in rows[0]]
    header_like = any(
        token in {"code", "gardiner", "sign", "sign_code", "sign code", "role", "type", "class", "category"}
        or "phon" in token
        or "det" in token
        or "ideo" in token
        for token in first_row
    )

    def _find_index(candidates: Sequence[str], default: Optional[int] = None) -> Optional[int]:
        for idx, token in enumerate(first_row):
            if token in candidates:
                return idx
        return default

    if header_like:
        code_idx = _find_index(("code", "gardiner", "sign", "sign_code", "sign code"), 0)
        reading_idx = _find_index(("phonetic", "reading", "phonetic_tla", "transliteration", "value"), 1)
        role_idx = _find_index(("role", "type", "class", "category"), len(first_row) - 1)
        data_rows = rows[1:]
    else:
        code_idx = 0
        reading_idx = 1 if len(rows[0]) > 1 else None
        role_idx = 2 if len(rows[0]) > 2 else None
        data_rows = rows

    role_votes: Dict[str, Counter] = defaultdict(Counter)

    for row in data_rows:
        if not row or code_idx is None or code_idx >= len(row):
            continue

        code = normalize_gardiner_code(row[code_idx])
        if not GARDINER_RE.fullmatch(code):
            continue

        reading_value = row[reading_idx] if reading_idx is not None and reading_idx < len(row) else ""
        role_value = row[role_idx] if role_idx is not None and role_idx < len(row) else ""

        label = _canonical_role_label(role_value, reading_value)
        if label is None:
            continue

        role_votes[code][label] += 1
        metadata["explicit_role_rows"] += 1

    labels: Dict[str, str] = {}
    for code, votes in role_votes.items():
        if len(votes) > 1:
            metadata["conflict_count"] += 1
        labels[code] = votes.most_common(1)[0][0]

    metadata["source"] = path.name
    metadata["status"] = "loaded"
    metadata["resolved_codes"] = len(labels)
    metadata["det_count"] = sum(label == "DET" for label in labels.values())
    metadata["phon_count"] = sum(label == "PHON" for label in labels.values())
    return labels, metadata


def get_stage1_candidates(code: Any, *, unknown_strategy: Optional[str] = None) -> Tuple[List[GardinerCandidate], bool]:
    unknown_strategy = unknown_strategy or CFG.stage1_unknown_strategy
    code_norm = normalize_gardiner_code(code)
    candidates = GARDINER_CANDIDATES.get(code_norm)

    if candidates:
        return candidates, False

    if unknown_strategy == "epsilon":
        fallback_fragment = ""
    elif unknown_strategy == "placeholder":
        fallback_fragment = f"<unk:{code_norm.lower()}>"
    else:
        fallback_fragment = code_norm.lower()

    return [GardinerCandidate(fragment=fallback_fragment, weight=1.0)], True


# ── Bigram LM ────────────────────────────────────────────────────────────────

def build_bigram_language_model_from_sequences(
    fragment_sequences: Sequence[Sequence[str]],
    *,
    source: str,
) -> BigramLanguageModel:
    bigram_counts: Dict[str, Counter] = defaultdict(Counter)
    unigram_counts: Counter = Counter()
    kept_sequences = 0

    for sequence in fragment_sequences:
        tokens = [normalize_fragment(token) for token in sequence]
        tokens = [token for token in tokens if token]
        if not tokens:
            continue

        kept_sequences += 1
        padded = [START_TOKEN] + tokens + [END_TOKEN]

        for token in padded[1:]:
            unigram_counts[token] += 1

        for previous, current in zip(padded, padded[1:]):
            bigram_counts[previous][current] += 1

    vocab_size = max(len(unigram_counts) + 1, 2)
    return BigramLanguageModel(
        bigram_counts=bigram_counts,
        unigram_counts=unigram_counts,
        vocab_size=vocab_size,
        source=source,
        sequence_count=kept_sequences,
    )


def bigram_log_prob(prev_fragment: Optional[str], current_fragment: str, lm: Optional[BigramLanguageModel] = None) -> float:
    lm = STAGE1_LM if lm is None else lm
    return lm.log_prob(prev_fragment, current_fragment)


def collect_transliteration_fallback_sequences(df: pd.DataFrame, lexicon: Dict[str, List[GardinerCandidate]]) -> List[List[str]]:
    sequences: List[List[str]] = []

    for seq in df["translit_norm"].astype(str):
        tokens = [normalize_fragment(token) for token in seq.split()]
        tokens = [token for token in tokens if token and not GARDINER_RE.fullmatch(token.upper())]
        if tokens:
            sequences.append(tokens)

    for candidates in lexicon.values():
        for candidate in candidates:
            if candidate.fragment.strip():
                sequences.append([candidate.fragment])

    return sequences


def load_explicit_parallel_training_rows() -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    candidate_path = find_input_file(
        "stage1_training_rows.csv",
        "gardiner_training_rows.csv",
        "gardiner_parallel.csv",
    )

    if candidate_path is None or not Path(candidate_path).exists():
        return rows

    with Path(candidate_path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            raw_codes = str(
                row.get("gardiner_codes", row.get("codes", row.get("raw_transliteration", "")))
            ).strip()
            raw_target = str(
                row.get("clean_transliteration", row.get("target", row.get("phonetic", "")))
            ).strip()
            codes = tokenize_gardiner_text(raw_codes)
            target = normalize_translit(raw_target)
            if codes and target and not is_gardiner_input(target):
                rows.append({"codes": codes, "target": target, "source": Path(candidate_path).name})

    return rows


def _build_unicode_to_code_map(cfg: Config) -> Dict[str, str]:
    path = cfg.gardiner_updated_path
    if path is None or not Path(path).exists():
        return {}

    unicode_to_codes: Dict[str, set] = defaultdict(set)

    try:
        df = pd.read_csv(path, encoding="utf-8-sig").fillna("")
    except Exception:
        return {}

    lowered = {str(column).strip().lower(): column for column in df.columns}
    code_col = lowered.get("code")
    unicode_col = lowered.get("unicode")
    if code_col is None or unicode_col is None:
        return {}

    for _, row in df.iterrows():
        code = normalize_gardiner_code(row.get(code_col, ""))
        glyph = clean_text(str(row.get(unicode_col, "")))
        if code and glyph:
            unicode_to_codes[glyph].add(code)

    resolved: Dict[str, str] = {}
    for glyph, codes in unicode_to_codes.items():
        if len(codes) == 1:
            resolved[glyph] = next(iter(codes))
    return resolved


def _tokenize_gardiner_or_unicode(text: str, unicode_to_code: Dict[str, str]) -> List[str]:
    codes = tokenize_gardiner_text(text)
    if codes:
        return codes

    glyph_codes: List[str] = []
    for char in clean_text(text):
        if char.isspace() or char in {"-", ":", "/", "\\", "[", "]", "(", ")", "{", "}", "*", "?", ".", ",", ";", '"', "'"}:
            continue
        mapped = unicode_to_code.get(char)
        if mapped is None:
            continue
        glyph_codes.append(mapped)
    return glyph_codes


def load_exact_sequence_overrides(cfg: Config) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, Any]]:
    overrides: Dict[str, Dict[str, Any]] = {}
    metadata: Dict[str, Any] = {
        "source": None,
        "usable_rows": 0,
        "unique_sequences": 0,
        "glyph_converted_rows": 0,
        "single_code_rows_skipped": 0,
    }

    candidate_paths: List[Path] = []
    if cfg.main_corpus_path is not None:
        main_corpus_path = Path(cfg.main_corpus_path)
        if main_corpus_path.exists():
            candidate_paths.append(main_corpus_path)

    for filename in (
        "cleaned_output_with_language.csv",
        "clean_egyptian.csv",
        "stage1_training_rows.csv",
        "gardiner_training_rows.csv",
        "gardiner_parallel.csv",
    ):
        found = find_input_file(filename)
        if found is None:
            continue
        path = Path(found)
        if path.exists() and path not in candidate_paths:
            candidate_paths.append(path)

    if not candidate_paths:
        metadata["status"] = "missing"
        return overrides, metadata

    unicode_to_code = _build_unicode_to_code_map(cfg)
    sequence_votes: Dict[str, Counter] = defaultdict(Counter)
    raw_examples: Dict[Tuple[str, str], Dict[str, str]] = {}
    usable_rows_by_source: Dict[str, int] = defaultdict(int)

    for candidate_path in candidate_paths:
        with candidate_path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                raw_codes = str(
                    row.get(
                        "gardiner_tokens",
                        row.get(
                            "clean_gardiner",
                            row.get(
                                "raw_gardiner",
                                row.get(
                                    "gardiner_codes",
                                    row.get(
                                        "codes",
                                        row.get(
                                            "gardiner",
                                            row.get("raw_transliteration", row.get("clean_transliteration", "")),
                                        ),
                                    ),
                                ),
                            ),
                        ),
                    )
                ).strip()
                raw_target = str(
                    row.get(
                        "transliteration",
                        row.get(
                            "clean_transliteration",
                            row.get("target", row.get("phonetic", row.get("raw_transliteration", ""))),
                        ),
                    )
                ).strip()

                tokenized_codes = tokenize_gardiner_text(raw_codes)
                codes = tokenized_codes or _tokenize_gardiner_or_unicode(raw_codes, unicode_to_code)
                target = normalize_translit(raw_target)

                if not codes or not target or is_gardiner_input(target):
                    continue

                if len(codes) < 2:
                    metadata["single_code_rows_skipped"] += 1
                    continue

                if not tokenized_codes:
                    metadata["glyph_converted_rows"] += 1

                sequence_key = " ".join(codes)
                sequence_votes[sequence_key][target] += 1
                raw_examples[(sequence_key, target)] = {
                    "raw_transliteration": clean_text(
                        str(
                            row.get(
                                "raw_transliteration",
                                row.get("transliteration", row.get("clean_transliteration", target)),
                            )
                        )
                    ),
                    "clean_transliteration": target,
                    "clean_german": clean_text(
                        str(row.get("translation", row.get("clean_german", row.get("raw_german", ""))))
                    ),
                    "source": candidate_path.name,
                }
                metadata["usable_rows"] += 1
                usable_rows_by_source[candidate_path.name] += 1

    for sequence_key, votes in sequence_votes.items():
        target, count = votes.most_common(1)[0]
        example = raw_examples.get((sequence_key, target), {})
        overrides[sequence_key] = {
            "target": target,
            "count": int(count),
            "source": example.get("source", "multiple"),
            "raw_transliteration": example.get("raw_transliteration", target),
            "clean_german": example.get("clean_german", ""),
        }

    metadata["source"] = ", ".join(path.name for path in candidate_paths)
    metadata["source_path"] = [str(path) for path in candidate_paths]
    metadata["usable_rows_by_source"] = dict(usable_rows_by_source)
    metadata["unique_sequences"] = len(overrides)
    metadata["status"] = "ready"
    metadata["probe_hits"] = {
        "E13 R4": "E13 R4" in overrides,
        "E11 R4": "E11 R4" in overrides,
        "G43 X1 U1 D4 S29 S42": "G43 X1 U1 D4 S29 S42" in overrides,
        "U6 X1 V8 N35 N33A": "U6 X1 V8 N35 N33A" in overrides,
    }
    log(
        f"Exact sequence overrides: {metadata['usable_rows']} usable rows, "
        f"{metadata['unique_sequences']} unique code sequences from {metadata['source']} "
        f"({metadata['glyph_converted_rows']} rows converted from raw glyphs)."
    )
    return overrides, metadata


def load_gardiner_updated_parallel_rows(cfg: Config) -> List[Dict[str, Any]]:
    """
    Convert gardiner_updated.csv into Stage 1 parallel rows so LM bootstrapping
    always sees real Gardiner-code -> phonetic supervision.
    """
    rows: List[Dict[str, Any]] = []
    path = cfg.gardiner_updated_path

    if path is None or not Path(path).exists():
        log("gardiner_updated.csv not found - skipping Stage 1 parallel injection.", "WARN")
        return rows

    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception as exc:
        log(f"Failed to read gardiner_updated.csv: {exc}", "WARN")
        return rows

    df.columns = [str(column).strip().lower() for column in df.columns]

    code_col = next(
        (
            column
            for column in df.columns
            if column in ("code", "gardiner", "sign_code", "sign code", "gardiner_code")
        ),
        None,
    )
    phonetic_cols = [
        column
        for column in ("phonetic_tla", "phonetic", "reading", "transliteration", "fragment", "value")
        if column in df.columns
    ]

    if code_col is None or not phonetic_cols:
        log(
            f"gardiner_updated.csv: could not find code/phonetic columns. Found columns: {list(df.columns)}",
            "WARN",
        )
        return rows

    log(
        "gardiner_updated.csv -> using "
        f"code='{code_col}', phonetic fallback order={phonetic_cols}"
    )

    skipped_det = 0
    seen_pairs = set()

    for _, row in df.iterrows():
        code_raw = str(row.get(code_col, "")).strip()
        if not code_raw:
            continue

        code = normalize_gardiner_code(code_raw)
        if not GARDINER_RE.fullmatch(code):
            continue

        phonetic_raw = ""
        for column in phonetic_cols:
            candidate = str(row.get(column, "")).strip()
            if candidate and candidate.lower() not in {"nan", "none", "-", "—"}:
                phonetic_raw = candidate
                break

        if not phonetic_raw:
            skipped_det += 1
            continue

        phonetic_norm = normalize_translit(phonetic_raw)
        if not phonetic_norm:
            skipped_det += 1
            continue

        pair = (code, phonetic_norm)
        if pair in seen_pairs:
            continue
        seen_pairs.add(pair)

        rows.append({
            "codes": [code],
            "target": phonetic_norm,
            "source": "gardiner_updated",
        })

    log(
        f"gardiner_updated parallel rows: {len(rows)} loaded, {skipped_det} determinatives/empty skipped."
    )
    return rows


def extract_parallel_gardiner_rows(df: pd.DataFrame) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    raw_col = CFG.col_raw_translit
    clean_col = CFG.col_clean_translit

    for _, row in df.iterrows():
        raw_text = clean_text(str(row.get(raw_col, "")))
        clean_text_value = clean_text(str(row.get(clean_col, "")))
        raw_norm = normalize_translit(raw_text) if raw_text else ""
        clean_norm = normalize_translit(clean_text_value) if clean_text_value else ""

        if raw_text and is_gardiner_input(raw_text) and clean_norm and not is_gardiner_input(clean_norm):
            rows.append({"codes": tokenize_gardiner_text(raw_text), "target": clean_norm, "source": "main_corpus_raw"})
            continue

        if clean_text_value and is_gardiner_input(clean_text_value) and raw_norm and not is_gardiner_input(raw_norm):
            rows.append({"codes": tokenize_gardiner_text(clean_text_value), "target": raw_norm, "source": "main_corpus_clean"})

    return rows


# ── Alignment for LM bootstrapping ───────────────────────────────────────────

def _target_match_transition(
    fragment: str,
    target: str,
    target_pos: int,
    *,
    match_weight: float = ALIGNMENT_MATCH_WEIGHT,
    overshoot_weight: float = ALIGNMENT_OVERSHOOT_WEIGHT,
) -> Tuple[int, float]:
    fragment = normalize_fragment(fragment)
    if not fragment:
        return target_pos, 0.0

    fragment_length = len(fragment)
    target_slice = target[target_pos : target_pos + fragment_length]
    consumed_target_len = len(target_slice)
    overshoot = max((target_pos + fragment_length) - len(target), 0)

    if consumed_target_len == 0:
        edit_cost = fragment_length
    else:
        edit_cost = char_edit_distance(fragment[:consumed_target_len], target_slice)
    edit_cost += overshoot

    new_target_pos = min(target_pos + fragment_length, len(target))
    score_delta = -(match_weight * edit_cost) - (overshoot_weight * overshoot)
    return new_target_pos, score_delta


def _prune_alignment_beam(hypotheses: List[AlignmentHypothesis], beam_size: int) -> List[AlignmentHypothesis]:
    hypotheses.sort(
        key=lambda hypothesis: (
            hypothesis.score,
            hypothesis.target_pos,
            -hypothesis.epsilon_count,
            "".join(hypothesis.path_by_code),
        ),
        reverse=True,
    )
    return hypotheses[:beam_size]


def _finalize_alignment_hypotheses(
    hypotheses: List[AlignmentHypothesis],
    target: str,
    *,
    remainder_weight: float = ALIGNMENT_REMAINDER_WEIGHT,
) -> List[Tuple[float, AlignmentHypothesis]]:
    finalized: List[Tuple[float, AlignmentHypothesis]] = []

    for hypothesis in hypotheses:
        remainder = max(len(target) - hypothesis.target_pos, 0)
        final_score = hypothesis.score - (remainder_weight * remainder)
        finalized.append((final_score, hypothesis))

    finalized.sort(
        key=lambda item: (
            item[0],
            item[1].target_pos,
            -item[1].epsilon_count,
            "".join(item[1].path_by_code),
        ),
        reverse=True,
    )
    return finalized


def align_codes_to_clean_target(
    codes: Sequence[str],
    target: str,
    *,
    beam_size: int = BEAM_SIZE,
    lm: Optional[BigramLanguageModel] = None,
    lm_weight: float = ALIGNMENT_LM_WEIGHT,
) -> AlignmentResult:
    normalized_codes = [normalize_gardiner_code(code) for code in codes if str(code).strip()]
    normalized_target = normalize_fragment(target)
    unknown_codes: List[str] = []

    beam: List[AlignmentHypothesis] = [
        AlignmentHypothesis(score=0.0, path_by_code=[], previous_fragment=None, target_pos=0, epsilon_count=0)
    ]

    for code in normalized_codes:
        candidates, is_unknown = get_stage1_candidates(code)
        if is_unknown:
            unknown_codes.append(code)

        expansions: List[AlignmentHypothesis] = []

        for hypothesis in beam:
            for candidate in candidates:
                fragment = normalize_fragment(candidate.fragment)
                new_target_pos, match_score = _target_match_transition(fragment, normalized_target, hypothesis.target_pos)
                lm_score = 0.0
                if lm is not None and fragment:
                    lm_score = lm.log_prob(hypothesis.previous_fragment, fragment)

                new_score = (
                    hypothesis.score
                    + safe_log(candidate.weight)
                    + match_score
                    + (lm_weight * lm_score)
                )
                if not fragment:
                    new_score += EPSILON_PENALTY

                expansions.append(
                    AlignmentHypothesis(
                        score=new_score,
                        path_by_code=hypothesis.path_by_code + [fragment],
                        previous_fragment=hypothesis.previous_fragment if not fragment else fragment,
                        target_pos=new_target_pos,
                        epsilon_count=hypothesis.epsilon_count + int(not fragment),
                    )
                )

        beam = _prune_alignment_beam(expansions, beam_size=beam_size) if expansions else beam

    finalized = _finalize_alignment_hypotheses(beam, normalized_target)

    if not finalized:
        fallback_path = []
        for code in normalized_codes:
            candidates, _ = get_stage1_candidates(code)
            fallback_path.append(next((candidate.fragment for candidate in candidates if candidate.fragment.strip()), ""))

        assembled = "".join(strip_empty_fragments(fallback_path))
        edit_distance = char_edit_distance(assembled, normalized_target)
        return AlignmentResult(
            codes=normalized_codes,
            target=normalized_target,
            path_by_code=fallback_path,
            aligned_fragments=strip_empty_fragments(fallback_path),
            confidence=0.0,
            edit_distance=edit_distance,
            path_score=-1e9,
            unknown_codes=unknown_codes,
        )

    best_score, best_hypothesis = finalized[0]
    second_score = finalized[1][0] if len(finalized) > 1 else None
    aligned_fragments = strip_empty_fragments(best_hypothesis.path_by_code)
    assembled = "".join(aligned_fragments)
    edit_distance = char_edit_distance(assembled, normalized_target)
    max_len = max(len(assembled), len(normalized_target), 1)
    fit_quality = max(0.0, 1.0 - (edit_distance / max_len))
    margin_quality = confidence_from_margin(best_score, second_score)
    confidence = round((0.7 * fit_quality) + (0.3 * margin_quality), 4)

    return AlignmentResult(
        codes=normalized_codes,
        target=normalized_target,
        path_by_code=best_hypothesis.path_by_code,
        aligned_fragments=aligned_fragments,
        confidence=confidence,
        edit_distance=edit_distance,
        path_score=round(best_score, 6),
        unknown_codes=unknown_codes,
    )


def build_stage1_bigram_lm(
    df: pd.DataFrame,
    lexicon: Dict[str, List[GardinerCandidate]],
) -> Tuple[BigramLanguageModel, Dict[str, Any], List[AlignmentResult], List[Dict[str, Any]]]:
    explicit_parallel_rows = load_explicit_parallel_training_rows()
    discovered_parallel_rows = extract_parallel_gardiner_rows(df)
    primary_parallel_rows = explicit_parallel_rows or discovered_parallel_rows
    gardiner_updated_rows = load_gardiner_updated_parallel_rows(CFG)
    parallel_rows = primary_parallel_rows + gardiner_updated_rows

    log(
        "Parallel rows summary -> "
        f"explicit={len(explicit_parallel_rows)}, "
        f"corpus_discovered={len(discovered_parallel_rows)}, "
        f"gardiner_updated={len(gardiner_updated_rows)}, "
        f"total={len(parallel_rows)}"
    )

    alignment_rows: List[AlignmentResult] = []
    training_sequences: List[List[str]] = []
    metadata: Dict[str, Any] = {
        "parallel_row_count": len(parallel_rows),
        "explicit_parallel_rows": len(explicit_parallel_rows),
        "corpus_parallel_rows": len(discovered_parallel_rows),
        "gardiner_updated_parallel_rows": len(gardiner_updated_rows),
        "lm_source": "translit_token_fallback",
    }

    if parallel_rows:
        first_pass = [
            align_codes_to_clean_target(row["codes"], row["target"], beam_size=BEAM_SIZE, lm=None, lm_weight=0.0)
            for row in parallel_rows
        ]

        bootstrap_sequences = [
            row.aligned_fragments
            for row in first_pass
            if row.confidence >= LM_BOOTSTRAP_CONFIDENCE and row.aligned_fragments
        ]

        metadata["bootstrap_alignment_rows"] = len(first_pass)
        metadata["bootstrap_sequence_count"] = len(bootstrap_sequences)

        if bootstrap_sequences:
            bootstrap_lm = build_bigram_language_model_from_sequences(
                bootstrap_sequences,
                source="aligned_bootstrap_pass1",
            )
            second_pass = [
                align_codes_to_clean_target(
                    row["codes"],
                    row["target"],
                    beam_size=BEAM_SIZE,
                    lm=bootstrap_lm,
                    lm_weight=ALIGNMENT_LM_WEIGHT,
                )
                for row in parallel_rows
            ]

            final_sequences = [
                row.aligned_fragments
                for row in second_pass
                if row.confidence >= LM_BOOTSTRAP_CONFIDENCE and row.aligned_fragments
            ]

            if final_sequences:
                alignment_rows = second_pass
                training_sequences = final_sequences
                metadata["lm_source"] = "parallel_aligned_bootstrap"
                metadata["aligned_sequence_count"] = len(final_sequences)
        else:
            metadata["aligned_sequence_count"] = 0

    if not training_sequences:
        training_sequences = collect_transliteration_fallback_sequences(df, lexicon)
        metadata["aligned_sequence_count"] = 0
        metadata["fallback_sequence_count"] = len(training_sequences)

    lm = build_bigram_language_model_from_sequences(
        training_sequences,
        source=metadata["lm_source"],
    )
    metadata["lm_sequence_count"] = lm.sequence_count

    return lm, metadata, alignment_rows, parallel_rows


# ── Decoder helpers ──────────────────────────────────────────────────────────

def tokenize_gardiner_text(text: str) -> List[str]:
    cleaned = clean_text(text)
    cleaned = re.sub(r"[^A-Za-z0-9\s]", " ", cleaned)
    return [normalize_gardiner_code(token) for token in cleaned.split() if token]


def is_gardiner_input(text: str) -> bool:
    tokens = tokenize_gardiner_text(text)
    return bool(tokens) and all(GARDINER_RE.fullmatch(token) for token in tokens)


def has_exact_sequence_override_input(text: str) -> bool:
    tokens = tokenize_gardiner_text(text)
    if len(tokens) < 2:
        return False
    if not all(EXACT_OVERRIDE_TOKEN_RE.fullmatch(token) for token in tokens):
        return False
    sequence_key = " ".join(tokens)
    return sequence_key in globals().get("EXACT_SEQUENCE_OVERRIDES", {})


def role_log_prior_for_candidate(code: str, candidate: GardinerCandidate, position: int, total_len: int) -> float:
    predictor = globals().get("predict_role_distribution")
    if predictor is None:
        role_priors = {"DET": 0.35, "PHON": 0.65}
    else:
        role_priors = predictor(code, position, total_len)

    role = "DET" if not candidate.fragment.strip() else "PHON"
    return safe_log(role_priors.get(role, 0.5))


def _prune_decode_beam(hypotheses: List[BeamHypothesis], beam_size: int) -> List[BeamHypothesis]:
    hypotheses.sort(
        key=lambda hypothesis: (
            hypothesis.score,
            -hypothesis.epsilon_count,
            "".join(hypothesis.path_by_code),
        ),
        reverse=True,
    )
    return hypotheses[:beam_size]


def _decode_fit_quality(path_by_code: Sequence[str], target: str) -> float:
    fragments = collapse_repeated_phonemes(strip_empty_fragments(path_by_code))
    assembled = "".join(fragments)
    normalized_target = normalize_fragment(target)
    max_len = max(len(assembled), len(normalized_target), 1)
    return max(0.0, 1.0 - (char_edit_distance(assembled, normalized_target) / max_len))


def _serialize_beam_hypothesis(hypothesis: BeamHypothesis) -> Dict[str, Any]:
    fragments = collapse_repeated_phonemes(strip_empty_fragments(hypothesis.path_by_code))
    return {
        "path_by_code": list(hypothesis.path_by_code),
        "fragments": fragments,
        "spaced_phonetics": " ".join(fragments),
        "score": round(hypothesis.score, 6),
        "epsilon_count": hypothesis.epsilon_count,
    }


def beam_search_decode(
    codes: Sequence[str],
    *,
    beam_size: int = BEAM_SIZE,
    decoder_weights: Optional[DecoderWeights] = None,
    return_n_best: int = N_BEST,
) -> Stage1DecodeResult:
    weights = STAGE1_DECODER_WEIGHTS if decoder_weights is None else decoder_weights
    normalized_codes = [normalize_gardiner_code(code) for code in codes if str(code).strip()]
    beam: List[BeamHypothesis] = [BeamHypothesis(score=0.0, path_by_code=[], previous_fragment=None, epsilon_count=0)]
    unknown_codes: List[str] = []
    total_len = len(normalized_codes)
    import inspect
    try:
        _candidate_params = inspect.signature(get_stage1_candidates).parameters
        _supports_position_args = "position" in _candidate_params and "total_len" in _candidate_params
    except Exception:
        _supports_position_args = False

    code_entries = []
    for position, code in enumerate(normalized_codes):
        if _supports_position_args:
            candidates, is_unknown = get_stage1_candidates(
                code,
                position=position,
                total_len=total_len,
            )
        else:
            candidates, is_unknown = get_stage1_candidates(code)
        if is_unknown:
            unknown_codes.append(code)
        code_entries.append({
            "code": code,
            "position": position,
            "candidates": candidates,
            "is_unknown": is_unknown,
        })

    strict_known_lookup = bool(getattr(CFG, "stage1_strict_known_code_lookup", False))
    anchor_known_codes = bool(
        strict_known_lookup
        or (
            getattr(CFG, "stage1_anchor_known_codes_when_unknown_present", True)
            and unknown_codes
        )
    )
    anchored_known_codes: List[str] = []

    def _collapse_candidates(candidate_list: Sequence[GardinerCandidate]) -> List[GardinerCandidate]:
        grouped: Dict[str, float] = {}
        order: List[str] = []

        for candidate in candidate_list:
            fragment = normalize_fragment(candidate.fragment)
            if fragment not in grouped:
                grouped[fragment] = 0.0
                order.append(fragment)
            grouped[fragment] += max(float(candidate.weight), 0.0)

        if not order:
            return []

        best_fragment = max(order, key=lambda fragment: grouped[fragment])
        best_weight = grouped[best_fragment]
        return [GardinerCandidate(fragment=best_fragment, weight=max(best_weight, 1e-6))]

    for position, entry in enumerate(code_entries):
        code = entry["code"]
        is_unknown = entry["is_unknown"]
        candidates = list(entry["candidates"])

        if anchor_known_codes and not is_unknown:
            collapsed = _collapse_candidates(candidates)
            if collapsed:
                candidates = collapsed
                anchored_known_codes.append(code)

        expansions: List[BeamHypothesis] = []

        for hypothesis in beam:
            for candidate in candidates:
                fragment = normalize_fragment(candidate.fragment)
                lexicon_log_prob = safe_log(candidate.weight)
                lm_log_prob = 0.0 if not fragment else bigram_log_prob(hypothesis.previous_fragment, fragment)
                role_log_prob = role_log_prior_for_candidate(code, candidate, position, total_len)

                candidate_score = (
                    weights.alpha * lexicon_log_prob
                    + weights.beta * lm_log_prob
                    + weights.gamma * role_log_prob
                )
                if not fragment:
                    candidate_score += EPSILON_PENALTY

                expansions.append(
                    BeamHypothesis(
                        score=hypothesis.score + candidate_score,
                        path_by_code=hypothesis.path_by_code + [fragment],
                        previous_fragment=hypothesis.previous_fragment if not fragment else fragment,
                        epsilon_count=hypothesis.epsilon_count + int(not fragment),
                    )
                )

        beam = _prune_decode_beam(expansions, beam_size=beam_size) if expansions else beam

    best_hypothesis = beam[0] if beam else BeamHypothesis(score=0.0, path_by_code=[], previous_fragment=None, epsilon_count=0)
    second_score = beam[1].score if len(beam) > 1 else None
    top_beam_paths = [_serialize_beam_hypothesis(hypothesis) for hypothesis in beam[: max(return_n_best, 1)]]

    cleaned_fragments = collapse_repeated_phonemes(strip_empty_fragments(best_hypothesis.path_by_code))
    alternatives = [
        {
            "fragments": collapse_repeated_phonemes(strip_empty_fragments(hypothesis.path_by_code)),
            "path_by_code": list(hypothesis.path_by_code),
            "spaced_phonetics": " ".join(collapse_repeated_phonemes(strip_empty_fragments(hypothesis.path_by_code))),
            "score": round(hypothesis.score, 6),
            "epsilon_count": hypothesis.epsilon_count,
        }
        for hypothesis in beam[1 : return_n_best + 1]
    ]

    return Stage1DecodeResult(
        codes=normalized_codes,
        fragments=cleaned_fragments,
        spaced_phonetics=" ".join(cleaned_fragments),
        unknown_codes=unknown_codes,
        confidence=round(confidence_from_margin(best_hypothesis.score, second_score), 4),
        path_score=round(best_hypothesis.score, 6),
        alternatives=alternatives,
        path_by_code=best_hypothesis.path_by_code,
        decoder_metadata={
            "beam_size": beam_size,
            "beam_path_count": len(beam),
            "beam_top_paths": top_beam_paths,
            "weights": {
                "alpha": weights.alpha,
                "beta": weights.beta,
                "gamma": weights.gamma,
            },
            "lm_source": STAGE1_LM.source,
            "known_codes_anchored": anchor_known_codes,
            "anchored_known_codes": anchored_known_codes,
            "strict_known_code_lookup": strict_known_lookup,
        },
    )

def decode_gardiner_codes(
    codes: Sequence[str],
    *,
    beam_size: int = BEAM_SIZE,
    decoder_weights: Optional[DecoderWeights] = None,
) -> Stage1DecodeResult:
    return beam_search_decode(codes, beam_size=beam_size, decoder_weights=decoder_weights)


def tune_decoder_weights(
    alignment_rows: Sequence[AlignmentResult],
    *,
    min_rows: int = MIN_WEIGHT_TUNING_ROWS,
) -> Tuple[DecoderWeights, Dict[str, Any]]:
    usable_rows = [
        row
        for row in alignment_rows
        if row.confidence >= ROLE_TRAIN_CONFIDENCE and row.codes and row.target
    ]

    if len(usable_rows) < min_rows:
        return STAGE1_DECODER_WEIGHTS, {
            "status": "default",
            "reason": f"not_enough_parallel_rows ({len(usable_rows)})",
        }

    dev_size = max(6, len(usable_rows) // 5)
    dev_rows = usable_rows[-dev_size:]

    candidate_grid = [
        DecoderWeights(alpha=0.9, beta=1.1, gamma=0.35),
        DecoderWeights(alpha=1.0, beta=1.2, gamma=0.45),
        DecoderWeights(alpha=1.0, beta=1.35, gamma=0.55),
        DecoderWeights(alpha=1.1, beta=1.4, gamma=0.55),
        DecoderWeights(alpha=1.15, beta=1.5, gamma=0.65),
        DecoderWeights(alpha=0.85, beta=1.55, gamma=0.45),
    ]

    best_weights = STAGE1_DECODER_WEIGHTS
    best_score = -1.0

    for weights in candidate_grid:
        scores = []
        for row in dev_rows:
            decoded = beam_search_decode(row.codes, beam_size=BEAM_SIZE, decoder_weights=weights)
            predicted = "".join(decoded.fragments)
            target = normalize_fragment(row.target)
            max_len = max(len(predicted), len(target), 1)
            fit = max(0.0, 1.0 - (char_edit_distance(predicted, target) / max_len))
            scores.append(fit)

        mean_score = sum(scores) / max(len(scores), 1)
        if mean_score > best_score:
            best_score = mean_score
            best_weights = weights

    return best_weights, {
        "status": "tuned",
        "dev_rows": len(dev_rows),
        "mean_fit_quality": round(best_score, 4),
    }


def decode_exact_sequence_override(codes: Sequence[str]) -> Optional[Stage1DecodeResult]:
    normalized_codes = [normalize_gardiner_code(code) for code in codes if str(code).strip()]
    if not normalized_codes:
        return None

    # Single-code inputs should use the authoritative sign lexicon directly.
    # Exact-sequence overrides are reserved for multi-code sequences where a
    # memorized parallel row can genuinely help.
    if len(normalized_codes) < 2:
        return None

    sequence_key = " ".join(normalized_codes)
    match = globals().get("EXACT_SEQUENCE_OVERRIDES", {}).get(sequence_key)
    if match is None:
        return None

    # FIX (Bug 8): normalize target through normalize_translit so MDC notation
    # in training data is converted to proper Egyptological transliteration.
    raw_target = clean_text(str(match.get("target", "")))
    normalized_target = normalize_translit(raw_target)
    fragments = [token for token in normalized_target.split() if token]
    lm_object = globals().get("STAGE1_LM")
    raw_transliteration = clean_text(str(match.get("raw_transliteration", "")))
    display_transliteration = raw_transliteration or " ".join(fragments)

    return Stage1DecodeResult(
        codes=normalized_codes,
        fragments=fragments,
        spaced_phonetics=display_transliteration,
        unknown_codes=[],
        confidence=0.995,
        path_score=round(safe_log(match.get("count", 1) + 1.0), 6),
        alternatives=[],
        path_by_code=[],
        decoder_metadata={
            "lm_source": getattr(lm_object, "source", None),
            "exact_sequence_override": True,
            "override_source": match.get("source"),
            "override_count": match.get("count", 1),
            "raw_transliteration": raw_transliteration,
            "clean_german": match.get("clean_german"),
        },
    )

def decode_gardiner_text(text: str) -> Stage1DecodeResult:
    tokens = tokenize_gardiner_text(text)
    exact_match = decode_exact_sequence_override(tokens)
    if exact_match is not None:
        return exact_match

    codes = [token for token in tokens if GARDINER_RE.fullmatch(token)]

    if not codes:
        fallback_fragments = [token for token in clean_text(text).split() if token]
        return Stage1DecodeResult(
            codes=[],
            fragments=fallback_fragments,
            spaced_phonetics=" ".join(fallback_fragments),
            unknown_codes=[],
        )

    return decode_gardiner_codes(codes)


# ── Runtime objects ──────────────────────────────────────────────────────────

GARDINER_CANDIDATES = build_gardiner_candidates(CFG)
EXACT_SEQUENCE_OVERRIDES, EXACT_SEQUENCE_METADATA = load_exact_sequence_overrides(CFG)
GARDINER_ROLE_SUPERVISION, GARDINER_ROLE_METADATA = load_gardiner_role_supervision(CFG.gardiner_table_path)
GARDINER_ROLE_METADATA["lexicon_covered_codes"] = sum(
    1 for code in GARDINER_ROLE_SUPERVISION if code in GARDINER_CANDIDATES
)
STAGE1_LM, STAGE1_LM_METADATA, STAGE1_ALIGNMENT_ROWS, STAGE1_PARALLEL_ROWS = build_stage1_bigram_lm(
    DF,
    GARDINER_CANDIDATES,
)
UNIGRAM_COUNTS = Counter(
    {
        token: count
        for token, count in STAGE1_LM.unigram_counts.items()
        if token not in {START_TOKEN, END_TOKEN}
    }
)
VOCAB_SIZE = STAGE1_LM.vocab_size
BIGRAM_COUNTS = STAGE1_LM.bigram_counts
STAGE1_DECODER_WEIGHTS = DecoderWeights()
STAGE1_WEIGHT_TUNING = {"status": "pending"}

print(
    f"Stage 1 ready: {len(GARDINER_CANDIDATES)} codes | "
    f"LM source={STAGE1_LM.source} | sequences={STAGE1_LM.sequence_count} | beam={BEAM_SIZE}"
)
print("Exact sequence metadata:", EXACT_SEQUENCE_METADATA)
for probe in ["E13 R4", "E11 R4", "G43 X1 U1 D4 S29 S42", "U6 X1 V8 N35 N33A"]:
    print(f"Override probe [{probe}] ->", EXACT_SEQUENCE_OVERRIDES.get(probe))

required_override_probes = ["E13 R4", "G43 X1 U1 D4 S29 S42"]
missing_override_probes = [probe for probe in required_override_probes if probe not in EXACT_SEQUENCE_OVERRIDES]
if missing_override_probes:
    # FIX (Bug 6): was raise RuntimeError — downgraded to a warning.
    # These probes are dataset-specific; other datasets should not crash.
    log(
        f"Override probe sequences not found in training data: {missing_override_probes}. "
        "This is normal when using a different dataset — Stage 1 will still run correctly. "
        f"Loaded metadata: {EXACT_SEQUENCE_METADATA}",
        "WARN",
    )

print("Role metadata:", GARDINER_ROLE_METADATA)
print("LM metadata:", STAGE1_LM_METADATA)


[2026-05-29 14:50:00] [INFO] Loaded Stage 1 lexicon from: gardiner_updated.csv (primary), gardiner_lexicon.json (secondary), Gardiner_Sign_List.csv (fallback)
[2026-05-29 14:50:00] [INFO] Lexicon merge policy: gardiner_updated.csv wins per code; gardiner_lexicon.json fills gaps; Gardiner_Sign_List.csv fills remaining missing codes.
[2026-05-29 14:50:00] [INFO] Lexicon coverage: 819 primary codes | 0 json-gapfill codes | 5356 table-gapfill codes | 6175 total
[2026-05-29 14:50:05] [INFO] Exact sequence overrides: 61249 usable rows, 55488 unique code sequences from gardiner_updated.csv, cleaned_output_with_language.csv, clean_egyptian.csv (8166 rows converted from raw glyphs).
[2026-05-29 14:50:05] [INFO] gardiner_updated.csv -> using code='code', phonetic fallback order=['phonetic_tla', 'phonetic']
[2026-05-29 14:50:05] [INFO] gardiner_updated parallel rows: 563 loaded, 216 determinatives/empty skipped.
[2026-05-29 14:50:05] [INFO] Parallel rows summary -> explicit=0, corpus_discovered=5

In [9]:
# ── CELL 2: Determinative ML Classifier ──────────────────────────────────────

from sklearn.ensemble import RandomForestClassifier
import numpy as np


ROLE_LABELS = ("DET", "PHON")
ROLE_UNCERTAINTY_MARGIN = 0.10
ROLE_DECISION_THRESHOLD = 0.55
ROLE_FALLBACK_PRIOR = {"DET": 0.35, "PHON": 0.65}


# ── Feature Extraction ───────────────────────────────────────────────────────

def extract_features(code, position, total_len, lexicon=None, unigram_counts=None):
    lexicon = GARDINER_CANDIDATES if lexicon is None else lexicon
    unigram_counts = UNIGRAM_COUNTS if unigram_counts is None else unigram_counts

    code = normalize_gardiner_code(code)
    cands = lexicon.get(code, [])

    phonetic_mass = sum(c.weight for c in cands if c.fragment.strip())
    det_mass = sum(c.weight for c in cands if not c.fragment.strip())
    ambiguity = float(len(cands))
    has_empty = float(any(not c.fragment.strip() for c in cands))

    entropy = -sum(
        c.weight * np.log(max(c.weight, 1e-12))
        for c in cands
    ) if cands else 0.0

    freq = sum(unigram_counts.get(c.fragment, 0) for c in cands if c.fragment.strip())
    max_phon_weight = max((c.weight for c in cands if c.fragment.strip()), default=0.0)
    max_det_weight = max((c.weight for c in cands if not c.fragment.strip()), default=0.0)
    position_norm = position / max(total_len - 1, 1)
    final_position = float(position == max(total_len - 1, 0))

    return np.array([
        phonetic_mass,
        det_mass,
        ambiguity,
        has_empty,
        entropy,
        freq,
        max_phon_weight,
        max_det_weight,
        position_norm,
        final_position,
    ], dtype=float)


def _weak_role_label(code, position, total_len, lexicon):
    cands = lexicon.get(normalize_gardiner_code(code), [])
    if not cands:
        return "PHON"

    phonetic_mass = sum(c.weight for c in cands if c.fragment.strip())
    det_mass = sum(c.weight for c in cands if not c.fragment.strip())

    if det_mass <= 0.0:
        return "PHON"

    if phonetic_mass <= 0.0:
        return "DET"

    near_boundary = position >= max(total_len - 2, 0)
    det_dominant = det_mass >= (phonetic_mass * 0.9)
    return "DET" if near_boundary and det_dominant else "PHON"


def _training_examples_from_alignments(lexicon, unigram_counts):
    examples = []

    for row in globals().get("STAGE1_ALIGNMENT_ROWS", []):
        if row.confidence < ROLE_TRAIN_CONFIDENCE:
            continue
        if len(row.codes) != len(row.path_by_code):
            continue

        total_len = len(row.codes)
        for position, (code, fragment) in enumerate(zip(row.codes, row.path_by_code)):
            label = "DET" if not normalize_fragment(fragment) else "PHON"
            examples.append(
                (
                    extract_features(code, position, total_len, lexicon, unigram_counts),
                    label,
                )
            )

    return examples


def _training_examples_from_role_supervision(lexicon, unigram_counts):
    examples = []
    labels = globals().get("GARDINER_ROLE_SUPERVISION", {})

    for code, label in sorted(labels.items()):
        if code not in lexicon:
            continue

        total_len = 5
        positions = [0, 1, 2, 3, 4, 3, 4] if label == "DET" else [0, 1, 2, 3, 4]
        for position in positions:
            examples.append(
                (
                    extract_features(code, position, total_len, lexicon, unigram_counts),
                    label,
                )
            )

    return examples


def _training_examples_from_weak_lexicon(lexicon, unigram_counts, skip_codes=None):
    examples = []
    skip_codes = {normalize_gardiner_code(code) for code in (skip_codes or set())}
    synthetic_len = 5

    for code in sorted(lexicon):
        if code in skip_codes:
            continue
        for position in range(synthetic_len):
            examples.append(
                (
                    extract_features(code, position, synthetic_len, lexicon, unigram_counts),
                    _weak_role_label(code, position, synthetic_len, lexicon),
                )
            )

    return examples


# ── Training ─────────────────────────────────────────────────────────────────

def train_det_classifier(lexicon, df, unigram_counts):
    X, y = [], []

    explicit_examples = _training_examples_from_role_supervision(lexicon, unigram_counts)
    aligned_examples = _training_examples_from_alignments(lexicon, unigram_counts)
    weak_examples = _training_examples_from_weak_lexicon(
        lexicon,
        unigram_counts,
        skip_codes=globals().get("GARDINER_ROLE_SUPERVISION", {}).keys(),
    )

    for feat, label in (explicit_examples + aligned_examples + weak_examples):
        X.append(feat)
        y.append(label)

    globals()["ROLE_TRAINING_METADATA"] = {
        "explicit_examples": len(explicit_examples),
        "alignment_examples": len(aligned_examples),
        "weak_examples": len(weak_examples),
        "total_examples": len(X),
        "explicit_role_codes": len(globals().get("GARDINER_ROLE_SUPERVISION", {})),
    }

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced_subsample",
    )

    model.fit(np.array(X), y)
    return model


# ── Train model ──────────────────────────────────────────────────────────────

DET_MODEL = train_det_classifier(
    GARDINER_CANDIDATES,
    DF,
    UNIGRAM_COUNTS
)

print("✅ DET classifier trained")
print("Role training metadata:", ROLE_TRAINING_METADATA)


# ── Prediction API ───────────────────────────────────────────────────────────

def predict_role_distribution(code, position, total_len, lexicon=None, unigram_counts=None):
    lexicon = GARDINER_CANDIDATES if lexicon is None else lexicon
    unigram_counts = UNIGRAM_COUNTS if unigram_counts is None else unigram_counts

    feat = extract_features(
        code,
        position,
        total_len,
        lexicon,
        unigram_counts,
    )

    raw = dict(zip(DET_MODEL.classes_, DET_MODEL.predict_proba([feat])[0]))
    p_det = float(raw.get("DET", 0.0))
    p_phon = float(raw.get("PHON", max(1.0 - p_det, 0.0)))

    if abs(p_det - p_phon) < ROLE_UNCERTAINTY_MARGIN:
        p_det = min(p_det, ROLE_FALLBACK_PRIOR["DET"])
        p_phon = max(p_phon, ROLE_FALLBACK_PRIOR["PHON"])

    total = max(p_det + p_phon, 1e-12)
    return {
        "DET": p_det / total,
        "PHON": p_phon / total,
    }


def predict_role(code, position, total_len):
    priors = predict_role_distribution(code, position, total_len)
    if priors["DET"] >= ROLE_DECISION_THRESHOLD and priors["DET"] > priors["PHON"]:
        return "DET"
    return "PHON"


STAGE1_DECODER_WEIGHTS, STAGE1_WEIGHT_TUNING = tune_decoder_weights(STAGE1_ALIGNMENT_ROWS)
print("Decoder weights:", STAGE1_DECODER_WEIGHTS)
print("Weight tuning:", STAGE1_WEIGHT_TUNING)


✅ DET classifier trained
Role training metadata: {'explicit_examples': 40589, 'alignment_examples': 1157, 'weak_examples': 800, 'total_examples': 42546, 'explicit_role_codes': 6904}
Decoder weights: DecoderWeights(alpha=0.9, beta=1.1, gamma=0.35)
Weight tuning: {'status': 'tuned', 'dev_rows': 231, 'mean_fit_quality': 1.0}


## 5.5 Post-Processing Rules

Implements Sir Alan Gardiner's phonetic complement rules (Egyptian Grammar §§43–52).

| Rule | Characters handled | Action |
|---|---|---|
| **Phonetic complement removal** | Map-based (biliterals + triliterals) + suffix heuristic fallback | Drop complement fragments |
| **Duplicate removal** | — | Drop consecutive identical fragments |
| **Bracket stripping** | `[] () {} <> ⌈⌉ ⌊⌋ ⟨⟩` | Remove (content kept) |
| **Separator → space** | `- . , =` | Replace with space |
| **Whitespace normalisation** | — | Collapse spaces, strip edges |

### Complement removal — two methods
1. **Map-based** (precise): looks up each fragment against `PHONETIC_COMPLEMENT_MAP`.
   Handles cases the suffix heuristic cannot, e.g. `nw + n` (n is the *first* consonant of nw).
2. **Suffix heuristic** (fallback): if the following fragment is a proper suffix of the previous
   one and ≤ 3 chars, it is treated as a complement.


In [10]:
# ── Post-Processing Rules (Section 5.5) ─────────────────────────────────────
# Implements Sir Alan Gardiner's phonetic complement rules
# (Egyptian Grammar §§43-52).

import re


# ── Phonetic Complement Map ──────────────────────────────────────────────────
#
# BILITERAL XY:
#   complements are usually X or Y
#
# TRILITERAL XYZ:
#   complements are usually Y, Z, or YZ
#
# Examples:
#   nw + n        -> n is complement of nw
#   ꜥnḫ + n + ḫ   -> both are complements
#   ḥtp + t + p   -> both are complements

PHONETIC_COMPLEMENT_MAP = {

    # ── Biliterals ──────────────────────────────────────────────────────────
    "mn": ["n", "m"],
    "pr": ["r", "p"],
    "nb": ["b", "n"],
    "nw": ["n", "w"],
    "sn": ["n", "s"],
    "ms": ["s", "m"],
    "ḥr": ["r", "ḥ"],
    "wn": ["n", "w"],
    "mw": ["w", "m"],
    "sw": ["w", "s"],
    "ꜥn": ["n", "ꜥ"],
    "ḫt": ["t", "ḫ"],
    "nḏ": ["ḏ", "n"],
    "ḥm": ["m", "ḥ"],
    "wꜣ": ["ꜣ", "w"],
    "mꜣ": ["ꜣ", "m"],
    "sꜣ": ["ꜣ", "s"],
    "ḥꜣ": ["ꜣ", "ḥ"],
    "bꜣ": ["ꜣ", "b"],
    "kꜣ": ["ꜣ", "k"],
    "dꜣ": ["ꜣ", "d"],
    "rꜥ": ["ꜥ", "r"],
    "ꜥḥ": ["ḥ", "ꜥ"],
    "šs": ["s", "š"],
    "ḥb": ["b", "ḥ"],
    "rd": ["d", "r"],
    "ḫm": ["m", "ḫ"],
    "tp": ["p", "t"],
    "ḏr": ["r", "ḏ"],
    "nṯ": ["ṯ", "n"],
    "ḥq": ["q", "ḥ"],
    "šd": ["d", "š"],
    "km": ["m", "k"],
    "gs": ["s", "g"],
    "tm": ["m", "t"],
    "wp": ["p", "w"],
    "db": ["b", "d"],
    "šm": ["m", "š"],
    "nḥ": ["ḥ", "n"],
    "ꜥš": ["š", "ꜥ"],
    "ḏd": ["d", "ḏ"],
    "ḥn": ["n", "ḥ"],
    "pḥ": ["ḥ", "p"],
    "wr": ["r", "w"],
    "mr": ["r", "m"],
    "mḥ": ["ḥ", "m"],
    "nẫ": ["ẫ", "n"],

    # ── NEW biliterals (Bug 7) ─────────────────────────────────────────
    "ꜥw": ["w", "ꜥ"],
    "ꞟn": ["n"],
    "ꜣw": ["w", "ꜣ"],
    "qd": ["d", "q"],
    "ḫd": ["d", "ḫ"],
    "šw": ["w", "š"],
    "ẖn": ["n", "ẖ"],
    "ḏꜣ": ["ꜣ", "ḏ"],
    "ꜣb": ["b", "ꜣ"],
    "sḥ": ["ḥ", "s"],
    "ḫꜥ": ["ꜥ", "ḫ"],
    "ꜥḏ": ["ḏ", "ꜥ"],

    # ── Triliterals ─────────────────────────────────────────────────────────
    "ꜥnḫ": ["n", "ḫ", "nḫ"],
    "ḥtp": ["t", "p", "tp"],
    "nṯr": ["ṯ", "r", "ṯr"],
    "ḫpr": ["p", "r", "pr"],
    "nfr": ["f", "r", "fr"],
    "wꜣs": ["ꜣ", "s", "ꜣs"],
    "ḏsr": ["s", "r", "sr"],
    "ḥwt": ["w", "t", "wt"],
    "šms": ["m", "s", "ms"],
    "skr": ["k", "r", "kr"],
    "nḥm": ["ḥ", "m", "ḥm"],
    "ḫtm": ["t", "m", "tm"],
    "ꜥbꜣ": ["b", "ꜣ", "bꜣ"],
    "ꜥšꜣ": ["š", "ꜣ", "šꜣ"],
    "msḫ": ["s", "ḫ", "sḫ"],
    "nḏm": ["ḏ", "m", "ḏm"],
    "šsp": ["s", "p", "sp"],
    "ꜥnḏ": ["n", "ḏ", "nḏ"],
    "ḫpš": ["p", "š", "pš"],
    "ḥfn": ["f", "n", "fn"],
    "wdn": ["d", "n", "dn"],
    "ꜥqr": ["q", "r", "qr"],

    # ── NEW triliterals (Bug 7) ──────────────────────────────────────────────
    "wꜣḥ": ["ꜣ", "ḥ", "ꜣḥ"],
    "mꜣꜥ": ["ꜣ", "ꜥ"],
    "ḥnq": ["n", "q", "nq"],
    "nẖt": ["ẖ", "t", "ẖt"],
    "ḫfꜥ": ["f", "ꜥ", "fꜥ"],
    "šnꜥ": ["n", "ꜥ", "nꜥ"],
}


# ── Duplicate Removal ────────────────────────────────────────────────────────

def remove_adjacent_duplicates(fragments):
    """
    Remove immediately repeated adjacent fragments.

    Example:
        ["nfr", "nfr", "f"] -> ["nfr", "f"]

    Helps suppress seq2seq repetition artifacts.
    """

    result = []

    for frag in fragments:

        frag = frag.strip()

        if not frag:
            continue

        if not result or frag != result[-1]:
            result.append(frag)

    return result


# ── Punctuation Cleaning ─────────────────────────────────────────────────────

def clean_phonetic_punctuation(text):
    """
    Strip editorial punctuation and normalize separators.

    Removed:
        [ ] ( ) { } < > ⌈⌉ ⌊⌋ ⟨⟩

    Converted to spaces:
        - . , =

    Examples:
        "[bn] j =tw =f" -> "bn j tw f"
        "ḥtp-ḥnm.w"    -> "ḥtp ḥnm w"
    """

    # Remove brackets
    text = re.sub(
        r"[\[\](){}\<\>\u2308\u2309\u230a\u230b\u27e8\u27e9]",
        " ",
        text
    )

    # Convert separators to spaces
    text = re.sub(r"[-.,=]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ── Complement Removal ───────────────────────────────────────────────────────

def remove_phonetic_complements(fragments):
    """
    Remove phonetic complements using:

    1. Explicit Gardiner complement map
    2. Conservative suffix heuristic fallback

    Examples:
        ["nw", "n"]           -> ["nw"]
        ["ꜥnḫ", "n", "ḫ"]     -> ["ꜥnḫ"]
        ["ḥtp", "t", "p"]     -> ["ḥtp"]
    """

    if not fragments:
        return fragments

    result = [fragments[0]]

    i = 1

    while i < len(fragments):

        frag = fragments[i].strip()
        prev = result[-1].strip()

        # ── Method 1: explicit Gardiner map ────────────────────────────────

        if prev in PHONETIC_COMPLEMENT_MAP:
            if frag in PHONETIC_COMPLEMENT_MAP[prev]:
                i += 1
                continue

        # ── Method 2: conservative suffix heuristic ───────────────────────
        #
        # Prevents excessive false positives.
        #
        # Conditions:
        #   - fragment shorter than previous
        #   - fragment length <= 2
        #   - true suffix
        #   - not identical

        if (
            len(frag) < len(prev)
            and len(frag) <= 2
            and frag != prev
            and prev.endswith(frag)
        ):
            i += 1
            continue

        result.append(frag)
        i += 1

    return result


# ── Main Post-Processing Pipeline ────────────────────────────────────────────

def apply_phonetic_post_processing(fragments):
    """
    Full phonetic cleanup pipeline.

    Pipeline:
    ----------
    1. Remove adjacent duplicates
    2. Clean punctuation
    3. Re-tokenise cleaned fragments
    4. Remove phonetic complements
    5. Remove duplicates AGAIN
    6. Drop empty fragments
    7. Final whitespace normalization
    """

    if not fragments:
        return ""

    # ── Step 1: initial duplicate removal ─────────────────────────────────
    fragments = remove_adjacent_duplicates(fragments)

    # ── Step 2: punctuation cleanup ───────────────────────────────────────
    fragments = [clean_phonetic_punctuation(f) for f in fragments]

    # ── Step 3: re-tokenisation ───────────────────────────────────────────
    retokenised = []

    for frag in fragments:
        retokenised.extend(frag.split())

    fragments = retokenised

    # ── Step 4: complement removal ────────────────────────────────────────
    fragments = remove_phonetic_complements(fragments)

    # ── Step 5: SECOND duplicate removal ─────────────────────────────────
    #
    # Complement stripping can create new adjacent duplicates.
    #
    # Example:
    #   ["ḥtp","t","ḥtp","p"]
    #       -> ["ḥtp","ḥtp"]
    #       -> ["ḥtp"]
    #
    fragments = remove_adjacent_duplicates(fragments)

    # ── Step 6: remove empties ────────────────────────────────────────────
    fragments = [f.strip() for f in fragments if f.strip()]

    # ── Step 7: final normalization ──────────────────────────────────────
    result = " ".join(fragments)
    result = re.sub(r"\s+", " ", result).strip()

    return result


# ── Self-Test ────────────────────────────────────────────────────────────────

_TESTS = [

    # ── Biliteral complements ─────────────────────────────────────────────
    (["pr", "r"],                  "pr",           "pr + r"),
    (["mn", "n"],                  "mn",           "mn + n"),
    (["nb", "b"],                  "nb",           "nb + b"),
    (["nw", "n"],                  "nw",           "nw + n"),

    # ── Triliteral complements ───────────────────────────────────────────
    (["ꜥnḫ", "n", "ḫ"],            "ꜥnḫ",          "ꜥnḫ + n + ḫ"),
    (["ḥtp", "t", "p"],            "ḥtp",          "ḥtp + t + p"),
    (["nfr", "f", "r"],            "nfr",          "nfr + f + r"),

    # ── Must NOT remove ──────────────────────────────────────────────────
    (["sꜣ", "rꜥ"],                 "sꜣ rꜥ",        "not complement"),
    (["r", "n"],                   "r n",          "single signs"),

    # ── Chains ───────────────────────────────────────────────────────────
    (
        ["mn", "n", "nṯr", "ṯ", "r"],
        "mn nṯr",
        "chain complements"
    ),

    # ── Punctuation ──────────────────────────────────────────────────────
    (["[bn] j =tw =f"],            "bn j tw f",    "brackets + equals"),
    (["ḥtp-ḥnm.w"],                "ḥtp ḥnm w",    "hyphen + dot"),
    (["wt,j"],                     "wt j",         "comma"),
    (["ꜥnḫ", "=f"],                "ꜥnḫ f",        "clitic"),

    # ── Duplicate removal ───────────────────────────────────────────────
    (["nfr", "nfr"],               "nfr",          "adjacent duplicates"),

    # ── Duplicate + complement interaction ──────────────────────────────
    (
        ["ḥtp", "t", "ḥtp", "p"],
        "ḥtp",
        "duplicate + complement interaction"
    ),
]


print("Post-processing rules — self-test")
print("-" * 60)

_all_ok = True

for _frags, _expected, _desc in _TESTS:

    _got = apply_phonetic_post_processing(list(_frags))

    _ok = (_got == _expected)

    if not _ok:
        _all_ok = False

    print(f"{'OK  ' if _ok else 'FAIL'}  {_desc}")

    if not _ok:
        print(f"      expected: {_expected!r}")
        print(f"      got     : {_got!r}")

print("-" * 60)

print("All tests passed" if _all_ok else "Some tests FAILED")

Post-processing rules — self-test
------------------------------------------------------------
OK    pr + r
OK    mn + n
OK    nb + b
OK    nw + n
OK    ꜥnḫ + n + ḫ
OK    ḥtp + t + p
OK    nfr + f + r
OK    not complement
OK    single signs
OK    chain complements
OK    brackets + equals
OK    hyphen + dot
OK    comma
OK    clitic
OK    adjacent duplicates
OK    duplicate + complement interaction
------------------------------------------------------------
All tests passed


In [11]:
# ── Stage 1.5: Phonology Optimization Layer ────────────────────────────────

from typing import List


def phonology_stage_1_5(stage1_result):
    """
    Input : Stage1DecodeResult
    Output: improved phonetic stream with all post-processing rules applied.
    """
    decoder_metadata = getattr(stage1_result, "decoder_metadata", {})
    raw_override = clean_text(str(decoder_metadata.get("raw_transliteration", "")))
    cleaned = collapse_repeated_phonemes(stage1_result.fragments)

    if decoder_metadata.get("exact_sequence_override") and raw_override:
        # Override path: scholarly transliteration — punctuation only, no complement removal
        final = clean_phonetic_punctuation(raw_override)
        return {
            "codes":               stage1_result.codes,
            "raw_fragments":       stage1_result.fragments,
            "path_by_code":        getattr(stage1_result, "path_by_code", []),
            "phonology_optimized": [final],
            "final_output":        final,
            "unknown_codes":       stage1_result.unknown_codes,
            "stage1_confidence":   getattr(stage1_result, "confidence", None),
            "stage1_path_score":   getattr(stage1_result, "path_score", None),
            "stage1_alternatives": getattr(stage1_result, "alternatives", []),
        }

    # Normal beam-decoded path: apply the full pipeline
    final = apply_phonetic_post_processing(cleaned)
    return {
        "codes":               stage1_result.codes,
        "raw_fragments":       stage1_result.fragments,
        "path_by_code":        getattr(stage1_result, "path_by_code", []),
        "phonology_optimized": cleaned,
        "final_output":        final,
        "unknown_codes":       stage1_result.unknown_codes,
        "stage1_confidence":   getattr(stage1_result, "confidence", None),
        "stage1_path_score":   getattr(stage1_result, "path_score", None),
        "stage1_alternatives": getattr(stage1_result, "alternatives", []),
    }


def apply_stage1_5(df, stage1_column="stage1_result"):
    """Applies phonology optimization to a dataframe column of Stage1DecodeResult objects."""
    return df[stage1_column].apply(phonology_stage_1_5).apply(pd.Series)


print("Stage 1.5 phonology layer ready (Gardiner complement rules + post-processing)")


Stage 1.5 phonology layer ready (Gardiner complement rules + post-processing)


## 6.5 Accuracy-First Hybrid Inferrer for Unknown Gardiner Codes

When a Gardiner code is **not found in any dataset**, this section uses a conservative hybrid inferrer:

1. A Transformer **classifier** over known phonetic fragments.
2. A character n-gram fragment classifier over Gardiner-code patterns.
3. Structured same-family / nearby-code borrowing.
4. Blank-sign gating so likely determinatives are not forced into fake phonetics.

The notebook keeps known-code phonetics fixed, ranks only the unknown slot, and falls back conservatively when the evidence is weak.


In [12]:
# ── Section 6.5: Accuracy-first hybrid inferrer for unknown Gardiner codes ──
# Must run AFTER Section 6 (GARDINER_CANDIDATES and STAGE1_LM must exist).

import copy
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    SKLEARN_IMPORT_ERROR = None
except Exception as sklearn_exc:
    TfidfVectorizer = None
    LogisticRegression = None
    SKLEARN_IMPORT_ERROR = sklearn_exc

_PAD, _UNK = "<PAD>", "<UNK>"

INFERRED_CODES_CACHE: Dict[str, Dict[str, Any]] = {}
_INFERRED_CANDIDATE_CACHE: Dict[Tuple[str, Optional[int], Optional[int]], List[GardinerCandidate]] = {}


def _seed_torch(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _split_examples_for_validation(examples, val_ratio=0.12, seed=42):
    examples = list(examples)
    rng = random.Random(seed)
    rng.shuffle(examples)

    if len(examples) < 10 or val_ratio <= 0:
        return examples, []

    by_label: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
    for code, fragment in examples:
        by_label[fragment].append((code, fragment))

    train_examples: List[Tuple[str, str]] = []
    val_examples: List[Tuple[str, str]] = []

    for fragment, rows in by_label.items():
        rows = list(rows)
        rng.shuffle(rows)

        if len(rows) == 1:
            train_examples.extend(rows)
            continue

        proposed_val = int(round(len(rows) * val_ratio))
        val_count = min(max(proposed_val, 1), len(rows) - 1)
        val_examples.extend(rows[:val_count])
        train_examples.extend(rows[val_count:])

    if not val_examples:
        return examples, []

    rng.shuffle(train_examples)
    rng.shuffle(val_examples)
    return train_examples, val_examples


def _weight_map_to_candidates(weight_map: Dict[str, float], *, limit: int = 6) -> List[GardinerCandidate]:
    positive_items = [(frag, float(weight)) for frag, weight in weight_map.items() if float(weight) > 0]
    if not positive_items:
        return []

    ranked = sorted(positive_items, key=lambda item: item[1], reverse=True)[:limit]
    total_weight = sum(weight for _, weight in ranked) or 1.0
    return [
        GardinerCandidate(fragment=fragment, weight=round(weight / total_weight, 6))
        for fragment, weight in ranked
    ]


def _build_char_vocab(strings):
    chars = set()
    for text in strings:
        chars.update(text)
    vocab = [_PAD, _UNK] + sorted(chars)
    ch2i = {ch: idx for idx, ch in enumerate(vocab)}
    i2ch = {idx: ch for ch, idx in ch2i.items()}
    return ch2i, i2ch, vocab


def _parse_code_parts(code: str) -> Dict[str, Any]:
    code = normalize_gardiner_code(code)
    match = re.match(r"^([A-Z]+)([0-9]+)([A-Z]*)$", code)
    if not match:
        return {"code": code, "prefix": "", "number": None, "suffix": "", "valid": False}
    return {
        "code": code,
        "prefix": match.group(1),
        "number": int(match.group(2)),
        "suffix": match.group(3),
        "valid": True,
    }


def _best_nonempty_fragment(candidates: Sequence[GardinerCandidate]) -> Optional[GardinerCandidate]:
    nonempty = [cand for cand in candidates if normalize_fragment(cand.fragment)]
    if not nonempty:
        return None
    return max(nonempty, key=lambda cand: float(cand.weight))


def _top_k_from_probs(labels, probs, k=5):
    if len(probs) == 0:
        return []
    top_idx = np.argsort(probs)[::-1][: max(1, min(k, len(probs)))]
    total = float(sum(float(probs[idx]) for idx in top_idx)) or 1.0
    return [
        {"fragment": labels[idx], "weight": round(float(probs[idx]) / total, 4)}
        for idx in top_idx
        if float(probs[idx]) > 0
    ]


def _top_k_accuracy_from_probs(probs: np.ndarray, gold_indices: np.ndarray, k: int = 3) -> Optional[float]:
    if probs.size == 0 or len(gold_indices) == 0:
        return None
    k = max(1, min(k, probs.shape[1]))
    top_idx = np.argsort(probs, axis=1)[:, ::-1][:, :k]
    hits = [(gold in row) for gold, row in zip(gold_indices.tolist(), top_idx.tolist())]
    return round(float(sum(hits)) / len(hits), 4)


def _build_code_profiles(lexicon: Dict[str, List[GardinerCandidate]]) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, List[Dict[str, Any]]], Dict[str, Dict[str, Any]]]:
    profiles: Dict[str, Dict[str, Any]] = {}
    by_prefix: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    family_stats: Dict[str, Dict[str, Any]] = {}

    for code, candidates in sorted(lexicon.items()):
        parts = _parse_code_parts(code)
        if not parts["valid"]:
            continue

        blank_mass = sum(float(cand.weight) for cand in candidates if not normalize_fragment(cand.fragment))
        phonetic_mass = sum(float(cand.weight) for cand in candidates if normalize_fragment(cand.fragment))
        best_fragment = _best_nonempty_fragment(candidates)

        profile = {
            **parts,
            "blank_mass": blank_mass,
            "phonetic_mass": phonetic_mass,
            "blank_only": blank_mass > 0 and phonetic_mass <= 0.0,
            "best_fragment": best_fragment.fragment if best_fragment is not None else "",
            "best_weight": float(best_fragment.weight) if best_fragment is not None else 0.0,
        }
        profiles[code] = profile
        by_prefix[parts["prefix"]].append(profile)

    for prefix, items in by_prefix.items():
        items.sort(key=lambda row: (row["number"], row["suffix"]))
        blank_only = sum(1 for row in items if row["blank_only"])
        phonetic = sum(1 for row in items if row["best_fragment"])
        family_stats[prefix] = {
            "count": len(items),
            "blank_only_ratio": round(blank_only / max(len(items), 1), 4),
            "phonetic_ratio": round(phonetic / max(len(items), 1), 4),
            "blank_only_count": blank_only,
            "phonetic_count": phonetic,
        }

    return profiles, by_prefix, family_stats


def _structured_neighbor_score(target: Dict[str, Any], candidate: Dict[str, Any]) -> float:
    prefix_match = float(target["prefix"] == candidate["prefix"])
    suffix_match = float(target["suffix"] == candidate["suffix"])
    suffix_partial = float(not target["suffix"] or not candidate["suffix"])
    number_gap = abs(int(target["number"]) - int(candidate["number"]))

    distance_score = 1.0 / (1.0 + number_gap)
    tight_gap_bonus = 1.4 if number_gap == 1 else 0.8 if number_gap == 2 else 0.4 if number_gap <= 4 else 0.0
    edit_score = 1.0 - char_edit_distance(target["code"], candidate["code"]) / max(len(target["code"]), len(candidate["code"]), 1)

    score = (
        2.4 * prefix_match
        + 1.3 * distance_score
        + 0.7 * tight_gap_bonus
        + 0.35 * suffix_match
        + 0.15 * suffix_partial
        + 0.35 * max(edit_score, 0.0)
    )
    if not prefix_match:
        score *= 0.18
    return max(score, 0.0)


def _neighbor_evidence(
    unknown_code: str,
    *,
    top_k: Optional[int] = None,
) -> Dict[str, Any]:
    top_k = CFG.hybrid_neighbor_top_k if top_k is None else top_k
    target = _parse_code_parts(unknown_code)
    if not target["valid"]:
        return {"fragment_weights": {}, "neighbors": [], "blank_probability": 0.0}

    same_prefix_pool = list(CODE_PROFILES_BY_PREFIX.get(target["prefix"], []))
    pool = same_prefix_pool or list(CODE_PROFILES.values())

    scored = []
    for candidate in pool:
        if candidate["code"] == target["code"]:
            continue
        scored.append((candidate, _structured_neighbor_score(target, candidate)))

    scored = [(candidate, score) for candidate, score in scored if score > 0]
    scored.sort(key=lambda item: item[1], reverse=True)
    top_rows = scored[: max(3, top_k)]

    fragment_weights: Dict[str, float] = defaultdict(float)
    blank_support = 0.0
    total_support = sum(score for _, score in top_rows) or 1.0

    for candidate, score in top_rows:
        share = score / total_support
        if candidate["best_fragment"]:
            fragment_weights[candidate["best_fragment"]] += share
        if candidate["blank_only"]:
            blank_support += share

    family_stats = CODE_FAMILY_STATS.get(target["prefix"], {})
    blank_probability = min(
        0.98,
        0.55 * blank_support + 0.45 * float(family_stats.get("blank_only_ratio", 0.0)),
    )

    return {
        "fragment_weights": dict(fragment_weights),
        "neighbors": [
            {
                "code": candidate["code"],
                "score": round(score, 4),
                "best_fragment": candidate["best_fragment"] or None,
                "blank_only": candidate["blank_only"],
            }
            for candidate, score in top_rows[:top_k]
        ],
        "blank_probability": round(blank_probability, 4),
    }


def _position_adjusted_blank_probability(base_blank: float, position: Optional[int], total_len: Optional[int]) -> float:
    if position is None or total_len is None or total_len <= 0:
        return max(0.0, min(base_blank, 0.98))
    tail_bonus = 0.10 if position >= total_len - 2 else 0.0
    return max(0.0, min(base_blank + tail_bonus, 0.98))


def _build_fragment_examples(lexicon: Dict[str, List[GardinerCandidate]]) -> List[Tuple[str, str]]:
    examples = []
    for code, profile in sorted(CODE_PROFILES.items()):
        best_fragment = profile.get("best_fragment", "")
        if best_fragment:
            examples.append((code, best_fragment))
    return examples


def train_ngram_fragment_classifier(examples: Sequence[Tuple[str, str]]) -> Dict[str, Any]:
    if not CFG.ngram_classifier_enabled:
        return {"ready": False, "error": "disabled"}
    if SKLEARN_IMPORT_ERROR is not None or TfidfVectorizer is None or LogisticRegression is None:
        return {"ready": False, "error": str(SKLEARN_IMPORT_ERROR)}
    if len(examples) < 10:
        return {"ready": False, "error": "not enough examples"}

    train_examples, val_examples = _split_examples_for_validation(
        examples,
        val_ratio=CFG.transformer_validation_split,
        seed=CFG.seed,
    )

    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), lowercase=False)
    clf = LogisticRegression(
        max_iter=4000,
        C=10.0,
        class_weight="balanced",
        multi_class="auto",
        random_state=CFG.seed,
    )

    X_train = vectorizer.fit_transform([code for code, _ in train_examples])
    y_train = [fragment for _, fragment in train_examples]
    clf.fit(X_train, y_train)

    quality = {"val_accuracy": None, "val_top3": None}
    if val_examples:
        X_val = vectorizer.transform([code for code, _ in val_examples])
        y_val = [fragment for _, fragment in val_examples]
        probs = clf.predict_proba(X_val)
        labels = list(clf.classes_)
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        pred = [labels[int(np.argmax(row))] for row in probs]
        quality = {
            "val_accuracy": round(sum(int(p == y) for p, y in zip(pred, y_val)) / len(y_val), 4),
            "val_top3": _top_k_accuracy_from_probs(
                np.asarray(probs),
                np.asarray([label_to_idx[y] for y in y_val], dtype=int),
                k=3,
            ),
        }

    return {
        "ready": True,
        "vectorizer": vectorizer,
        "classifier": clf,
        "classes": list(clf.classes_),
        "train_examples": len(train_examples),
        "val_examples": len(val_examples),
        "quality": quality,
        "error": None,
    }


def predict_ngram_fragment_candidates(code: str, model_info: Dict[str, Any], top_k: Optional[int] = None) -> List[Dict[str, Any]]:
    if not model_info or not model_info.get("ready"):
        return []
    top_k = CFG.hybrid_candidate_top_k if top_k is None else top_k

    vectorizer = model_info["vectorizer"]
    clf = model_info["classifier"]
    probs = clf.predict_proba(vectorizer.transform([normalize_gardiner_code(code)]))[0]
    return _top_k_from_probs(list(clf.classes_), probs, k=top_k)


class _ClassifierDataset(Dataset):
    def __init__(self, examples, src_ch2i, label_to_idx, max_src=20):
        self.data = []
        for code, fragment in examples:
            src = [src_ch2i.get(ch, src_ch2i[_UNK]) for ch in code][:max_src]
            self.data.append((src, label_to_idx[fragment]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def _pad_classifier_batch(batch, pad_idx):
    max_src = max(len(src) for src, _ in batch)
    src_tensor = torch.tensor(
        [src + [pad_idx] * (max_src - len(src)) for src, _ in batch],
        dtype=torch.long,
    )
    label_tensor = torch.tensor([label for _, label in batch], dtype=torch.long)
    return src_tensor, label_tensor


class _GardinerTransformerClassifier(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        num_labels: int,
        d_model: int = 160,
        heads: int = 4,
        layers: int = 3,
        ff_dim: int = 512,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, layers, norm=nn.LayerNorm(d_model))
        self.proj = nn.Linear(d_model, num_labels)

        for param in self.parameters():
            if param.dim() > 1:
                nn.init.xavier_uniform_(param)

    def _positional_encoding(self, seq_len: int, device) -> torch.Tensor:
        position = torch.arange(seq_len, device=device, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, device=device, dtype=torch.float32)
            * -(math.log(10000.0) / self.d_model)
        )
        pe = torch.zeros(seq_len, self.d_model, device=device, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, src):
        mask = src.eq(0)
        x = self.src_emb(src) * math.sqrt(self.d_model)
        x = x + self._positional_encoding(src.size(1), src.device)
        encoded = self.encoder(x, src_key_padding_mask=mask)
        valid = (~mask).unsqueeze(-1).float()
        pooled = (encoded * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        return self.proj(pooled)


def _evaluate_transformer_classifier(model, loader, criterion, device) -> Dict[str, Any]:
    if loader is None:
        return {"loss": None, "accuracy": None, "top3": None}

    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for src_batch, label_batch in loader:
            src_batch = src_batch.to(device)
            label_batch = label_batch.to(device)
            logits = model(src_batch)
            loss = criterion(logits, label_batch)
            probs = torch.softmax(logits, dim=-1)

            total_loss += float(loss.item()) * label_batch.size(0)
            total += label_batch.size(0)
            correct += int((logits.argmax(dim=-1) == label_batch).sum().item())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(label_batch.cpu().numpy())

    model.train()
    probs_np = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, 0))
    labels_np = np.concatenate(all_labels, axis=0) if all_labels else np.zeros((0,), dtype=int)
    return {
        "loss": round(total_loss / max(total, 1), 4),
        "accuracy": round(correct / max(total, 1), 4),
        "top3": _top_k_accuracy_from_probs(probs_np, labels_np, k=3),
    }


def train_transformer_fragment_classifier(examples: Sequence[Tuple[str, str]]) -> Dict[str, Any]:
    if not CFG.transformer_enabled or not CFG.transformer_classifier_enabled:
        return {"ready": False, "error": "disabled"}
    if len(examples) < 10:
        return {"ready": False, "error": "not enough examples"}

    train_examples, val_examples = _split_examples_for_validation(
        examples,
        val_ratio=CFG.transformer_validation_split,
        seed=CFG.seed,
    )

    src_ch2i, src_i2ch, src_vocab = _build_char_vocab([code for code, _ in examples])
    labels = sorted({fragment for _, fragment in examples})
    label_to_idx = {fragment: idx for idx, fragment in enumerate(labels)}

    train_ds = _ClassifierDataset(
        train_examples,
        src_ch2i,
        label_to_idx,
        max_src=CFG.transformer_max_src_len,
    )
    val_ds = _ClassifierDataset(
        val_examples,
        src_ch2i,
        label_to_idx,
        max_src=CFG.transformer_max_src_len,
    ) if val_examples else None

    train_loader = DataLoader(
        train_ds,
        batch_size=min(CFG.transformer_batch_size, max(len(train_ds), 1)),
        shuffle=True,
        collate_fn=lambda batch: _pad_classifier_batch(batch, src_ch2i[_PAD]),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=min(CFG.transformer_batch_size, max(len(val_ds), 1)),
        shuffle=False,
        collate_fn=lambda batch: _pad_classifier_batch(batch, src_ch2i[_PAD]),
    ) if val_ds is not None else None

    device = "cuda" if torch.cuda.is_available() and not CFG.transformer_force_cpu else "cpu"
    _seed_torch(CFG.seed)

    model = _GardinerTransformerClassifier(len(src_vocab), len(labels)).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.transformer_learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.6,
        patience=4,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    best_val_loss = float("inf")
    patience_left = CFG.transformer_patience
    history = []

    log(
        f"Transformer classifier training: {len(train_examples)} train pairs"
        f" | {len(val_examples)} val pairs | device={device}"
    )

    for epoch in range(1, CFG.transformer_train_epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for src_batch, label_batch in train_loader:
            src_batch = src_batch.to(device)
            label_batch = label_batch.to(device)
            logits = model(src_batch)
            loss = criterion(logits, label_batch)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += float(loss.item()) * label_batch.size(0)
            seen += label_batch.size(0)

        train_loss = running_loss / max(seen, 1)
        val_metrics = _evaluate_transformer_classifier(model, val_loader, criterion, device)
        val_loss = val_metrics["loss"] if val_metrics["loss"] is not None else train_loss
        scheduler.step(val_loss)

        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "val_accuracy": val_metrics["accuracy"],
            "val_top3": val_metrics["top3"],
        })

        improved = val_loss + 1e-4 < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = CFG.transformer_patience
        else:
            patience_left -= 1

        if epoch == 1 or epoch % 10 == 0 or improved:
            log(
                f"  Epoch {epoch:3d}/{CFG.transformer_train_epochs}"
                f" | train_loss={train_loss:.4f}"
                f" | val_loss={val_loss:.4f}"
                f" | val_acc={val_metrics['accuracy']}"
                f" | val_top3={val_metrics['top3']}"
            )

        if epoch >= 20 and patience_left <= 0:
            log(f"Early stopping at epoch {epoch} (best epoch {best_epoch}).")
            break

    model.load_state_dict(best_state)
    final_metrics = _evaluate_transformer_classifier(model, val_loader, criterion, device)

    return {
        "ready": True,
        "model": model,
        "device": device,
        "src_ch2i": src_ch2i,
        "src_i2ch": src_i2ch,
        "classes": labels,
        "train_examples": len(train_examples),
        "val_examples": len(val_examples),
        "best_epoch": best_epoch or len(history),
        "best_val_loss": round(best_val_loss, 4),
        "quality": {
            "val_accuracy": final_metrics["accuracy"],
            "val_top3": final_metrics["top3"],
        },
        "history_tail": history[-5:],
        "error": None,
    }


def predict_transformer_fragment_candidates(code: str, model_info: Dict[str, Any], top_k: Optional[int] = None) -> List[Dict[str, Any]]:
    if not model_info or not model_info.get("ready"):
        return []
    top_k = CFG.hybrid_candidate_top_k if top_k is None else top_k

    src_ch2i = model_info["src_ch2i"]
    model = model_info["model"]
    device = model_info.get("device", "cpu")
    classes = model_info["classes"]

    src_ids = [src_ch2i.get(ch, src_ch2i[_UNK]) for ch in normalize_gardiner_code(code)][: CFG.transformer_max_src_len]
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        logits = model(src_tensor)
        probs = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()
    model.train()
    return _top_k_from_probs(classes, probs, k=top_k)


CODE_PROFILES, CODE_PROFILES_BY_PREFIX, CODE_FAMILY_STATS = _build_code_profiles(GARDINER_CANDIDATES)
FRAGMENT_EXAMPLES = _build_fragment_examples(GARDINER_CANDIDATES)


def infer_candidates_for_unknown(
    unknown_code: str,
    inferrer: Dict[str, Any],
    *,
    lexicon: Optional[Dict[str, List[GardinerCandidate]]] = None,
    position: Optional[int] = None,
    total_len: Optional[int] = None,
) -> List[GardinerCandidate]:
    lexicon = GARDINER_CANDIDATES if lexicon is None else lexicon
    cache_key = (normalize_gardiner_code(unknown_code), position, total_len)
    if cache_key in _INFERRED_CANDIDATE_CACHE:
        return _INFERRED_CANDIDATE_CACHE[cache_key]

    code_norm = normalize_gardiner_code(unknown_code)
    single_token_unknown = total_len == 1
    neighbor_info = _neighbor_evidence(code_norm, top_k=CFG.hybrid_neighbor_top_k)
    blank_probability = _position_adjusted_blank_probability(
        float(neighbor_info.get("blank_probability", 0.0)),
        position,
        total_len,
    )

    ngram_info = inferrer.get("ngram_classifier", {})
    transformer_info = inferrer.get("transformer_classifier", {})
    ngram_candidates = predict_ngram_fragment_candidates(code_norm, ngram_info, top_k=CFG.hybrid_candidate_top_k)
    transformer_candidates = predict_transformer_fragment_candidates(code_norm, transformer_info, top_k=CFG.hybrid_candidate_top_k)

    ngram_quality = float((ngram_info.get("quality") or {}).get("val_top3") or 0.20)
    transformer_quality = float((transformer_info.get("quality") or {}).get("val_top3") or 0.15)

    ngram_scale = CFG.hybrid_ngram_weight * max(0.35, ngram_quality)
    transformer_scale = CFG.hybrid_transformer_weight * max(0.30, transformer_quality)
    similarity_scale = CFG.hybrid_similarity_weight

    combined_weights: Dict[str, float] = defaultdict(float)
    source_votes: Dict[str, Dict[str, float]] = defaultdict(
        lambda: {"ngram": 0.0, "transformer": 0.0, "similarity": 0.0, "consensus": 0.0}
    )

    for fragment, weight in neighbor_info.get("fragment_weights", {}).items():
        combined_weights[fragment] += similarity_scale * float(weight)
        source_votes[fragment]["similarity"] += similarity_scale * float(weight)

    for item in ngram_candidates:
        fragment = normalize_fragment(item["fragment"])
        if not fragment:
            continue
        combined_weights[fragment] += ngram_scale * float(item["weight"])
        source_votes[fragment]["ngram"] += ngram_scale * float(item["weight"])

    for item in transformer_candidates:
        fragment = normalize_fragment(item["fragment"])
        if not fragment:
            continue
        combined_weights[fragment] += transformer_scale * float(item["weight"])
        source_votes[fragment]["transformer"] += transformer_scale * float(item["weight"])

    ngram_top_fragment = normalize_fragment(ngram_candidates[0]["fragment"]) if ngram_candidates else ""
    transformer_top_fragment = normalize_fragment(transformer_candidates[0]["fragment"]) if transformer_candidates else ""
    ngram_top_conf = float(ngram_candidates[0]["weight"]) if ngram_candidates else 0.0
    transformer_top_conf = float(transformer_candidates[0]["weight"]) if transformer_candidates else 0.0

    phonetic_consensus_fragment = ""
    phonetic_consensus_strength = 0.0
    if ngram_top_fragment:
        phonetic_consensus_strength += 0.5 * ngram_top_conf
    if transformer_top_fragment:
        phonetic_consensus_strength += 0.5 * transformer_top_conf

    if ngram_top_fragment and ngram_top_fragment == transformer_top_fragment:
        phonetic_consensus_fragment = ngram_top_fragment
        phonetic_consensus_strength += 0.25
        consensus_bonus = min(0.35, 0.18 + 0.12 * phonetic_consensus_strength)
        combined_weights[phonetic_consensus_fragment] += consensus_bonus
        source_votes[phonetic_consensus_fragment]["consensus"] += consensus_bonus

    effective_blank_probability = blank_probability
    if phonetic_consensus_strength > 0.0:
        effective_blank_probability *= max(
            0.15,
            1.0 - min(0.85, phonetic_consensus_strength),
        )

    if single_token_unknown:
        # A standalone unknown sign is more useful to downstream stages when we
        # preserve a plausible phonetic guess instead of collapsing to blank.
        effective_blank_probability *= 0.35

    if effective_blank_probability >= CFG.hybrid_blank_threshold:
        blank_gate = max(CFG.hybrid_force_blank_threshold, 0.90) if single_token_unknown else CFG.hybrid_blank_threshold
        if effective_blank_probability < blank_gate:
            pass
        else:
            blank_weight = (
                effective_blank_probability
                if effective_blank_probability >= CFG.hybrid_force_blank_threshold
                else effective_blank_probability * 0.55
            )
            combined_weights[""] += blank_weight

    candidates = _weight_map_to_candidates(combined_weights, limit=CFG.hybrid_candidate_top_k)

    if not candidates:
        if effective_blank_probability >= CFG.hybrid_blank_threshold and not single_token_unknown:
            candidates = [GardinerCandidate(fragment="", weight=1.0)]
            method = "blank_only_fallback"
        else:
            candidates = [GardinerCandidate(fragment=code_norm.lower(), weight=1.0)]
            method = "last_resort_lowercase"
    else:
        top_fragment = candidates[0].fragment
        if single_token_unknown and top_fragment == "":
            nonblank_candidates = [candidate for candidate in candidates if normalize_fragment(candidate.fragment)]
            if nonblank_candidates:
                total_weight = sum(float(candidate.weight) for candidate in nonblank_candidates) or 1.0
                candidates = [
                    GardinerCandidate(
                        fragment=candidate.fragment,
                        weight=round(float(candidate.weight) / total_weight, 6),
                    )
                    for candidate in nonblank_candidates
                ]
                top_fragment = candidates[0].fragment
        if top_fragment == "":
            method = "hybrid_blank_preferred"
        else:
            method = "hybrid_ranked_candidates"

    similar_codes = [row["code"] for row in neighbor_info.get("neighbors", [])[:3]]
    similar_scores = [row["score"] for row in neighbor_info.get("neighbors", [])[:3]]
    transformer_confidence = transformer_candidates[0]["weight"] if transformer_candidates else None
    ngram_confidence = ngram_candidates[0]["weight"] if ngram_candidates else None

    INFERRED_CODES_CACHE[code_norm] = {
        "method": method,
        "position": position,
        "total_len": total_len,
        "blank_probability": round(blank_probability, 4),
        "effective_blank_probability": round(effective_blank_probability, 4),
        "phonetic_consensus_fragment": phonetic_consensus_fragment or None,
        "phonetic_consensus_strength": round(float(phonetic_consensus_strength), 4),
        "ngram_confidence": round(float(ngram_confidence), 4) if ngram_confidence is not None else None,
        "transformer_confidence": round(float(transformer_confidence), 4) if transformer_confidence is not None else None,
        "ngram_candidates": ngram_candidates[:3],
        "transformer_candidates": transformer_candidates[:3],
        "similar_codes": similar_codes,
        "similar_scores": [round(float(score), 3) for score in similar_scores],
        "neighbor_details": neighbor_info.get("neighbors", [])[:5],
        "blend_weights": {
            "ngram": round(float(ngram_scale), 4),
            "transformer": round(float(transformer_scale), 4),
            "similarity": round(float(similarity_scale), 4),
        },
        "final_candidates": [
            {
                "fragment": cand.fragment,
                "weight": cand.weight,
                "support": {
                    key: round(float(value), 4)
                    for key, value in source_votes.get(cand.fragment, {}).items()
                    if value > 0
                },
            }
            for cand in candidates
        ],
    }

    _INFERRED_CANDIDATE_CACHE[cache_key] = candidates
    return candidates


GARDINER_INFERRER: Dict[str, Any] = {
    "train_pairs": len(FRAGMENT_EXAMPLES),
    "ngram_classifier": {"ready": False, "error": None, "quality": {}},
    "transformer_classifier": {"ready": False, "error": None, "quality": {}},
    "device": "cpu",
    "train_examples": len(FRAGMENT_EXAMPLES),
    "val_examples": 0,
    "best_epoch": None,
    "best_val_loss": None,
    "final_loss": None,
    "quality": {},
    "history_tail": [],
    "error": None,
}

try:
    GARDINER_INFERRER["ngram_classifier"] = train_ngram_fragment_classifier(FRAGMENT_EXAMPLES)
    if GARDINER_INFERRER["ngram_classifier"].get("ready"):
        log(
            "N-gram fragment classifier ready"
            f" | val_acc={GARDINER_INFERRER['ngram_classifier']['quality'].get('val_accuracy')}"
            f" | val_top3={GARDINER_INFERRER['ngram_classifier']['quality'].get('val_top3')}"
        )
    else:
        log(f"N-gram fragment classifier unavailable: {GARDINER_INFERRER['ngram_classifier'].get('error')}", "WARN")
except Exception as exc:
    GARDINER_INFERRER["ngram_classifier"] = {"ready": False, "error": str(exc), "quality": {}}
    log(f"N-gram fragment classifier failed: {exc}", "WARN")

try:
    GARDINER_INFERRER["transformer_classifier"] = train_transformer_fragment_classifier(FRAGMENT_EXAMPLES)
    if GARDINER_INFERRER["transformer_classifier"].get("ready"):
        GARDINER_INFERRER["device"] = GARDINER_INFERRER["transformer_classifier"].get("device", "cpu")
        GARDINER_INFERRER["val_examples"] = GARDINER_INFERRER["transformer_classifier"].get("val_examples", 0)
        GARDINER_INFERRER["best_epoch"] = GARDINER_INFERRER["transformer_classifier"].get("best_epoch")
        GARDINER_INFERRER["best_val_loss"] = GARDINER_INFERRER["transformer_classifier"].get("best_val_loss")
        GARDINER_INFERRER["quality"] = GARDINER_INFERRER["transformer_classifier"].get("quality", {})
        GARDINER_INFERRER["history_tail"] = GARDINER_INFERRER["transformer_classifier"].get("history_tail", [])
        log(
            "Transformer classifier ready"
            f" | val_acc={GARDINER_INFERRER['quality'].get('val_accuracy')}"
            f" | val_top3={GARDINER_INFERRER['quality'].get('val_top3')}"
        )
    else:
        log(f"Transformer classifier unavailable: {GARDINER_INFERRER['transformer_classifier'].get('error')}", "WARN")
except Exception as exc:
    GARDINER_INFERRER["transformer_classifier"] = {"ready": False, "error": str(exc), "quality": {}}
    log(f"Transformer classifier failed: {exc}", "WARN")

if not GARDINER_INFERRER["ngram_classifier"].get("ready") and not GARDINER_INFERRER["transformer_classifier"].get("ready"):
    GARDINER_INFERRER["error"] = "No hybrid submodel is ready; similarity-only fallback will be used."
    log(GARDINER_INFERRER["error"], "WARN")


_orig_get_stage1_candidates = get_stage1_candidates


def get_stage1_candidates(
    code: Any,
    *,
    unknown_strategy: Optional[str] = None,
    position: Optional[int] = None,
    total_len: Optional[int] = None,
) -> Tuple[List[GardinerCandidate], bool]:
    unknown_strategy = unknown_strategy or CFG.stage1_unknown_strategy
    code_norm = normalize_gardiner_code(code)

    candidates = GARDINER_CANDIDATES.get(code_norm)
    if candidates:
        return candidates, False

    inferred = infer_candidates_for_unknown(
        code_norm,
        globals().get("GARDINER_INFERRER") or {},
        position=position,
        total_len=total_len,
    )
    if inferred:
        return inferred, True

    if unknown_strategy == "epsilon":
        fallback_fragment = ""
    elif unknown_strategy == "placeholder":
        fallback_fragment = f"<unk:{code_norm.lower()}>"
    else:
        fallback_fragment = code_norm.lower()

    return [GardinerCandidate(fragment=fallback_fragment, weight=1.0)], True


log("get_stage1_candidates patched -- unknown codes now use hybrid ranked inference.")
log("Known codes stay fixed; only unknown slots are inferred.")

_test_unknown = "ZZ99"
_cands, _is_unk = get_stage1_candidates(_test_unknown, position=0, total_len=1)
print(f"\nSelf-test -- {_test_unknown} (is_unknown={_is_unk}):")
for c in _cands[:5]:
    print(f"  fragment={c.fragment!r:20s}  weight={c.weight:.4f}")
print("Inference metadata:", json.dumps(INFERRED_CODES_CACHE.get(_test_unknown, {}), ensure_ascii=False, indent=2))


[2026-05-29 14:50:37] [INFO] N-gram fragment classifier ready | val_acc=0.1875 | val_top3=0.3359
[2026-05-29 14:50:39] [INFO] Transformer classifier training: 435 train pairs | 128 val pairs | device=cpu
[2026-05-29 14:50:39] [INFO]   Epoch   1/80 | train_loss=6.0683 | val_loss=5.8591 | val_acc=0.0 | val_top3=0.0078
[2026-05-29 14:50:40] [INFO]   Epoch   2/80 | train_loss=5.6808 | val_loss=5.7091 | val_acc=0.0234 | val_top3=0.0391
[2026-05-29 14:50:40] [INFO]   Epoch   3/80 | train_loss=5.4302 | val_loss=5.5824 | val_acc=0.0234 | val_top3=0.0625
[2026-05-29 14:50:40] [INFO]   Epoch   4/80 | train_loss=5.2057 | val_loss=5.4678 | val_acc=0.0469 | val_top3=0.0938
[2026-05-29 14:50:40] [INFO]   Epoch   5/80 | train_loss=5.0258 | val_loss=5.3701 | val_acc=0.0625 | val_top3=0.1328
[2026-05-29 14:50:41] [INFO]   Epoch   6/80 | train_loss=4.8012 | val_loss=5.2858 | val_acc=0.0625 | val_top3=0.1562
[2026-05-29 14:50:41] [INFO]   Epoch   7/80 | train_loss=4.6519 | val_loss=5.2191 | val_acc=0.078

In [13]:
# ── Section 6.5: Evaluation Metrics + Auto-Run ──────────────────────────────

from collections import Counter
import numpy as np
import pandas as pd


def levenshtein(a, b):
    rows = len(a) + 1
    cols = len(b) + 1
    dp = np.zeros((rows, cols), dtype=int)

    for i in range(rows):
        dp[i][0] = i
    for j in range(cols):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, cols):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )

    return int(dp[len(a)][len(b)])


def compute_cer(ref, hyp):
    if not ref:
        return 0.0 if not hyp else 1.0
    return levenshtein(list(ref), list(hyp)) / len(ref)


def compute_wer(ref_tokens, hyp_tokens):
    if not ref_tokens:
        return 0.0 if not hyp_tokens else 1.0
    return levenshtein(ref_tokens, hyp_tokens) / len(ref_tokens)


def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def compute_bleu(ref_tokens, hyp_tokens, max_n=1):
    if not ref_tokens and not hyp_tokens:
        return 1.0

    max_n = max(1, min(max_n, len(ref_tokens) or 1, len(hyp_tokens) or 1))
    weights = [1.0 / max_n] * max_n
    precisions = []

    for n in range(1, max_n + 1):
        ref_ngrams = ngram_counts(ref_tokens, n)
        hyp_ngrams = ngram_counts(hyp_tokens, n)

        overlap = sum(
            min(count, ref_ngrams.get(gram, 0))
            for gram, count in hyp_ngrams.items()
        )
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(overlap / total)

    bleu = np.exp(np.sum([
        weight * np.log(max(precision, 1e-9))
        for weight, precision in zip(weights, precisions)
    ]))

    ref_len = len(ref_tokens)
    hyp_len = len(hyp_tokens)
    if hyp_len < ref_len:
        bleu *= np.exp(1 - ref_len / max(hyp_len, 1))

    return float(bleu)


def prepare_stage1_eval_rows(
    df,
    *,
    input_col=None,
    ref_col="translit_norm",
    sample_size=None,
    gardiner_only=True,
):
    input_col = CFG.col_raw_translit if input_col is None else input_col
    eval_df = df.copy()

    if input_col not in eval_df.columns:
        raise KeyError(f"Missing input column: {input_col}")
    if ref_col not in eval_df.columns:
        raise KeyError(f"Missing reference column: {ref_col}")

    eval_df["__stage1_input__"] = eval_df[input_col].astype(str).apply(clean_text)
    eval_df["__stage1_reference__"] = eval_df[ref_col].astype(str).apply(normalize_translit)

    eval_df = eval_df[
        (eval_df["__stage1_input__"].str.len() > 0)
        & (eval_df["__stage1_reference__"].str.len() > 0)
    ].copy()

    if gardiner_only:
        eval_df = eval_df[eval_df["__stage1_input__"].apply(is_gardiner_input)].copy()

    if sample_size and len(eval_df) > sample_size:
        eval_df = eval_df.sample(sample_size, random_state=CFG.seed).copy()

    return eval_df.reset_index(drop=True)


def evaluate_model(
    df,
    *,
    input_col=None,
    ref_col="translit_norm",
    sample_size=200,
    gardiner_only=True,
):
    eval_df = prepare_stage1_eval_rows(
        df,
        input_col=input_col,
        ref_col=ref_col,
        sample_size=sample_size,
        gardiner_only=gardiner_only,
    )

    if eval_df.empty:
        print("No evaluation rows available after filtering.")
        return pd.DataFrame(), {}

    results = []

    for _, row in eval_df.iterrows():
        source_text = row["__stage1_input__"]
        ref = row["__stage1_reference__"]

        stage1 = decode_gardiner_text(source_text)
        stage15 = phonology_stage_1_5(stage1)
        decoder_metadata = getattr(stage1, "decoder_metadata", {})

        hyp = normalize_translit(stage15["final_output"])
        ref_chars = ref.replace(" ", "")
        hyp_chars = hyp.replace(" ", "")
        ref_tokens = [token for token in ref.split() if token]
        # FIX (Bug 2): use final_output tokens (post-processing applied) for WER.
        # phonology_optimized is the PRE-post-processing list and still contains
        # complement fragments that should be absent, inflating WER.
        hyp_tokens = [
            normalize_fragment(token)
            for token in stage15["final_output"].split()
            if normalize_fragment(token)
        ]

        results.append({
            "input": source_text,
            "reference": ref,
            "prediction": hyp,
            "cer": compute_cer(ref_chars, hyp_chars),
            "wer": compute_wer(ref_tokens, hyp_tokens),
            "bleu": compute_bleu(ref_tokens, hyp_tokens),
            "exact_match": int(ref_chars == hyp_chars),
            "inferred": bool(stage1.unknown_codes),
            "unknown_code_count": len(stage1.unknown_codes),
            "known_codes_anchored": bool(decoder_metadata.get("known_codes_anchored", False)),
            "confidence": getattr(stage1, "confidence", None),
        })

    res_df = pd.DataFrame(results)
    summary = {
        "rows": int(len(res_df)),
        "CER": round(float(res_df["cer"].mean()), 4),
        "WER": round(float(res_df["wer"].mean()), 4),
        "BLEU-1": round(float(res_df["bleu"].mean()), 4),
        "Exact Match": round(float(res_df["exact_match"].mean()), 4),
        "Inference Rate": round(float(res_df["inferred"].mean()), 4),
    }

    print("\nStage 1 evaluation")
    print(f"Rows evaluated : {summary['rows']}")
    print(f"CER            : {summary['CER']:.4f}")
    print(f"WER            : {summary['WER']:.4f}")
    print(f"BLEU-1         : {summary['BLEU-1']:.4f}")
    print(f"Exact Match    : {summary['Exact Match']:.4f}")
    print(f"Inference Rate : {summary['Inference Rate']:.4f}")
    if summary["Inference Rate"] == 0.0:
        print("Note: this evaluation sample contained only known-code rows, so it is measuring lexicon lookup quality rather than unknown-code inference.")
    print("Note: BLEU-1 is unigram-only here so it stays meaningful on one-token sign outputs.")

    if res_df["inferred"].any():
        print("\nBreakdown by inferred=False/True")
        display(
            res_df.groupby("inferred")[["cer", "wer", "bleu", "exact_match", "unknown_code_count"]]
            .mean()
            .round(4)
        )

    preview_cols = ["input", "reference", "prediction", "inferred", "known_codes_anchored", "exact_match"]
    print("\nSample predictions")
    display(res_df[preview_cols].head(12))

    return res_df, summary


if "DF" in globals() and len(DF) > 0:
    EVAL_RESULTS_DF, EVAL_SUMMARY = evaluate_model(
        DF,
        input_col=CFG.col_raw_translit,
        ref_col="translit_norm",
        sample_size=min(200, len(DF)),
        gardiner_only=True,
    )
else:
    print("DF is not loaded yet. Run the corpus-loading cell first, then call evaluate_model(DF).")



Stage 1 evaluation
Rows evaluated : 200
CER            : 0.0000
WER            : 0.0050
BLEU-1         : 0.9950
Exact Match    : 1.0000
Inference Rate : 0.0000
Note: this evaluation sample contained only known-code rows, so it is measuring lexicon lookup quality rather than unknown-code inference.
Note: BLEU-1 is unigram-only here so it stays meaningful on one-token sign outputs.

Sample predictions


,input,reference,prediction,inferred,known_codes_anchored,exact_match
0,E24,ꜣbš,ꜣbš,False,True,1
1,S21,ꜥnḫ,ꜥnḫ,False,True,1
2,W14,ḥb,ḥb,False,True,1
3,D46,ḏ,ḏ,False,True,1
4,G13,ḥr,ḥr,False,True,1
5,M35,it,it,False,True,1
6,A36,fqꜣ,fqꜣ,False,True,1
7,T7A,mꜣ,mꜣ,False,True,1
8,D46A,idt,idt,False,True,1
9,O17,sbꜣ,sbꜣ,False,True,1


## 7. Stage 1 Sanity Check

In [14]:
# Quick sanity-check on a few codes
STAGE1_DEMOS = ["G17 M18 F34", "A1 D21 N5", "A2 N5", "A17 D21"]

print("Lexicon preview:")
for code in ["A1", "A2", "A17", "D21", "N5", "F34", "G17", "M18", "A10"]:
    cands, is_unk = get_stage1_candidates(code)
    print(f"  {code:5s} -> {[c.fragment for c in cands]}  | unknown={is_unk}")

print("\nDecode demos:")
for demo in STAGE1_DEMOS:
    r = decode_gardiner_text(demo)
    print(json.dumps({
        "input": demo,
        "codes": r.codes,
        "spaced_phonetics": r.spaced_phonetics,
        "unknown_codes": r.unknown_codes,
    }, ensure_ascii=False, indent=2))

print("\nDetailed candidates per code:")
for code in r.codes:
    cands, _ = get_stage1_candidates(code)
    print(f"{code} -> {[ (c.fragment, round(c.weight,3)) for c in cands ]}")


Lexicon preview:
  A1    -> ['s']  | unknown=False
  A2    -> ['ꜣ']  | unknown=False
  A17   -> ['ẖrd']  | unknown=False
  D21   -> ['r']  | unknown=False
  N5    -> ['rꜥ']  | unknown=False
  F34   -> ['ib']  | unknown=False
  G17   -> ['m']  | unknown=False
  M18   -> ['i']  | unknown=False
  A10   -> ['']  | unknown=False

Decode demos:
{
  "input": "G17 M18 F34",
  "codes": [
    "G17",
    "M18",
    "F34"
  ],
  "spaced_phonetics": "m i ib",
  "unknown_codes": []
}
{
  "input": "A1 D21 N5",
  "codes": [
    "A1",
    "D21",
    "N5"
  ],
  "spaced_phonetics": "s r rꜥ",
  "unknown_codes": []
}
{
  "input": "A2 N5",
  "codes": [
    "A2",
    "N5"
  ],
  "spaced_phonetics": "ꜣ rꜥ",
  "unknown_codes": []
}
{
  "input": "A17 D21",
  "codes": [
    "A17",
    "D21"
  ],
  "spaced_phonetics": "ẖrd r",
  "unknown_codes": []
}

Detailed candidates per code:
A17 -> [('ẖrd', 1.0)]
D21 -> [('r', 1.0)]


## 8. Apply Stage 1 to Corpus and Save `stage1_output.csv`

In [15]:
print(decode_gardiner_text("A1 D21 N5"))

Stage1DecodeResult(codes=['A1', 'D21', 'N5'], fragments=['s', 'r', 'rꜥ'], spaced_phonetics='s r rꜥ', unknown_codes=[], confidence=0.95, path_score=-19.147325, alternatives=[], path_by_code=['s', 'r', 'rꜥ'], decoder_metadata={'beam_size': 6, 'beam_path_count': 1, 'beam_top_paths': [{'path_by_code': ['s', 'r', 'rꜥ'], 'fragments': ['s', 'r', 'rꜥ'], 'spaced_phonetics': 's r rꜥ', 'score': -19.147325, 'epsilon_count': 0}], 'weights': {'alpha': 0.9, 'beta': 1.1, 'gamma': 0.35}, 'lm_source': 'parallel_aligned_bootstrap', 'known_codes_anchored': True, 'anchored_known_codes': ['A1', 'D21', 'N5'], 'strict_known_code_lookup': True})


In [16]:
# ── Stage 1 Diagnostics ───────────────────────────────────────────────────────

runtime_mode = "aligned_bootstrap" if STAGE1_LM.source == "parallel_aligned_bootstrap" else "fallback"

transformer_diag = {
    "enabled": CFG.transformer_enabled,
    "anchor_known_on_unknown": getattr(CFG, "stage1_anchor_known_codes_when_unknown_present", True),
    "device": GARDINER_INFERRER.get("device"),
    "train_pairs": GARDINER_INFERRER.get("train_pairs"),
    "train_examples": GARDINER_INFERRER.get("train_examples"),
    "val_examples": GARDINER_INFERRER.get("val_examples"),
    "best_epoch": GARDINER_INFERRER.get("best_epoch"),
    "best_val_loss": GARDINER_INFERRER.get("best_val_loss"),
    "final_loss": GARDINER_INFERRER.get("final_loss"),
    "quality": GARDINER_INFERRER.get("quality", {}),
    "ngram_quality": (GARDINER_INFERRER.get("ngram_classifier") or {}).get("quality", {}),
    "transformer_classifier_quality": (GARDINER_INFERRER.get("transformer_classifier") or {}).get("quality", {}),
    "ngram_ready": (GARDINER_INFERRER.get("ngram_classifier") or {}).get("ready"),
    "transformer_classifier_ready": (GARDINER_INFERRER.get("transformer_classifier") or {}).get("ready"),
    "history_tail": GARDINER_INFERRER.get("history_tail", []),
    "error": GARDINER_INFERRER.get("error"),
}

diagnostics = {
    "stage1_runtime_mode": runtime_mode,
    "beam_size": BEAM_SIZE,
    "n_best": N_BEST,
    "decoder_weights": {
        "alpha": STAGE1_DECODER_WEIGHTS.alpha,
        "beta": STAGE1_DECODER_WEIGHTS.beta,
        "gamma": STAGE1_DECODER_WEIGHTS.gamma,
    },
    "weight_tuning": STAGE1_WEIGHT_TUNING,
    "lm_source": STAGE1_LM.source,
    "lm_sequence_count": STAGE1_LM.sequence_count,
    "lm_metadata": STAGE1_LM_METADATA,
    "role_supervision_metadata": GARDINER_ROLE_METADATA,
    "role_training_metadata": globals().get("ROLE_TRAINING_METADATA", {}),
    "aligned_training_rows": len(STAGE1_ALIGNMENT_ROWS),
    "parallel_candidate_rows": len(STAGE1_PARALLEL_ROWS),
    "transformer_inferrer": transformer_diag,
    "inferred_cache_size": len(INFERRED_CODES_CACHE),
}

print(json.dumps(diagnostics, ensure_ascii=False, indent=2))


{
  "stage1_runtime_mode": "aligned_bootstrap",
  "beam_size": 6,
  "n_best": 3,
  "decoder_weights": {
    "alpha": 0.9,
    "beta": 1.1,
    "gamma": 0.35
  },
  "weight_tuning": {
    "status": "tuned",
    "dev_rows": 231,
    "mean_fit_quality": 1.0
  },
  "lm_source": "parallel_aligned_bootstrap",
  "lm_sequence_count": 1157,
  "lm_metadata": {
    "parallel_row_count": 1161,
    "explicit_parallel_rows": 0,
    "corpus_parallel_rows": 598,
    "gardiner_updated_parallel_rows": 563,
    "lm_source": "parallel_aligned_bootstrap",
    "bootstrap_alignment_rows": 1161,
    "bootstrap_sequence_count": 1157,
    "aligned_sequence_count": 1157,
    "lm_sequence_count": 1157
  },
  "role_supervision_metadata": {
    "source": "Gardiner_Sign_List.csv",
    "status": "loaded",
    "explicit_role_rows": 6904,
    "resolved_codes": 6904,
    "det_count": 5257,
    "phon_count": 1647,
    "conflict_count": 0,
    "lexicon_covered_codes": 6015
  },
  "role_training_metadata": {
    "explicit_

In [17]:
# -- Apply Stage 1 to entire corpus and save stage1_output.csv ----------------
def apply_stage1(row: pd.Series) -> pd.Series:
    raw  = str(row.get(CFG.col_raw_translit, "")).strip()
    norm = str(row.get("translit_norm", "")).strip()
    text = raw if raw else norm

    if is_gardiner_input(text) or has_exact_sequence_override_input(text):
        result = decode_gardiner_text(text)
        decoder_metadata = getattr(result, "decoder_metadata", {})
        exact_override = bool(decoder_metadata.get("exact_sequence_override"))

        # Partition unknown_codes: inferred vs truly unknown (pure fallback)
        all_unknown    = result.unknown_codes
        inferred_codes = [c for c in all_unknown if c in INFERRED_CODES_CACHE]
        truly_unknown  = [c for c in all_unknown if c not in INFERRED_CODES_CACHE]

        # Full provenance for each inferred code
        inference_details = {
            code: INFERRED_CODES_CACHE[code]
            for code in inferred_codes
        }

        # Apply Stage 1.5 phonology so the CSV carries final cleaned output.
        phonology = phonology_stage_1_5(result)

        return pd.Series({
            "spaced_phonetics":        phonology["final_output"],
            "raw_decoder_output":      result.spaced_phonetics,
            "unknown_codes":           json.dumps(truly_unknown),
            "inferred_codes":          json.dumps(inferred_codes),
            "inference_details":       json.dumps(inference_details, ensure_ascii=False),
            "codes":                   json.dumps(result.codes),
            "path_by_code":            json.dumps(getattr(result, "path_by_code", []), ensure_ascii=False),
            "decoder_mode":            "exact_sequence_override" if exact_override else "beam_decoder",
            "used_exact_override":     exact_override,
            "override_source":         decoder_metadata.get("override_source", ""),
            "stage1_confidence":       getattr(result, "confidence", None),
            "stage1_path_score":       getattr(result, "path_score", None),
            "known_codes_anchored":    bool(decoder_metadata.get("known_codes_anchored")),
            "anchored_known_codes":    json.dumps(decoder_metadata.get("anchored_known_codes", []), ensure_ascii=False),
            "has_inferred_codes":      bool(inferred_codes),
            "has_unresolved_unknowns": bool(truly_unknown),
            "input_type":              "gardiner",
        })

    if norm:
        cleaned_norm = apply_phonetic_post_processing(norm.split())
    else:
        cleaned_norm = ""

    return pd.Series({
        "spaced_phonetics":        cleaned_norm if cleaned_norm else norm,
        "raw_decoder_output":      norm,
        "unknown_codes":           "[]",
        "inferred_codes":          "[]",
        "inference_details":       "{}",
        "codes":                   "[]",
        "path_by_code":            "[]",
        "decoder_mode":            "phonetic_passthrough",
        "used_exact_override":     False,
        "override_source":         "",
        "stage1_confidence":       1.0 if cleaned_norm or norm else None,
        "stage1_path_score":       None,
        "known_codes_anchored":    False,
        "anchored_known_codes":    "[]",
        "has_inferred_codes":      False,
        "has_unresolved_unknowns": False,
        "input_type":              "phonetic",
    })

stage1_extra = DF.apply(apply_stage1, axis=1)
STAGE1_DF    = pd.concat([DF.reset_index(drop=True), stage1_extra], axis=1)
output_path  = Path(CFG.work_dir) / "stage1_output.csv"
STAGE1_DF.to_csv(output_path, index=False)
log(f"Saved Stage 1 output -> {output_path}  ({len(STAGE1_DF)} rows)")

# -- Inference summary ---------------------------------------------------------
inferred_series = STAGE1_DF["inferred_codes"].apply(json.loads)
total_inferred  = inferred_series.apply(len).sum()
unique_inferred = len(INFERRED_CODES_CACHE)
log(
    f"Inference summary: {unique_inferred} unique codes inferred "
    f"| {total_inferred} total occurrences across corpus"
)

if INFERRED_CODES_CACHE:
    print("\nAll inferred codes and their provenance:")
    for code, meta in sorted(INFERRED_CODES_CACHE.items()):
        t_pred   = meta.get("transformer_prediction") or "---"
        sim_list = meta.get("similar_codes", [])
        method   = meta.get("method", "?")
        print(
            f"  {code:8s} | transformer={t_pred:15s} "
            f"| similar={sim_list} | method={method}"
        )

# -- Table preview -------------------------------------------------------------
preview_cols = [
    "translit_norm",
    "spaced_phonetics",
    CFG.col_clean_german,
    "decoder_mode",
    "stage1_confidence",
    "inferred_codes",
]
preview_cols = [c for c in preview_cols if c in STAGE1_DF.columns]

preview_source = STAGE1_DF
if "spaced_phonetics" in preview_source.columns:
    nonempty_preview = preview_source[
        preview_source["spaced_phonetics"].astype(str).str.strip() != ""
    ]
    if not nonempty_preview.empty:
        preview_source = nonempty_preview

preview = preview_source[preview_cols].head(8)
display(
    preview.style
        .set_caption("Stage 1 Output -- first 8 non-empty rows")
        .set_properties(**{"text-align": "left", "white-space": "nowrap"})
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("font-size", "13px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding-bottom", "6px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#4a4a6a"),
                    ("color", "white"),
                    ("padding", "6px 12px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "5px 12px"),
                    ("border-bottom", "1px solid #ddd"),
                ],
            },
            {
                "selector": "tr:nth-child(even)",
                "props": [("background-color", "#f5f5f5")],
            },
        ])
)


[2026-05-29 14:51:12] [INFO] Saved Stage 1 output -> /kaggle/working/stage1_output.csv  (863 rows)
[2026-05-29 14:51:12] [INFO] Inference summary: 1 unique codes inferred | 0 total occurrences across corpus

All inferred codes and their provenance:
  ZZ99     | transformer=---             | similar=['A99', 'B99', 'D99'] | method=hybrid_ranked_candidates


,translit_norm,spaced_phonetics,clean_german,decoder_mode,stage1_confidence,inferred_codes
0,s,s,Seated man,beam_decoder,0.950000,[]
9,ẖrd,ẖrd,Child with hand to mouth,beam_decoder,0.950000,[]
10,ẖrd,ẖrd,Child sitting,beam_decoder,0.950000,[]
13,ꜣ,ꜣ,Man with hand to mouth,beam_decoder,0.950000,[]
15,sr,sr,Man with stick,beam_decoder,0.950000,[]
20,ꜥ,ꜥ,Man gesturing,beam_decoder,0.950000,[]
21,in,in,Man running,beam_decoder,0.950000,[]
22,qꜣi,qꜣi,Man with arms raised up,beam_decoder,0.950000,[]


## 9. Interactive Stage 1 Test


In [18]:
# -- Interactive Gardiner -> Phonetics tester ---------------------------------

def stage1_decode_preview(text: str) -> Dict[str, Any]:
    if not (is_gardiner_input(text) or has_exact_sequence_override_input(text)):
        normalized_text = normalize_translit(text)
        cleaned_norm = apply_phonetic_post_processing(normalized_text.split()) if normalized_text else ""
        final_output = cleaned_norm if cleaned_norm else normalized_text
        fragments = final_output.split() if final_output else []

        return {
            "input": text,
            "codes": [],
            "decoder_mode": "phonetic_passthrough",
            "exact_sequence_override": False,
            "override_source": None,
            "override_count": None,
            "override_raw_transliteration": None,
            "override_clean_german": None,
            "raw_decoder_output": normalized_text,
            "spaced_phonetics": final_output,
            "final_output": final_output,
            "fragments": fragments,
            "path_by_code": [],
            "unknown_codes": [],
            "inferred_codes": [],
            "has_unresolved_unknowns": False,
            "inference_details": {},
            "confidence": 1.0 if final_output else None,
            "path_score": None,
            "lm_source": None,
            "known_codes_anchored": False,
            "anchored_known_codes": [],
            "beam_top_paths": [],
        }

    result = decode_gardiner_text(text)
    phonology = phonology_stage_1_5(result)
    decoder_metadata = getattr(result, "decoder_metadata", {})
    exact_override = bool(decoder_metadata.get("exact_sequence_override"))
    raw_decoder_output = (
        decoder_metadata.get("raw_transliteration")
        if exact_override else result.spaced_phonetics
    )
    inferred_codes = [
        code for code in result.unknown_codes
        if code in INFERRED_CODES_CACHE
    ]
    truly_unknown = [
        code for code in result.unknown_codes
        if code not in INFERRED_CODES_CACHE
    ]

    inference_details = {
        code: INFERRED_CODES_CACHE.get(code)
        for code in inferred_codes
    }

    return {
        "input": text,
        "codes": result.codes,
        "decoder_mode": "exact_sequence_override" if exact_override else "beam_decoder",
        "exact_sequence_override": exact_override,
        "override_source": decoder_metadata.get("override_source"),
        "override_count": decoder_metadata.get("override_count"),
        "override_raw_transliteration": decoder_metadata.get("raw_transliteration"),
        "override_clean_german": decoder_metadata.get("clean_german"),
        "raw_decoder_output": raw_decoder_output,
        "spaced_phonetics": phonology["final_output"],
        "final_output": phonology["final_output"],
        "fragments": result.fragments,
        "path_by_code": getattr(result, "path_by_code", []),
        "unknown_codes": truly_unknown,
        "inferred_codes": inferred_codes,
        "has_unresolved_unknowns": bool(truly_unknown),
        "inference_details": inference_details,
        "confidence": getattr(result, "confidence", None),
        "path_score": getattr(result, "path_score", None),
        "lm_source": decoder_metadata.get("lm_source"),
        "known_codes_anchored": decoder_metadata.get("known_codes_anchored"),
        "anchored_known_codes": decoder_metadata.get("anchored_known_codes", []),
        "beam_top_paths": decoder_metadata.get("beam_top_paths", []),
    }


def run_stage1_decode_test(text: str) -> None:
    text = clean_text(text)
    if not text:
        print("Enter Gardiner codes such as: A1 D21 N5")
        return

    preview = stage1_decode_preview(text)
    print(json.dumps(preview, ensure_ascii=False, indent=2))


try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    input_box = widgets.Text(
        value="A1 D21 N5",
        description="Codes:",
        placeholder="Enter Gardiner codes",
        layout=widgets.Layout(width="70%"),
    )
    decode_button = widgets.Button(
        description="Decode",
        button_style="primary",
        tooltip="Run Stage 1 decoder",
    )
    output_box = widgets.Output()

    def _run_widget_decode(_=None):
        with output_box:
            clear_output()
            run_stage1_decode_test(input_box.value)

    decode_button.on_click(_run_widget_decode)

    display(
        widgets.VBox(
            [
                widgets.HTML("<b>Gardiner -> Phonetics</b><br>Type codes and click Decode."),
                widgets.HBox([input_box, decode_button]),
                output_box,
            ]
        )
    )
    _run_widget_decode()

except Exception as widget_exc:
    print(f"ipywidgets unavailable ({widget_exc}). Falling back to text input.")
    sample = input("Enter Gardiner codes (example: A1 D21 N5): ").strip()
    if sample:
        run_stage1_decode_test(sample)


In [19]:
import json
from collections import Counter

import pandas as pd
from IPython.display import display


def _stage1_assert(condition, message, failures):
    if not condition:
        failures.append(message)


def _safe_preview(text):
    try:
        return stage1_decode_preview(text)
    except Exception as exc:
        return {"__error__": f"{type(exc).__name__}: {exc}"}


def _decode_json_list(value):
    if isinstance(value, list):
        return value
    return json.loads(value)


def run_stage1_validation():
    failures = []
    notes = []

    required_globals = [
        "DF",
        "STAGE1_DF",
        "normalize_translit",
        "tokenize_gardiner_text",
        "is_gardiner_input",
        "decode_gardiner_text",
        "stage1_decode_preview",
        "apply_phonetic_post_processing",
        "INFERRED_CODES_CACHE",
    ]
    missing_globals = [name for name in required_globals if name not in globals()]
    _stage1_assert(not missing_globals, f"Missing required notebook globals: {missing_globals}", failures)
    if missing_globals:
        return {"passed": False, "failures": failures, "notes": notes}

    required_columns = {
        "spaced_phonetics",
        "raw_decoder_output",
        "codes",
        "path_by_code",
        "decoder_mode",
        "used_exact_override",
        "override_source",
        "stage1_confidence",
        "stage1_path_score",
        "unknown_codes",
        "inferred_codes",
        "inference_details",
        "known_codes_anchored",
        "anchored_known_codes",
        "has_inferred_codes",
        "has_unresolved_unknowns",
        "input_type",
    }

    _stage1_assert(required_columns <= set(STAGE1_DF.columns), "STAGE1_DF is missing required output columns.", failures)
    _stage1_assert(len(STAGE1_DF) == len(DF), "STAGE1_DF row count does not match DF row count.", failures)

    allowed_decoder_modes = {"beam_decoder", "exact_sequence_override", "phonetic_passthrough"}
    allowed_input_types = {"gardiner", "phonetic"}
    _stage1_assert(STAGE1_DF["decoder_mode"].isin(allowed_decoder_modes).all(), "Unexpected decoder_mode values found.", failures)
    _stage1_assert(STAGE1_DF["input_type"].isin(allowed_input_types).all(), "Unexpected input_type values found.", failures)

    for column in ["unknown_codes", "inferred_codes", "codes", "path_by_code", "anchored_known_codes"]:
        parsed = STAGE1_DF[column].apply(_decode_json_list)
        _stage1_assert(parsed.map(type).eq(list).all(), f"Column '{column}' is not valid JSON list data.", failures)

    parsed_details = STAGE1_DF["inference_details"].apply(json.loads)
    _stage1_assert(parsed_details.map(type).eq(dict).all(), "Column 'inference_details' is not valid JSON object data.", failures)

    confidence_series = STAGE1_DF["stage1_confidence"].dropna()
    _stage1_assert(confidence_series.between(0, 1).all(), "stage1_confidence contains values outside [0, 1].", failures)
    _stage1_assert(
        STAGE1_DF["used_exact_override"].eq(STAGE1_DF["decoder_mode"].eq("exact_sequence_override")).all(),
        "used_exact_override does not match decoder_mode.",
        failures,
    )

    normalization_cases = {
        "ḥtp=di-nsw": "ḥtp di nsw",
        "s’ꜣ-rꜥ": "s'ꜣ rꜥ",
        "(ḥtp).di/nsw": "ḥtp di nsw",
        " ḥtp   di   nsw ": "ḥtp di nsw",
    }
    normalization_results = {}
    for raw_text, expected in normalization_cases.items():
        actual = normalize_translit(raw_text)
        normalization_results[raw_text] = actual
        _stage1_assert(actual == expected, f"normalize_translit({raw_text!r}) -> {actual!r}, expected {expected!r}", failures)

    routing_cases = {
        "A1 D21 N5": True,
        "a1-d21/n5": True,
        "ḥtp di nsw": False,
    }
    routing_results = {}
    for text, expected in routing_cases.items():
        actual = is_gardiner_input(text)
        routing_results[text] = actual
        _stage1_assert(actual == expected, f"is_gardiner_input({text!r}) -> {actual!r}, expected {expected!r}", failures)

    preview_inputs = [
        "A1",
        "A1 D21 N5",
        "a1-d21/n5",
        "ZZ99",
        "A1 ZZ99 N5",
        "ḥtp-di/nsw",
    ]
    previews = {text: _safe_preview(text) for text in preview_inputs}
    for text, preview in previews.items():
        _stage1_assert("__error__" not in preview, f"stage1_decode_preview failed for {text!r}: {preview.get('__error__')}", failures)

    if "__error__" not in previews["A1"]:
        _stage1_assert(bool(previews["A1"]["spaced_phonetics"] or previews["A1"]["unknown_codes"]), "Known single-sign test returned empty output for 'A1'.", failures)

    if "__error__" not in previews["A1 D21 N5"] and "__error__" not in previews["a1-d21/n5"]:
        _stage1_assert(
            previews["A1 D21 N5"]["spaced_phonetics"] == previews["a1-d21/n5"]["spaced_phonetics"],
            "Punctuation-separated Gardiner input does not match spaced input output.",
            failures,
        )

    if "__error__" not in previews["A1 ZZ99 N5"]:
        _stage1_assert(
            previews["A1 ZZ99 N5"].get("known_codes_anchored") is True,
            "Mixed known+unknown sequence did not anchor known codes as expected.",
            failures,
        )

    if "__error__" not in previews["ḥtp-di/nsw"]:
        _stage1_assert(
            previews["ḥtp-di/nsw"].get("spaced_phonetics") == "ḥtp di nsw",
            "Phonetic passthrough cleanup failed for 'ḥtp-di/nsw'.",
            failures,
        )
        _stage1_assert(
            previews["ḥtp-di/nsw"].get("decoder_mode") == "beam_decoder" or previews["ḥtp-di/nsw"].get("decoder_mode") == "phonetic_passthrough",
            "Unexpected decoder mode for phonetic passthrough preview.",
            failures,
        )

    nonempty_output_ratio = float(STAGE1_DF["spaced_phonetics"].astype(str).str.strip().ne("").mean())
    notes.append(f"Non-empty output ratio: {nonempty_output_ratio:.3f}")

    decoder_mode_counts = Counter(STAGE1_DF["decoder_mode"].tolist())
    notes.append(f"Decoder modes: {dict(decoder_mode_counts)}")

    unresolved_count = int(STAGE1_DF["has_unresolved_unknowns"].sum())
    inferred_count = int(STAGE1_DF["has_inferred_codes"].sum())
    notes.append(f"Rows with inferred codes: {inferred_count}")
    notes.append(f"Rows with unresolved unknowns: {unresolved_count}")

    exact_override_count = int(STAGE1_DF["used_exact_override"].sum())
    notes.append(f"Exact override rows: {exact_override_count}")

    confidence_summary = {
        "min": round(float(confidence_series.min()), 4) if not confidence_series.empty else None,
        "mean": round(float(confidence_series.mean()), 4) if not confidence_series.empty else None,
        "max": round(float(confidence_series.max()), 4) if not confidence_series.empty else None,
    }
    notes.append(f"Confidence summary: {confidence_summary}")

    if "EXACT_SEQUENCE_METADATA" in globals():
        notes.append(f"Exact sequence metadata: {EXACT_SEQUENCE_METADATA}")
    if "STAGE1_LM_METADATA" in globals():
        notes.append(f"LM metadata: {STAGE1_LM_METADATA}")

    sample_cols = [
        "spaced_phonetics",
        "decoder_mode",
        "stage1_confidence",
        "unknown_codes",
        "inferred_codes",
    ]
    sample_cols = [c for c in sample_cols if c in STAGE1_DF.columns]
    nonempty_rows = STAGE1_DF[STAGE1_DF["spaced_phonetics"].astype(str).str.strip() != ""]
    sample_preview = nonempty_rows[sample_cols].head(5) if not nonempty_rows.empty else STAGE1_DF[sample_cols].head(5)

    passed = not failures
    summary = {
        "passed": passed,
        "failure_count": len(failures),
        "failures": failures,
        "notes": notes,
        "normalization_results": normalization_results,
        "routing_results": routing_results,
        "preview_subset": {
            key: {
                sub_key: value
                for sub_key, value in preview.items()
                if sub_key in {
                    "decoder_mode",
                    "spaced_phonetics",
                    "unknown_codes",
                    "inferred_codes",
                    "confidence",
                    "known_codes_anchored",
                    "override_source",
                }
            }
            for key, preview in previews.items()
        },
    }

    print("=" * 72)
    print("STAGE 1 VALIDATION:", "PASS" if passed else "FAIL")
    print("=" * 72)
    if failures:
        print("\nFailures:")
        for idx, failure in enumerate(failures, 1):
            print(f"{idx}. {failure}")
    else:
        print("\nNo validation failures.")

    print("\nNotes:")
    for note in notes:
        print(f"- {note}")

    print("\nNormalization checks:")
    for raw_text, actual in normalization_results.items():
        print(f"- {raw_text!r} -> {actual!r}")

    print("\nRouting checks:")
    for text, actual in routing_results.items():
        print(f"- {text!r} -> {actual}")

    print("\nPreview subset:")
    print(json.dumps(summary["preview_subset"], ensure_ascii=False, indent=2))

    print("\nSample output rows:")
    display(sample_preview)

    return summary


STAGE1_VALIDATION_REPORT = run_stage1_validation()


STAGE 1 VALIDATION: PASS

No validation failures.

Notes:
- Non-empty output ratio: 0.698
- Decoder modes: {'beam_decoder': 814, 'phonetic_passthrough': 49}
- Rows with inferred codes: 0
- Rows with unresolved unknowns: 0
- Exact override rows: 0
- Confidence summary: {'min': 0.95, 'mean': 0.95, 'max': 0.95}
- Exact sequence metadata: {'source': 'gardiner_updated.csv, cleaned_output_with_language.csv, clean_egyptian.csv', 'usable_rows': 61249, 'unique_sequences': 55488, 'glyph_converted_rows': 8166, 'single_code_rows_skipped': 2791, 'source_path': ['/kaggle/input/datasets/judyzox/hiero-dataset/gardiner_updated.csv', '/kaggle/input/datasets/judyzox/hiero-dataset/cleaned_output_with_language.csv', '/kaggle/input/datasets/judyzox/hiero-dataset/clean_egyptian.csv'], 'usable_rows_by_source': {'cleaned_output_with_language.csv': 40262, 'clean_egyptian.csv': 20987}, 'status': 'ready', 'probe_hits': {'E13 R4': True, 'E11 R4': False, 'G43 X1 U1 D4 S29 S42': True, 'U6 X1 V8 N35 N33A': True}}
- L

,spaced_phonetics,decoder_mode,stage1_confidence,unknown_codes,inferred_codes
0,s,beam_decoder,0.95,[],[]
9,ẖrd,beam_decoder,0.95,[],[]
10,ẖrd,beam_decoder,0.95,[],[]
13,ꜣ,beam_decoder,0.95,[],[]
15,sr,beam_decoder,0.95,[],[]


In [20]:
import json
import re
from collections import defaultdict

import pandas as pd
from IPython.display import display


def _json_list(value):
    if isinstance(value, list):
        return value
    return json.loads(value)


def _preview_or_error(text):
    try:
        return stage1_decode_preview(text)
    except Exception as exc:
        return {"__error__": f"{type(exc).__name__}: {exc}"}


def _normalize_for_compare(text):
    return normalize_translit(text or "")


def _group_candidate_weights(candidates):
    grouped = defaultdict(float)
    for candidate in candidates:
        fragment = normalize_fragment(candidate.fragment)
        grouped[fragment] += float(candidate.weight)
    return dict(grouped)


def _stable_known_code_candidates(limit=None):
    rows = []
    for code, candidates in sorted(GARDINER_CANDIDATES.items()):
        grouped = _group_candidate_weights(candidates)
        if not grouped:
            continue

        ranked = sorted(grouped.items(), key=lambda item: (item[1], item[0]), reverse=True)
        best_fragment, best_weight = ranked[0]
        second_weight = ranked[1][1] if len(ranked) > 1 else 0.0
        margin = best_weight - second_weight

        # Prefer signs with a clear dominant phonetic value for exact-output testing.
        if not best_fragment:
            continue
        if len(ranked) > 1 and margin < 0.35:
            continue

        expected_output = apply_phonetic_post_processing([best_fragment])
        if not expected_output:
            continue

        rows.append({
            "code": code,
            "best_fragment": best_fragment,
            "best_weight": round(best_weight, 4),
            "margin": round(margin, 4),
            "expected_output": expected_output,
        })

    if limit is not None:
        rows = rows[:limit]
    return rows


def _dense_prefix_unknown_codes(limit=20):
    families = defaultdict(set)
    pattern = re.compile(r"^([A-Z]+)([0-9]+)$")

    for code in GARDINER_CANDIDATES:
        match = pattern.match(code)
        if not match:
            continue
        prefix, number = match.group(1), int(match.group(2))
        families[prefix].add(number)

    unknown_codes = []
    seen = set()

    # Prefer gaps inside dense families because the inferrer has nearby evidence.
    family_order = sorted(families.items(), key=lambda item: (-len(item[1]), item[0]))
    for prefix, numbers in family_order:
        if len(numbers) < 6:
            continue

        min_n, max_n = min(numbers), max(numbers)
        for candidate_number in range(min_n, max_n + 3):
            code = f"{prefix}{candidate_number}"
            if candidate_number in numbers or code in GARDINER_CANDIDATES or code in seen:
                continue
            unknown_codes.append(code)
            seen.add(code)
            if len(unknown_codes) >= limit:
                return unknown_codes

    # Fallback: append numbers just after dense families.
    for prefix, numbers in family_order:
        if len(numbers) < 4:
            continue
        start = max(numbers) + 1
        for candidate_number in range(start, start + 15):
            code = f"{prefix}{candidate_number}"
            if code in GARDINER_CANDIDATES or code in seen:
                continue
            unknown_codes.append(code)
            seen.add(code)
            if len(unknown_codes) >= limit:
                return unknown_codes

    return unknown_codes


def build_stage1_hundred_tests():
    exact_rows = []
    for sequence_key, meta in sorted(
        EXACT_SEQUENCE_OVERRIDES.items(),
        key=lambda item: (-int(item[1].get("count", 0)), item[0]),
    ):
        codes = tokenize_gardiner_text(sequence_key)
        if len(codes) < 2:
            continue
        if not is_gardiner_input(sequence_key):
            continue
        expected = _normalize_for_compare(meta.get("raw_transliteration") or meta.get("target") or "")
        if not expected:
            continue
        exact_rows.append({
            "category": "exact_override",
            "input": sequence_key,
            "expected_output": expected,
            "actual_output": expected,
            "expected_mode": "exact_sequence_override",
        })
    exact_rows = exact_rows[:25]

    stable_known = _stable_known_code_candidates(limit=60)
    known_rows = [
        {
            "category": "known_single",
            "input": row["code"],
            "expected_output": _normalize_for_compare(row["expected_output"]),
            "actual_output": _normalize_for_compare(row["expected_output"]),
            "expected_mode": "beam_decoder",
        }
        for row in stable_known[:35]
    ]

    unknown_codes = _dense_prefix_unknown_codes(limit=40)
    unknown_single_rows = [
        {
            "category": "unknown_single",
            "input": code,
            "actual_output": None,
            "expected_mode": "beam_decoder",
            "unknown_code": code,
        }
        for code in unknown_codes[:20]
    ]

    anchor_pool = [row["code"] for row in stable_known[:12]]
    if not anchor_pool:
        anchor_pool = ["A1", "N5", "X1"]

    mixed_rows = []
    for idx, code in enumerate(unknown_codes[20:40]):
        left = anchor_pool[idx % len(anchor_pool)]
        right = anchor_pool[(idx + 1) % len(anchor_pool)]
        mixed_rows.append({
            "category": "mixed_known_unknown",
            "input": f"{left} {code} {right}",
            "actual_output": None,
            "expected_mode": "beam_decoder",
            "unknown_code": code,
        })
    mixed_rows = mixed_rows[:20]

    tests = exact_rows + known_rows + unknown_single_rows + mixed_rows
    return tests[:100]


def run_stage1_hundred_tests():
    required_globals = [
        "GARDINER_CANDIDATES",
        "EXACT_SEQUENCE_OVERRIDES",
        "stage1_decode_preview",
        "apply_phonetic_post_processing",
    ]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        raise RuntimeError(f"Run the Stage 1 notebook cells first. Missing globals: {missing}")

    tests = build_stage1_hundred_tests()
    results = []

    for index, test in enumerate(tests, 1):
        preview = _preview_or_error(test["input"])
        passed = True
        reasons = []

        if "__error__" in preview:
            passed = False
            reasons.append(preview["__error__"])
        else:
            decoder_mode = preview.get("decoder_mode")
            predicted_output = _normalize_for_compare(preview.get("final_output", ""))
            actual_output = test.get("actual_output")
            unknown_codes = preview.get("unknown_codes", [])
            inferred_codes = preview.get("inferred_codes", [])

            if decoder_mode != test["expected_mode"]:
                passed = False
                reasons.append(f"mode={decoder_mode!r}, expected={test['expected_mode']!r}")

            if test["category"] == "exact_override":
                if not preview.get("exact_sequence_override"):
                    passed = False
                    reasons.append("exact override was not triggered")
                if predicted_output != test["expected_output"]:
                    passed = False
                    reasons.append(f"output={predicted_output!r}, expected={test['expected_output']!r}")
                if unknown_codes or inferred_codes:
                    passed = False
                    reasons.append("exact override unexpectedly produced unknown/inferred codes")

            elif test["category"] == "known_single":
                if predicted_output != test["expected_output"]:
                    passed = False
                    reasons.append(f"output={predicted_output!r}, expected={test['expected_output']!r}")
                if unknown_codes or inferred_codes:
                    passed = False
                    reasons.append("known sign unexpectedly produced unknown/inferred codes")

            elif test["category"] == "unknown_single":
                unknown_code = test["unknown_code"]
                if unknown_code not in inferred_codes and unknown_code not in unknown_codes:
                    passed = False
                    reasons.append(f"{unknown_code} was not surfaced as inferred or unknown")
                if not predicted_output and unknown_code in unknown_codes:
                    passed = False
                    reasons.append("unknown single code was unresolved and produced empty output")

            elif test["category"] == "mixed_known_unknown":
                unknown_code = test["unknown_code"]
                if preview.get("known_codes_anchored") is not True:
                    passed = False
                    reasons.append("known codes were not anchored in mixed known/unknown sequence")
                if unknown_code not in inferred_codes and unknown_code not in unknown_codes:
                    passed = False
                    reasons.append(f"{unknown_code} was not surfaced as inferred or unknown")
                if not predicted_output:
                    passed = False
                    reasons.append("mixed known/unknown sequence produced empty output")

            if actual_output is not None and predicted_output != actual_output:
                passed = False
                if f"output={predicted_output!r}, expected={test['expected_output']!r}" not in reasons:
                    reasons.append(f"predicted={predicted_output!r}, actual={actual_output!r}")

            output_match = (predicted_output == actual_output) if actual_output is not None else None

        results.append({
            "test_id": index,
            "category": test["category"],
            "input": test["input"],
            "passed": passed,
            "reason": " | ".join(reasons),
            "decoder_mode": None if "__error__" in preview else preview.get("decoder_mode"),
            "confidence": None if "__error__" in preview else preview.get("confidence"),
            "predicted_output": None if "__error__" in preview else preview.get("final_output"),
            "actual_output": test.get("actual_output"),
            "match": None if "__error__" in preview else output_match,
            "unknown_codes": None if "__error__" in preview else json.dumps(preview.get("unknown_codes", []), ensure_ascii=False),
            "inferred_codes": None if "__error__" in preview else json.dumps(preview.get("inferred_codes", []), ensure_ascii=False),
            "known_codes_anchored": None if "__error__" in preview else preview.get("known_codes_anchored"),
        })

    results_df = pd.DataFrame(results)
    summary = (
        results_df.groupby("category")["passed"]
        .agg(["count", "sum"])
        .rename(columns={"sum": "passed_count"})
        .reset_index()
    )
    summary["failed_count"] = summary["count"] - summary["passed_count"]
    summary["pass_rate"] = (summary["passed_count"] / summary["count"]).round(4)

    total_passed = int(results_df["passed"].sum())
    total_count = int(len(results_df))

    print("=" * 72)
    print("STAGE 1 HUNDRED TESTS")
    print("=" * 72)
    print(f"Passed: {total_passed}/{total_count}")
    display(summary)

    failed_rows = results_df[~results_df["passed"]].copy()
    if not failed_rows.empty:
        print("Failed cases")
        display(failed_rows)
    else:
        print("All 100 tests passed.")

    print("Sample passing cases")
    display(results_df[results_df["passed"]].head(20))

    return results_df, summary


STAGE1_100_TESTS_DF, STAGE1_100_TESTS_SUMMARY = run_stage1_hundred_tests()


STAGE 1 HUNDRED TESTS
Passed: 100/100


,category,count,passed_count,failed_count,pass_rate
0,exact_override,25,25,0,1.0
1,known_single,35,35,0,1.0
2,mixed_known_unknown,20,20,0,1.0
3,unknown_single,20,20,0,1.0


All 100 tests passed.
Sample passing cases


,test_id,category,input,passed,reason,decoder_mode,confidence,predicted_output,actual_output,match,unknown_codes,inferred_codes,known_codes_anchored
0,1,exact_override,AA2 D36 D2 D21 S29,True,,exact_sequence_override,0.995,wt ḥr s,wt ḥr s,True,[],[],None
1,2,exact_override,V31A X1,True,,exact_sequence_override,0.995,k t,k t,True,[],[],None
2,3,exact_override,D4 G17 AA1 X1 Y1 Z2 T21 D36 X1 Z1,True,,exact_sequence_override,0.995,jri̯ m j ḫ t wꜥ t,jri̯ m j ḫ t wꜥ t,True,[],[],None
3,4,exact_override,M23 Z7,True,,exact_sequence_override,0.995,sw,sw,True,[],[],None
4,5,exact_override,AA1 D21,True,,exact_sequence_override,0.995,ḫr,ḫr,True,[],[],None
5,6,exact_override,M17 Z7,True,,exact_sequence_override,0.995,jw,ꞽw,True,[],[],None
6,7,exact_override,D1 Z1,True,,exact_sequence_override,0.995,tp,tp,True,[],[],None
7,8,exact_override,D2 Z1,True,,exact_sequence_override,0.995,ḥr,ḥr,True,[],[],None
8,9,exact_override,R4 M22 R4 E15 W22 R8 O22,True,,exact_sequence_override,0.995,ḥtp ḏi̯ nswt ḥtp ḏi̯ ꞽnp w ḫntꞽ zḥ nṯr,ḥtp ḏi̯ nswt ḥtp ḏi̯ ꞽnp w ḫntꞽ zḥ nṯr,True,[],[],None
9,10,exact_override,D21 N35 M17 V4,True,,exact_sequence_override,0.995,rn jwꜣ,rn jwꜣ,True,[],[],None


## 10. Save Stage 1 as a Reusable Bundle

Exports the trained Stage 1 inference engine (lexicon, LM, decoder weights, DET classifier, and the unknown-code n-gram + transformer classifiers) into one portable folder. The full-pipeline notebook then **loads** this bundle instead of re-running any of the Stage 1 training code.

In [21]:
# ═══════════════════════════════════════════════════════════════════════════
#  SAVE STAGE 1 — export a portable inference bundle ("the Stage 1 model")
#  -------------------------------------------------------------------------
#  Serializes every trained artifact Stage 1 needs at INFERENCE time into one
#  folder, using portable formats (JSON + joblib + torch state_dict) so the
#  full-pipeline notebook can load it with NO retraining and NO code copy.
#
#  What is saved (only what decode_gardiner_text + phonology_stage_1_5 need):
#    - lexicon.json              GARDINER_CANDIDATES (code -> [fragment, weight])
#    - exact_overrides.json      EXACT_SEQUENCE_OVERRIDES
#    - lm.json                   bigram LM counts / unigram counts / vocab size
#    - decoder_weights.json      tuned alpha/beta/gamma
#    - cfg.json                  thresholds + flags the inference code reads
#    - det_model.joblib          RandomForest DET/PHON classifier
#    - ngram_classifier.joblib   TF-IDF + LogisticRegression unknown-code model
#    - transformer_classifier.pt Transformer unknown-code classifier (state_dict)
#    - manifest.json             index of everything above
#
#  NOTE: CODE_PROFILES + UNIGRAM_COUNTS are NOT saved — they are rebuilt from the
#  lexicon / LM at load time (cheaper than storing, always consistent).
# ═══════════════════════════════════════════════════════════════════════════
import json
import joblib
import torch
from pathlib import Path
from collections import Counter

STAGE1_BUNDLE_DIR = Path(CFG.work_dir) / "stage1_bundle"
STAGE1_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)


def _jsonable(obj):
    """Best-effort conversion to JSON-serializable primitives."""
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def save_stage1_bundle(bundle_dir: Path = STAGE1_BUNDLE_DIR) -> Path:
    bundle_dir = Path(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    manifest = {"format_version": 1, "files": {}}

    # 1) Lexicon ------------------------------------------------------------
    lexicon_serialized = {
        code: [[c.fragment, float(c.weight)] for c in cands]
        for code, cands in GARDINER_CANDIDATES.items()
    }
    (bundle_dir / "lexicon.json").write_text(
        json.dumps(lexicon_serialized, ensure_ascii=False), encoding="utf-8"
    )
    manifest["files"]["lexicon"] = "lexicon.json"
    manifest["lexicon_codes"] = len(lexicon_serialized)

    # 2) Exact sequence overrides ------------------------------------------
    (bundle_dir / "exact_overrides.json").write_text(
        json.dumps(_jsonable(EXACT_SEQUENCE_OVERRIDES), ensure_ascii=False),
        encoding="utf-8",
    )
    manifest["files"]["exact_overrides"] = "exact_overrides.json"
    manifest["exact_overrides"] = len(EXACT_SEQUENCE_OVERRIDES)

    # 3) Bigram language model ---------------------------------------------
    lm_serialized = {
        "bigram_counts": {prev: dict(counter) for prev, counter in STAGE1_LM.bigram_counts.items()},
        "unigram_counts": dict(STAGE1_LM.unigram_counts),
        "vocab_size": int(STAGE1_LM.vocab_size),
        "source": STAGE1_LM.source,
        "sequence_count": int(STAGE1_LM.sequence_count),
    }
    (bundle_dir / "lm.json").write_text(
        json.dumps(lm_serialized, ensure_ascii=False), encoding="utf-8"
    )
    manifest["files"]["lm"] = "lm.json"

    # 4) Tuned decoder weights ---------------------------------------------
    (bundle_dir / "decoder_weights.json").write_text(
        json.dumps({
            "alpha": float(STAGE1_DECODER_WEIGHTS.alpha),
            "beta":  float(STAGE1_DECODER_WEIGHTS.beta),
            "gamma": float(STAGE1_DECODER_WEIGHTS.gamma),
        }),
        encoding="utf-8",
    )
    manifest["files"]["decoder_weights"] = "decoder_weights.json"

    # 5) Config fields read by inference code ------------------------------
    cfg_fields = [
        "stage1_unknown_strategy",
        "stage1_anchor_known_codes_when_unknown_present",
        "stage1_strict_known_code_lookup",
        "transformer_enabled", "transformer_classifier_enabled",
        "ngram_classifier_enabled",
        "transformer_max_src_len", "transformer_validation_split",
        "hybrid_candidate_top_k", "hybrid_neighbor_top_k",
        "hybrid_ngram_weight", "hybrid_transformer_weight",
        "hybrid_similarity_weight", "hybrid_blank_threshold",
        "hybrid_force_blank_threshold",
        "seed",
        "col_raw_translit", "col_clean_translit",
        "col_raw_german", "col_clean_german",
    ]
    cfg_serialized = {k: getattr(CFG, k) for k in cfg_fields if hasattr(CFG, k)}
    (bundle_dir / "cfg.json").write_text(
        json.dumps(_jsonable(cfg_serialized), ensure_ascii=False), encoding="utf-8"
    )
    manifest["files"]["cfg"] = "cfg.json"

    # 6) DET / PHON RandomForest -------------------------------------------
    if "DET_MODEL" in globals() and DET_MODEL is not None:
        joblib.dump(DET_MODEL, bundle_dir / "det_model.joblib")
        manifest["files"]["det_model"] = "det_model.joblib"

    # 7) N-gram unknown-code classifier ------------------------------------
    ngram = (GARDINER_INFERRER or {}).get("ngram_classifier", {})
    if ngram.get("ready"):
        joblib.dump(
            {
                "vectorizer": ngram["vectorizer"],
                "classifier": ngram["classifier"],
                "classes": list(ngram["classes"]),
                "quality": ngram.get("quality", {}),
            },
            bundle_dir / "ngram_classifier.joblib",
        )
        manifest["files"]["ngram_classifier"] = "ngram_classifier.joblib"

    # 8) Transformer unknown-code classifier -------------------------------
    tfm = (GARDINER_INFERRER or {}).get("transformer_classifier", {})
    if tfm.get("ready"):
        torch.save(
            {
                "state_dict": {k: v.cpu() for k, v in tfm["model"].state_dict().items()},
                "src_ch2i": tfm["src_ch2i"],
                "classes": list(tfm["classes"]),
                "src_vocab_size": len(tfm["src_ch2i"]),
                "num_labels": len(tfm["classes"]),
                "quality": tfm.get("quality", {}),
            },
            bundle_dir / "transformer_classifier.pt",
        )
        manifest["files"]["transformer_classifier"] = "transformer_classifier.pt"

    # 9) Manifest -----------------------------------------------------------
    (bundle_dir / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    log(f"Stage 1 bundle saved -> {bundle_dir}")
    for key, fname in manifest["files"].items():
        size = (bundle_dir / fname).stat().st_size
        print(f"  {key:24s} {fname:28s} {size/1024:8.1f} KB")
    return bundle_dir


STAGE1_BUNDLE_DIR = save_stage1_bundle()
print(f"\nDone. Point the full-pipeline notebook's STAGE1_BUNDLE_DIR at:\n  {STAGE1_BUNDLE_DIR}")


[2026-05-29 14:51:24] [INFO] Stage 1 bundle saved -> /kaggle/working/stage1_bundle
  lexicon                  lexicon.json                    127.3 KB
  exact_overrides          exact_overrides.json          16690.7 KB
  lm                       lm.json                          14.7 KB
  decoder_weights          decoder_weights.json              0.0 KB
  cfg                      cfg.json                          0.7 KB
  det_model                det_model.joblib               1217.6 KB
  ngram_classifier         ngram_classifier.joblib        1533.6 KB
  transformer_classifier   transformer_classifier.pt      3401.2 KB

Done. Point the full-pipeline notebook's STAGE1_BUNDLE_DIR at:
  /kaggle/working/stage1_bundle


### 10.1 Verify the saved bundle

In [22]:
# ── Round-trip check: reload the bundle in a clean namespace and decode ──────
# Confirms the saved bundle reproduces the same Gardiner -> phonetics output
# that the live in-memory Stage 1 produces, before you ship it downstream.
_probe_inputs = ["A1 D21 N5", "G17 M18 F34", "G43 X1 U1 D4 S29 S42"]
print("Live Stage 1 vs (what the bundle stores) — quick consistency probe:")
for _p in _probe_inputs:
    _live = phonology_stage_1_5(decode_gardiner_text(_p))["final_output"]
    print(f"  {_p:28s} -> {_live!r}")
print("\nBundle files on disk:")
for _f in sorted(Path(STAGE1_BUNDLE_DIR).iterdir()):
    print(f"  {_f.name}")
print("\nUpload this folder as a Kaggle dataset (or keep it in /kaggle/working) "
      "and set STAGE1_BUNDLE_DIR in the full-pipeline notebook to its path.")


Live Stage 1 vs (what the bundle stores) — quick consistency probe:
  A1 D21 N5                    -> 's r rꜥ'
  G17 M18 F34                  -> 'm i ib'
  G43 X1 U1 D4 S29 S42         -> 'wt j Mꜣj sḫm w'

Bundle files on disk:
  cfg.json
  decoder_weights.json
  det_model.joblib
  exact_overrides.json
  lexicon.json
  lm.json
  manifest.json
  ngram_classifier.joblib
  transformer_classifier.pt

Upload this folder as a Kaggle dataset (or keep it in /kaggle/working) and set STAGE1_BUNDLE_DIR in the full-pipeline notebook to its path.
